# Fine-tune Qwen3.5-4B (LoRA / QLoRA) for LifeMate on Google Colab

Orchestrates the repo `Le-Minh-Nhut/zaloai` — the training loop, collator, checkpoint format,
evaluation and export all come from the package. This notebook only decides *where* things live,
*how much* memory to use, and *when* to sync to Drive.

**Before you click anything:** `Runtime -> Change runtime type -> GPU`. A CPU runtime cannot run
this notebook and still bills Compute Units (CU) while it sits idle.

| section | what it costs | CU (approx., ESTIMATE) |
| --- | --- | --- |
| 1-3 setup, install, clone | 2-4 min, no model | 0.1 |
| 4-5 Drive + dataset build | 3-8 min CPU | 0.2 |
| 6-7 model load + 1-step probe | 5-15 min GPU | T4 0.3 / L4 0.8 / A100 1.5 |
| 8 training (3 epochs, 4368 records) | hours — see the cell's own estimate | the expensive part |
| 9-13 eval, export, smoke test, cleanup | 10-30 min | 0.3-1.0 |

Rates are Colab's published-ish CU/hour for a T4 / L4 / A100 class GPU; treat every number here as
an **ESTIMATE**. Nothing in this notebook was executed on a Colab runtime when it was written.

Ground rules baked into the cells below:

* the model id is passed explicitly (`Qwen/Qwen3.5-4B`); the repo default may change to
  `openbmb/MiniCPM5-2B`, and this notebook must never silently follow it;
* the test splits are never read, never trained on, never scored. Only `train` is used, with the
  repo's own train/val split (coverage: `docs/COMPETITION_COMPLIANCE.md`);
* any mock-environment number is printed with `MOCK ENV / SYNTHETIC` next to it;
* checkpoints are written by the repo atomically, and synced to Drive through a staging directory
  plus a completion marker that lists a **size and sha256 for every file**; the previous generation
  is renamed to `step-<n>.prev-<k>` instead of being deleted, so a crash at any point leaves one
  complete, verifiable copy. A half-copied directory can never pass as resumable: resuming re-hashes
  every file the marker lists.
* section 9's continuation demonstration is **opt-in** (`RUN_RESUME_PROOF`, default `False`): with the
  default the cell only *inspects* the checkpoint, and enabling it extends the finished experiment by
  two optimizer steps (new `step-<n>` dirs, mirrored to Drive) instead of pretending to be read-only.
* the vocabulary behind every VRAM number is **resolved from verified metadata**
  (`lifemate.models.profiles`, or the model's own `config.json`) — never typed in. Qwen3.5-4B's text
  config declares 248320; the Qwen2-era 151936 under-estimated the fp32 logits term by 1.63x.


## 1. Environment setup — GPU, precision, dependencies, disk

Purpose: know exactly which GPU we got, whether it can do bf16 (**T4 cannot** — its sm_75 has no
bf16 kernels, so the notebook selects fp16 there and bf16 everywhere else), install only the
dependencies that are genuinely missing, and refuse to continue on a runtime that cannot hold a 4B
model. Cost: seconds of compute plus ~1-3 min of pip on a cold runtime (0.1-0.3 CU).

The repo's own estimate for *its default 2B model* is roughly 11-14 GB at `max_seq_length=8192`; for
the 4B model this notebook trains, section 2b derives the number per tier (~17.3 GB for 16-bit LoRA
at 8192 — **ESTIMATE**). The floor here is 14 GB of VRAM — T4 16 GB, L4 24 GB, A100 40 GB, A100
80 GB. Anything smaller stops with an actionable error instead of an OOM after forty minutes of
training.

**Dependencies are gated on VERSIONS, not on presence.** Cell 1c compares what is installed
against the floors `pyproject.toml` pins (`transformers>=5.22`, `peft>=0.21`, `accelerate>=0.33`,
`pyyaml>=6`) and prints a `package | installed | required | action` table, installing only what is
absent or too old. A presence-only check passed on Colab's preinstalled `transformers` 4.4x, and
the run then died at the first model load: `model_type qwen3_5` is unknown there, while
MiniCPM5-2B's `config.json` declares `transformers_version: 5.6.2`. Two deliberate exclusions:
`torch` is **checked, never installed** (Colab's build is coupled to the runtime's CUDA, so
replacing it breaks the runtime; below 2.15 the cell says to pick a newer runtime), and `numpy`'s
major version is never forced back (if an install moves it, the cell says so and prints the
revert command).

**fp16 warning, printed by this section whenever the GPU has no bf16 (T4).** fp16 training needs
dynamic loss scaling; the repo trainer supplies it with `torch.amp.GradScaler`
(`src/lifemate/training/trainer.py::_make_grad_scaler`), and that fp16+scaler path is **not verified
on a T4 here** — the scaler backs off and skips a step when it sees inf/NaN, so expect occasional
skipped steps. On a T4 the honest default is `TRAINING_METHOD="qlora"` (section 2b derives it).

This section also exports `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`. Long sequences
fragment the caching allocator; expandable segments let it reuse freed blocks instead of stranding
them. It is set in `os.environ` **before** any child process starts, because the variable is read
when a process creates its CUDA context — so `scripts/train.py` inherits it.


In [ ]:
# --- 1a. Probe the runtime: Colab?, GPU?, bf16?, disk -------------------------
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

try:
    import torch
except ImportError:  # a fresh non-Colab box may have no torch yet
    torch = None

GPU = {"available": False, "reason": "torch is not importable"}
PRECISION = "float16"
if torch is not None and torch.cuda.is_available():
    _props = torch.cuda.get_device_properties(0)
    _free, _total = torch.cuda.mem_get_info(0)
    GPU = {
        "available": True,
        "name": str(_props.name),
        "capability": f"{_props.major}.{_props.minor}",
        "vram_total_gb": round(_total / 1e9, 2),
        "vram_free_gb": round(_free / 1e9, 2),
        "bf16_supported": bool(torch.cuda.is_bf16_supported()),
        "cuda": torch.version.cuda,
        "torch": torch.__version__,
        "n_gpus": torch.cuda.device_count(),
    }
    # T4 (sm_75) has NO bf16: asking for it yields NaNs or a silent fp32 upcast.
    PRECISION = "bfloat16" if GPU["bf16_supported"] else "float16"

# Long sequences fragment the caching allocator; expandable_segments lets it reuse freed blocks
# instead of stranding them. Exported here, not inside the trainer: the variable is read when a
# process creates its CUDA context, so it must be in the environment BEFORE the child starts.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if PRECISION == "float16":
    print(
        "\n!! PRECISION=float16: this GPU reports no bf16 support (T4 is sm_75), so every autocast\n"
        "   region runs in fp16. fp16 needs dynamic loss scaling, which the repo trainer provides\n"
        "   via torch.amp.GradScaler (src/lifemate/training/trainer.py::_make_grad_scaler); when the\n"
        "   scaler detects inf/NaN it backs off and SKIPS that optimizer step. That fp16+scaler path\n"
        "   is NOT verified on a T4 here. On a T4 prefer TRAINING_METHOD=\"qlora\" in section 2: the\n"
        "   4-bit base frees ~6 GB of frozen weights, and that is what lets 8192 tokens fit at all.\n"
    )
print(f"allocator      : PYTORCH_CUDA_ALLOC_CONF={os.environ['PYTORCH_CUDA_ALLOC_CONF']} "
      f"(inherited by scripts/train.py)")

_root = "/content" if IN_COLAB else str(Path.home())
_disk = shutil.disk_usage(_root)
print(json.dumps(
    {
        "colab": IN_COLAB,
        "python": sys.version.split()[0],
        "gpu": GPU,
        "chosen_precision": PRECISION,
        "disk_free_gb": round(_disk.free / 1e9, 2),
        "disk_total_gb": round(_disk.total / 1e9, 2),
    },
    indent=2,
))


In [ ]:
# --- 1b. GPU suitability gate + readiness micro-test -------------------------
MIN_VRAM_GB = 14.0
problems = []

if not IN_COLAB:
    print("note: not running on Colab; continuing on this host's GPU if present")
if not GPU.get("available"):
    problems.append(
        "no CUDA GPU visible. Runtime -> Change runtime type -> GPU (T4/L4/A100), "
        "then re-run this cell. A CPU runtime cannot train a 4B model."
    )
else:
    if GPU["vram_total_gb"] < MIN_VRAM_GB:
        problems.append(
            f"{GPU['name']} exposes {GPU['vram_total_gb']} GB of VRAM, below the {MIN_VRAM_GB} GB "
            "floor for this notebook's 4B run. Use a T4/L4/A100 runtime (a 16 GB T4 is the minimum, "
            "and only with the qlora method section 2b derives)."
        )
    if float(GPU["capability"]) < 7.0:
        problems.append(
            f"compute capability {GPU['capability']} is below 7.0 (Volta); bf16/fp16 autocast "
            "kernels are unavailable. Use a T4 or newer."
        )
if _disk.free / 1e9 < 40:
    problems.append(
        f"only {round(_disk.free / 1e9, 1)} GB free on {_root}. Weights (~8.7 GB), the HF cache, "
        "checkpoints and logs need ~40 GB. Free space or mount Drive for checkpoints first."
    )
if problems:
    raise RuntimeError("Runtime is not suitable for this notebook:\n - " + "\n - ".join(problems))

GPU_TIER = "unknown"
if GPU["available"]:
    _gb = GPU["vram_total_gb"]
    GPU_TIER = (
        "A100-80GB" if _gb >= 70 else
        "A100-40GB" if _gb >= 35 else
        "L4-24GB" if _gb >= 22 else
        "T4-16GB"
    )
    # Prove the kernels we are about to rely on actually run on this device.
    _dt = torch.bfloat16 if PRECISION == "bfloat16" else torch.float16
    _a = torch.randn(64, 64, device="cuda", dtype=_dt)
    _c = (_a @ _a).float().sum().item()
    torch.cuda.synchronize()
    del _a
    torch.cuda.empty_cache()
    print(f"GPU tier      : {GPU_TIER} ({GPU['name']}, {GPU['vram_total_gb']} GB, sm_{GPU['capability'].replace('.', '')})")
    print(f"precision     : {PRECISION} (bf16_supported={GPU['bf16_supported']})")
    print(f"readiness test: 64x64 matmul on device OK (checksum {_c:.2f})")
else:
    print("GPU tier      : none")


In [ ]:
# --- 1c. Dependency gate: the FLOOR is the requirement, presence is not ---------
# Colab preinstalls some transformers, and it is usually NOT new enough. The old check here asked
# "is the module importable?" and answered yes for whatever was installed, so the first real model
# load died later on `model_type qwen3_5` (Qwen3.5) or on MiniCPM5-2B's declared
# transformers_version 5.6.2 -- an error that looks unrelated to dependencies. Every entry below is
# now checked as an ACTUAL VERSION against the floor in pyproject.toml [train].
#
# Colab's torch is deliberately NOT in the install list: it is built against the runtime's CUDA and
# replacing it breaks the runtime. torch is CHECKED here, and a too-old torch is reported as
# "pick a newer runtime" -- never as something to pip-install.
#
# numpy is neither installed nor pinned; pip can still pull a numpy downgrade in as a transitive
# dependency, so its MAJOR version is compared before/after and a change is shouted about with the
# exact revert command instead of being forced.
import importlib.metadata
from importlib.metadata import PackageNotFoundError

try:  # packaging ships with any modern Python (and with Colab)
    from packaging.requirements import Requirement

    VERSION_CHECK = "packaging.requirements.Requirement"
except ImportError:  # documented fallback: numeric-tuple compare, `>=` only
    VERSION_CHECK = "numeric-tuple fallback (>= only; a pre-release sorts below the floor)"
PACKAGING = VERSION_CHECK.startswith("packaging")

# distribution name -> import name, only where they differ. Explicit mapping; never string-mangled.
MODULE_NAMES = {"pyyaml": "yaml", "pillow": "PIL", "scikit-learn": "sklearn"}

# Floors copied from pyproject.toml [project.optional-dependencies].train, verified on the training
# box (torch 2.15.0+cu132 / transformers 5.22.2 / peft 0.21.2 / python 3.12.11).
REQUIRED = ["transformers>=5.22", "peft>=0.21", "accelerate>=0.33", "pyyaml>=6"]
TORCH_FLOOR = "2.15"          # checked, never installed
BITSANDBYTES_FLOOR = "0.43"   # QLoRA only; enforced in 3b


def dist_version(dist):
    """Installed version of the DISTRIBUTION `dist` (not a module name), or None when absent."""
    try:
        return importlib.metadata.version(dist)
    except PackageNotFoundError:
        return None


def _numeric_key(version):
    """Fallback compare key: leading numeric components only ('5.22.0rc1' -> (5, 22, 0))."""
    match = re.match(r"\s*(\d+(?:\.\d+)*)", str(version or ""))
    return tuple(int(part) for part in match.group(1).split(".")) if match else ()


def _split_spec(spec):
    """(distribution name, printable restriction) of a pip spec; imports nothing."""
    if PACKAGING:
        parsed = Requirement(spec)
        return parsed.name, (str(parsed.specifier) or "(any)")
    dist, sep, rest = spec.partition(">=")
    return dist.strip(), (">=" + rest.strip()) if sep else "(any)"


def requirement_ok(spec, version):
    """True when installed `version` satisfies `spec`. A pre-release does NOT meet a floor."""
    if PACKAGING:
        return version is not None and Requirement(spec).specifier.contains(version)
    if version is None:
        return False
    _, required = _split_spec(spec)
    if not required.startswith(">="):
        raise RuntimeError(
            f"without `packaging` only `>=` specs are supported, and {spec!r} is not one. "
            "Run `pip install packaging` (it is normally already present)."
        )
    return _numeric_key(version) >= _numeric_key(required[2:])


def dependency_state(specs):
    """One row per spec: what is installed, what is required, and the action needed."""
    rows = []
    for spec in specs:
        dist, required = _split_spec(spec)
        installed = dist_version(dist)
        ok = requirement_ok(spec, installed)
        rows.append({
            "package": dist,
            "installed": installed or "absent",
            "required": required,
            "action": "none" if ok else ("install" if installed is None else "upgrade"),
            "ok": ok,
        })
    return rows


def print_state(rows, title):
    print(title)
    print(f"  {'package':<14}| {'installed':<14}| {'required':<12}| action")
    for row in rows:
        print(f"  {row['package']:<14}| {row['installed']:<14}| {row['required']:<12}| {row['action']}")


def pip_install(*specs):
    print("installing:", ", ".join(specs), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *specs], check=True)


def check_requirements(specs):
    """Install only absent/too-old specs, then prove the floors hold. Returns the final rows."""
    numpy_before = dist_version("numpy")
    rows = dependency_state(specs)
    print_state(rows, f"dependency gate [{VERSION_CHECK}]:")
    stale = [spec for spec, row in zip(specs, rows) if not row["ok"]]
    if not stale:
        print("every floor met; nothing installed")
        return rows
    pip_install(*stale)
    rows = dependency_state(specs)
    print_state(rows, "after pip install:")
    below = [row for row in rows if not row["ok"]]
    if below:
        raise RuntimeError(
            "still below the floor after installing: "
            + ", ".join(f"{r['package']} {r['installed']} needs {r['required']}" for r in below)
            + ". On Colab that almost always means the session is still holding the OLD import: "
            "Runtime -> Restart session, then run again from section 1. Nothing was trained."
        )
    for row in rows:
        module = MODULE_NAMES.get(row["package"], row["package"].replace("-", "_"))
        loaded = sys.modules.get(module)
        if loaded is not None and str(getattr(loaded, "__version__", row["installed"])) != str(row["installed"]):
            print(
                f"!! {row['package']} {row['installed']} was installed, but this session already "
                f"imported {getattr(loaded, '__version__', '?')}. Restart the session before training, "
                "so the new version is the one that builds the model."
            )
    numpy_after = dist_version("numpy")
    if numpy_before and numpy_after and _numeric_key(numpy_after)[:1] < _numeric_key(numpy_before)[:1]:
        print(
            f"\n!! numpy changed MAJOR version {numpy_before} -> {numpy_after} as a dependency of that\n"
            "   install. Colab's preinstalled wheels were built against the old one, so unrelated\n"
            "   imports may now fail in this session. Restart the session first; if it is still\n"
            f'   broken, undo with: pip install "numpy=={numpy_before}".\n'
        )
    return rows


check_requirements(REQUIRED)

# torch: CHECKED, never installed. Colab's build is coupled to its CUDA runtime.
_torch_version = dist_version("torch")
if _torch_version is None:
    print(
        f"!! torch is absent; this notebook needs torch>={TORCH_FLOOR}. Nothing pip can fix here -- "
        "select a GPU runtime, which ships torch."
    )
elif not requirement_ok(f"torch>={TORCH_FLOOR}", _torch_version):
    print(
        f"\n!! torch {_torch_version} is below {TORCH_FLOOR}, the build this stack was verified on.\n"
        "   Do NOT `pip install torch` on Colab: its torch is compiled against the runtime's CUDA\n"
        "   and replacing it breaks the runtime. Switch to a newer Colab runtime image\n"
        "   (Runtime -> Change runtime type), or expect missing kernels/APIs at model load.\n"
    )
else:
    print(f"torch {_torch_version} >= {TORCH_FLOOR} OK")


## 2. Configuration — one cell, Colab `#@param` forms

Purpose: every knob that changes the run lives here, in one place, so a resumed run can be proven
to use the same numbers. `MAX_SEQ_LENGTH = 0` means "derive it from the measured token distribution"
(section 5 fills it in). Cost: none.

Rules the derived defaults follow (all **ESTIMATE**; nothing below was measured on Colab):

* `TRAINING_METHOD=lora` keeps the frozen base in bf16/fp16 and trains adapters — the repo's
  architecture-aware discovery picks the projections (including Qwen's hybrid linear-attention
  blocks), never a hardcoded Llama list;
* `qlora` puts the *base* in 4-bit NF4 so a T4 has room for a long context; the trained artifact
  is still a normal LoRA adapter;
* effective batch stays 8 sequences (`micro_batch * grad_accum`), which is what the shipped configs
  assume;
* T4 gets the smallest micro-batch because it has no bf16 and the least memory.

**Resume is decided in code, not by prose** (cell 6d applies it, section 8 carries it out):

| `RESUME_MODE` | meaning, enforced |
| --- | --- |
| `"auto"` (default) | discover the newest checkpoint of this run that **verifies** (marker sha256, model identity, `config_hash`, dataset hash, loadable `optimizer_state.pt`), Drive first then local, and resume it. If nothing verifies the notebook **stops** and lists every rejected candidate with its reason — it never starts a fresh run by accident. |
| `"fresh"` | a NEW experiment. If `checkpoints/<RUN_NAME>/step-*` already exists the run name is suffixed with a UTC stamp (`<run>-2026-11-14T101500Z`) and the rename is printed loudly; existing checkpoints are never overwritten or mixed into. |
| `"explicit"` | use `RESUME_FROM_CHECKPOINT` and refuse anything that fails verification. |

**Feature flags live in the next cell (2a)**, not here: they are a separate form so their defaults,
validation and fingerprints stay in one place. Every one defaults OFF, and a disabled feature costs
no GPU work and no dataset rebuild.

`SAVE_EVERY_N_STEPS` (default 50) is passed to `train.py` as `--save-steps/--eval-steps` **when this
checkout exposes them** (feature-detected against the script's own `--help`; a checkout without them
keeps its per-epoch strategy and the cell says so instead of failing). `DELETE_LOCAL_AFTER_VERIFIED_SYNC`
(default `False`) is a dry run unless enabled — see section 4 and cell 13.

`RUN_RESUME_PROOF` (default `False`) controls section 9: off, that cell only reads the checkpoint;
on, it launches a 2-step continuation that **mutates** the run. See section 7 of
`notebooks/README_colab.md`.

**The GPU tier decides the method, not only the sequence length** (cell 2b, all numbers **ESTIMATE**).
Qwen3.5-4B is a top-level `Qwen3_5ForConditionalGeneration` whose text side is ~4B parameters
(hidden 2560, 32 layers). Frozen fp16 weights are therefore ~8.8 GB; at 8192 tokens the activation
term adds ~5.4 GB, the logits term ~4.1 GB (bf16 `[seq, vocab]`, see below) and the cross-entropy
term ~1.0 GB (chunked) — **19.8 GB ESTIMATE** against ~14.9 GB usable on a T4 16 GB. So plain `lora`
at that length on `T4-16GB` is derived to `qlora` (**13.5 GB ESTIMATE**) with the arithmetic printed,
and plain `lora` is capped at **3840** there (**14.8 GB ESTIMATE**, ~100% of usable → "tight"); the
cap moved 4096 → 3840 because the bf16 logits tensor is now priced, and 4096 projected 15.1 GB.
L4 keeps `lora` at 8192 (**19.8 GB**) and drops to `qlora` at 12288 (**18.2 GB**); A100 40/80 keep
`lora` up to 16384 (**29.3 GB**) / 32768 (**48.2 GB**). Every one of those figures is recomputed by
`revalidate_tier_caps()` in cell 2b at run time, which raises if a cap its own arithmetic rejects is
still recommended. The vocabulary comes from verified metadata (Qwen3.5's text config declares
`vocab_size=248320`, *not* the Qwen2-era 151936, which under-estimated the logits term by 1.63x).

The CE term is priced for the path the trainer runs, and that path allocates **two** logits tensors:
with `training.loss_chunk_tokens` (shipped default 1024) the model is called without `labels`, so the
head returns full `[seq, vocab]` logits in the compute dtype (`seq * micro_batch * vocab * 2` bytes =
4.1 GB at 8192, held alive through `backward()`), and only `min(chunk, seq) * micro_batch * vocab * 4`
bytes of that tensor are upcast to fp32 per chunk (~1.0 GB). The notebook reports them as
`logits_compute_gb` and `logits_chunked_gb` and prices both. Unchunked at seq 8192 the fp32 temporary
alone would be 8.1 GB — which is why both are printed, with `logits_unchunked_gb` for contrast.

**The requested `MAX_SEQ_LENGTH` is never lowered silently.** When it cannot fit the tier, cell 2b
prints `!! MAX_SEQ_LENGTH <old> -> <new>` with the estimate that forced the clamp and the way to
keep the old value (a smaller sequence, or a bigger runtime). `MAX_SEQ_LENGTH = 0` means "take the
tier's recommendation".


In [ ]:
#@title 2. Run configuration { display-mode: "form" }
#@markdown Model + tuning
MODEL_ID = "Qwen/Qwen3.5-4B" #@param {type:"string"}
TRAINING_METHOD = "lora" #@param ["lora", "qlora"]
MAX_SEQ_LENGTH = 8192 #@param {type:"integer"}
MICRO_BATCH_SIZE = 1 #@param {type:"integer"}
GRADIENT_ACCUMULATION_STEPS = 8 #@param {type:"integer"}
NUM_EPOCHS = 3 #@param {type:"number"}
LEARNING_RATE = 0.0002 #@param {type:"number"}
#@markdown Storage + resume
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
RESUME_MODE = "auto" #@param ["fresh", "auto", "explicit"]
RESUME_FROM_CHECKPOINT = "" #@param {type:"string"}
SAVE_EVERY_N_STEPS = 50 #@param {type:"integer"}
DELETE_LOCAL_AFTER_VERIFIED_SYNC = False #@param {type:"boolean"}
MAX_STEPS = 0 #@param {type:"integer"}
SEED = 42 #@param {type:"integer"}
#@markdown Section 9: extend the finished run by 2 steps to prove resume (MUTATES the run's checkpoints)
RUN_RESUME_PROOF = False #@param {type:"boolean"}
#@markdown Repo pin and run identity (not part of the required list, but needed downstream)
BRANCH_OR_COMMIT = "main" #@param {type:"string"}
RUN_NAME = "qwen35-4b-lora" #@param {type:"string"}
DATASET_MODE = "auto" #@param ["auto", "rebuild"]

if int(MAX_SEQ_LENGTH) != 0 and not (1024 <= int(MAX_SEQ_LENGTH) <= 32768):
    raise ValueError(f"MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} outside the supported 1024..32768 range")
if TRAINING_METHOD not in ("lora", "qlora"):
    raise ValueError(f"TRAINING_METHOD must be lora or qlora, got {TRAINING_METHOD!r}")
if int(MICRO_BATCH_SIZE) < 1 or int(GRADIENT_ACCUMULATION_STEPS) < 1:
    raise ValueError("MICRO_BATCH_SIZE and GRADIENT_ACCUMULATION_STEPS must be >= 1")
# Resume semantics are enforced in code (section 8 calls resolve_resume_target), not by prose:
#   fresh    -> never overwrite an experiment: a populated run name is renamed with a UTC stamp;
#   auto     -> resume the newest checkpoint that VERIFIES, else stop (never silently start over);
#   explicit -> use RESUME_FROM_CHECKPOINT and refuse anything that fails verification.
if RESUME_MODE not in ("fresh", "auto", "explicit"):
    raise ValueError(f'RESUME_MODE must be "fresh", "auto" or "explicit", got {RESUME_MODE!r}')
if RESUME_MODE == "explicit" and not str(RESUME_FROM_CHECKPOINT).strip():
    raise ValueError('RESUME_MODE="explicit" needs RESUME_FROM_CHECKPOINT=<path to a step-<n> dir>')
if int(SAVE_EVERY_N_STEPS) < 1:
    raise ValueError(f"SAVE_EVERY_N_STEPS={SAVE_EVERY_N_STEPS} must be >= 1 optimizer step")
print(
    f"model={MODEL_ID} method={TRAINING_METHOD} (requested; 2b may override) seq={MAX_SEQ_LENGTH or 'auto'} "
    f"micro_batch={MICRO_BATCH_SIZE} accum={GRADIENT_ACCUMULATION_STEPS} "
    f"epochs={NUM_EPOCHS} lr={LEARNING_RATE} max_steps={MAX_STEPS or 'full'} seed={SEED} "
    f"resume_mode={RESUME_MODE} save_every={SAVE_EVERY_N_STEPS} "
    f"delete_local_after_sync={DELETE_LOCAL_AFTER_VERIFIED_SYNC} resume_proof={RUN_RESUME_PROOF}"
)


In [ ]:
#@title 2a. Feature flags — the training switchboard { display-mode: "form" }
#@markdown Every flag defaults **OFF**: a disabled feature costs no GPU work and no dataset rebuild,
#@markdown and the generated commands are byte-identical to a feature-free run.
#@markdown
#@markdown `NOISY_ASR_RATIO` is only meaningful while `NOISY_ASR_TRAINING` is on. A ratio set while the flag is off is **INERT**: accepted, printed as inert, and it moves no fingerprint.
NOISY_ASR_TRAINING = False #@param {type:"boolean"}
NOISY_ASR_RATIO = 0.25 #@param {type:"number"}
NOISY_ASR_TRANSCRIPTS = "" #@param {type:"string"}
SLOT_MASKING = False #@param {type:"boolean"}
TOOL_DROPOUT = False #@param {type:"boolean"}
ASR_NBEST = False #@param {type:"boolean"}
NBEST_TRAINING_FORMAT = False #@param {type:"boolean"}
NBEST_TRANSCRIPTS = "" #@param {type:"string"}
PROMPT_ITN = False #@param {type:"boolean"}
AUDIO_VAD = False #@param {type:"boolean"}
AUDIO_RMS_NORMALIZATION = False #@param {type:"boolean"}
AUGMENTATION_SEED = 42 #@param {type:"integer"}

# --- 2a-i. Param -> the REAL config path. No second flag list lives in the notebook. ----------
# `lifemate.features.config.FeatureConfig` IS the state; `lifemate.features.registry.REGISTRY` owns
# the id <-> field mapping and the spelling of every id.
FEATURE_FLAGS = {
    "NOISY_ASR_TRAINING": ("training", "noisy_asr_training_enabled"),
    "SLOT_MASKING": ("training", "slot_masking_enabled"),
    "TOOL_DROPOUT": ("training", "tool_dropout_enabled"),
    # The DATASET builder's `--nbest-training-format` is the same technique the registry calls
    # `prompt.nbest_format` (whose dependency is `asr.nbest`), so the param drives that path.
    "NBEST_TRAINING_FORMAT": ("prompt", "nbest_prompt_enabled"),
    "PROMPT_ITN": ("prompt", "itn_enabled"),
    "AUDIO_VAD": ("audio", "vad_enabled"),
    "AUDIO_RMS_NORMALIZATION": ("audio", "rms_normalization_enabled"),
    "ASR_NBEST": ("asr", "nbest_enabled"),
}

#: Scopes that can change what a TRAINING run sees; `scripts/train.py` refuses the rest.
TRAINING_SCOPES_HERE = ("training", "both")

def feature_params():
    """The `#@param` values as one dict, so every f-string below reads the same source."""
    return {
        "NOISY_ASR_TRAINING": bool(NOISY_ASR_TRAINING),
        "NOISY_ASR_RATIO": float(NOISY_ASR_RATIO),
        "NOISY_ASR_TRANSCRIPTS": str(NOISY_ASR_TRANSCRIPTS).strip(),
        "SLOT_MASKING": bool(SLOT_MASKING),
        "TOOL_DROPOUT": bool(TOOL_DROPOUT),
        "ASR_NBEST": bool(ASR_NBEST),
        "NBEST_TRAINING_FORMAT": bool(NBEST_TRAINING_FORMAT),
        "NBEST_TRANSCRIPTS": str(NBEST_TRANSCRIPTS).strip(),
        "PROMPT_ITN": bool(PROMPT_ITN),
        "AUDIO_VAD": bool(AUDIO_VAD),
        "AUDIO_RMS_NORMALIZATION": bool(AUDIO_RMS_NORMALIZATION),
        "AUGMENTATION_SEED": int(AUGMENTATION_SEED),
    }

def _fmt_number(value):
    """`0.25` -> "0.25", `1.0` -> "1": argv text that parses back to the same float."""
    return f"{float(value):g}"

def feature_param_notes(params):
    """INERT / informational notes. A disabled feature's parameter is stated, never ignored."""
    notes = []
    if not params["NOISY_ASR_TRAINING"] and float(params["NOISY_ASR_RATIO"]) != 0.25:
        notes.append(
            f"INERT: NOISY_ASR_RATIO={_fmt_number(params['NOISY_ASR_RATIO'])} with "
            "NOISY_ASR_TRAINING=False — a disabled feature's parameters change nothing and are "
            "excluded from both feature fingerprints, so this value neither fails nor counts"
        )
    if not params["NBEST_TRAINING_FORMAT"] and params["NBEST_TRANSCRIPTS"]:
        notes.append("INERT: NBEST_TRANSCRIPTS is set but NBEST_TRAINING_FORMAT=False")
    return notes

def check_feature_params(params):
    """Guards that must name the COLAB control; the registry validates everything else.

    Only an ENABLED feature's parameters are checked. A ratio outside (0, 1] has to fail HERE:
    letting it reach `TrainingFeatureConfig` would refuse it inside the training subprocess, after
    the weights were already loaded.
    """
    if params["NOISY_ASR_TRAINING"]:
        ratio = float(params["NOISY_ASR_RATIO"])
        if not 0.0 < ratio <= 1.0:
            raise ValueError(
                f"NOISY_ASR_RATIO={_fmt_number(ratio)} is outside (0, 1] while "
                "NOISY_ASR_TRAINING=True. It is the share of samples whose gold transcript is "
                "replaced by ASR output: set 0 < NOISY_ASR_RATIO <= 1, or set "
                "NOISY_ASR_TRAINING=False."
            )
        if not params["NOISY_ASR_TRANSCRIPTS"]:
            raise ValueError(
                "NOISY_ASR_TRAINING=True needs NOISY_ASR_TRANSCRIPTS=<path to the transcript "
                "JSONL>. scripts/prepare_dataset.py refuses to invent ASR text "
                "(`<feature> is enabled but no transcript file was given`): build the file with "
                "`python3 scripts/transcribe_split.py --transcript-records ...` and put its path "
                "here. Nothing was prepared or trained."
            )
    if params["NBEST_TRAINING_FORMAT"]:
        if not params["ASR_NBEST"]:
            raise ValueError(
                "NBEST_TRAINING_FORMAT=True requires ASR_NBEST=True: the numbered N-best prompt is "
                "rendered from the k-best hypotheses that only asr.nbest produces (registry "
                "dependency prompt.nbest_format -> asr.nbest). Enable ASR_NBEST, or set "
                "NBEST_TRAINING_FORMAT=False."
            )
        if not params["NBEST_TRANSCRIPTS"]:
            raise ValueError(
                "NBEST_TRAINING_FORMAT=True needs NBEST_TRANSCRIPTS=<path to the N-best transcript "
                "JSONL> (`scripts/transcribe_split.py --transcript-records` writes it). Nothing was "
                "prepared or trained."
            )
    if params["AUGMENTATION_SEED"] < 0:
        raise ValueError(
            f"AUGMENTATION_SEED={params['AUGMENTATION_SEED']} must be >= 0 (it seeds the augmentation "
            "selection, so a negative value is not reproducible)"
        )
    return True

def build_feature_config(params=None):
    """ONE `FeatureConfig` from the notebook params, built with the real dataclasses.

    Not a dict and not a notebook-local copy: the sections and fields are the objects
    `src/lifemate/features` owns, so `enabled_features`, both fingerprints and every script see
    exactly this state.
    """
    from lifemate.features.config import FeatureConfig  # noqa: PLC0415 - repo may not be on sys.path yet

    params = feature_params() if params is None else params
    check_feature_params(params)
    cfg = FeatureConfig()
    for name, (section, field_name) in FEATURE_FLAGS.items():
        setattr(getattr(cfg, section), field_name, bool(params.get(name, False)))
    # Real TrainingFeatureConfig fields that are not registry ids (no `--enable-feature` spelling):
    # the dataset builder consumes them, and the config stays self-describing.
    cfg.training.nbest_training_format_enabled = bool(params.get("NBEST_TRAINING_FORMAT", False))
    cfg.training.noisy_asr_ratio = float(params.get("NOISY_ASR_RATIO", 0.25))
    cfg.training.augmentation_seed = int(params.get("AUGMENTATION_SEED", 42))
    return cfg

def validate_feature_config(cfg):
    """`REGISTRY.validate_or_raise` — dependencies, availability and every per-param rule."""
    from lifemate.features.registry import REGISTRY  # noqa: PLC0415

    return REGISTRY.validate_or_raise(cfg)

def enabled_training_ids(cfg):
    """Enabled ids whose registry scope is ``training`` or ``both``, in registry order.

    Inference-only ids are deliberately excluded: `scripts/train.py` refuses them, and passing one
    would promise an experiment the run cannot perform.
    """
    from lifemate.features.config import enabled_features  # noqa: PLC0415
    from lifemate.features.registry import REGISTRY  # noqa: PLC0415

    enabled = set(enabled_features(cfg))
    return [feature_id for feature_id in REGISTRY.ids
            if feature_id in enabled and REGISTRY.spec(feature_id).scope in TRAINING_SCOPES_HERE]

def feature_train_args(cfg, *, override_supported=False):
    """Extra argv for `scripts/train.py`; `[]` when every flag is off.

    ``--enable-feature <id>`` is the script's own switchboard. The numeric parameters are NOT
    duplicated here unless the checkout accepts ``--override``: they always reach the run through
    the generated YAML, which `load_config` applies with the same dotted-override engine.
    """
    if cfg is None:
        raise RuntimeError(
            "Feature flags were not initialised (cell 2a did not run). Run 2a, then 3c, before "
            "building a training command."
        )
    ids = enabled_training_ids(cfg)
    args = []
    for feature_id in ids:
        args += ["--enable-feature", feature_id]
    if override_supported:
        enabled_field = ("training", "noisy_asr_ratio")
        if cfg.training.noisy_asr_training_enabled:
            args += ["--override", "features.%s.%s=%s" % (
                enabled_field[0], enabled_field[1], _fmt_number(cfg.training.noisy_asr_ratio))]
    return args

def feature_config_overrides(cfg):
    """`features.*` dotted overrides for the generated YAML; `{}` when every flag is off."""
    if cfg is None:
        return {}
    overrides = {f"features.{section}.{field_name}": bool(getattr(getattr(cfg, section), field_name))
                 for section, field_name in FEATURE_FLAGS.values()}
    overrides["features.training.nbest_training_format_enabled"] = bool(
        cfg.training.nbest_training_format_enabled)
    if not any(value for value in overrides.values()):
        return {}
    overrides["features.training.noisy_asr_ratio"] = float(cfg.training.noisy_asr_ratio)
    overrides["features.training.augmentation_seed"] = int(cfg.training.augmentation_seed)
    return overrides

def feature_prepare_args(cfg, params=None):
    """Extra argv for `scripts/prepare_dataset.py`; `[]` when every flag is off.

    Only an ENABLED feature contributes a flag, so the builder's own "is anything active?" gate
    stays the single decision point and an all-off prepare command is byte-identical to the
    feature-free one.
    """
    if cfg is None:
        raise RuntimeError(
            "Feature flags were not initialised (cell 2a did not run); refusing to guess a "
            "dataset-augmentation command."
        )
    params = feature_params() if params is None else params
    args = []
    if cfg.training.noisy_asr_training_enabled:
        args += ["--noisy-asr-enabled",
                 "--noisy-asr-ratio", _fmt_number(cfg.training.noisy_asr_ratio),
                 "--noisy-asr-transcripts", params["NOISY_ASR_TRANSCRIPTS"]]
    if cfg.training.slot_masking_enabled:
        args += ["--slot-masking-enabled"]
    if cfg.training.tool_dropout_enabled:
        args += ["--tool-dropout-enabled"]
    if cfg.training.nbest_training_format_enabled:
        args += ["--nbest-training-format", "--nbest-transcripts", params["NBEST_TRANSCRIPTS"]]
    if args:
        args += ["--augmentation-seed", str(int(cfg.training.augmentation_seed))]
    return args

def feature_resume_identity(cfg):
    """`{"training_feature_fingerprint", "training_features"}` — the trainer's own two values.

    Computed with the same functions the trainer stores in `training_state.json`, so a resume
    comparison is never a second implementation that can disagree.
    """
    if cfg is None:
        return {}
    from lifemate.features.config import training_feature_fingerprint  # noqa: PLC0415

    return {"training_feature_fingerprint": training_feature_fingerprint(cfg),
            "training_features": enabled_training_ids(cfg)}

FEATURE_CONFIG, FEATURE_REPORT, FEATURE_NOTES = None, None, []

def run_feature_section():
    """Build, validate, report. Returns `(cfg, report)`; raises on an invalid combination."""
    global FEATURE_CONFIG, FEATURE_REPORT, FEATURE_NOTES
    from lifemate.config import AsrConfig  # noqa: PLC0415
    from lifemate.features.report import feature_report_dict, format_feature_report  # noqa: PLC0415

    params = feature_params()
    FEATURE_NOTES = feature_param_notes(params)
    cfg = build_feature_config(params)
    # The registry owns dependencies, conflicts, availability and the per-param rules. An invalid
    # combination stops HERE — before a dataset rebuild and before any GPU work.
    validate_feature_config(cfg)
    FEATURE_CONFIG = cfg
    FEATURE_REPORT = feature_report_dict(cfg, model_id=MODEL_ID, asr_id=AsrConfig().model_id)
    print(format_feature_report(cfg, model_id=MODEL_ID, asr_id=AsrConfig().model_id))
    print(f"training ids   : {enabled_training_ids(cfg) or 'none'} "
          "(scope training|both only; inference-only ids are refused by scripts/train.py)")
    print(f"fingerprints   : all {FEATURE_REPORT['fingerprint']} | training "
          f"{FEATURE_REPORT['training_fingerprint']}")
    for note in FEATURE_NOTES:
        print(f"feature note   : {note}")
    prepared = feature_prepare_args(cfg, params)
    print(f"prepare args   : {' '.join(prepared) or '(none: byte-identical to a feature-free run)'}")
    trained = feature_train_args(cfg, override_supported="--override" in
                                 globals().get("TRAIN_FLAGS", set()))
    print(f"train args     : {' '.join(trained) or '(none)'}")
    return FEATURE_CONFIG, FEATURE_REPORT

try:
    import lifemate as _lifemate_feature_probe  # noqa: F401 - availability probe only
except ModuleNotFoundError:
    print("section 2a deferred: the switchboard lives in lifemate.features and the checkout is not "
          "importable yet. Section 3c builds and validates it as soon as the repo is on sys.path.")
else:
    FEATURE_CONFIG, FEATURE_REPORT = run_feature_section()


In [ ]:
# --- 2b. Derivation: the GPU TIER decides the METHOD, not only the sequence length ----
# Architecture facts behind every number below — Qwen3.5-4B is a top-level
# Qwen3_5ForConditionalGeneration (text side ~4B params, hidden 2560, 32 layers) plus a vision tower.
BASE_PARAMS_B = 4.4
HIDDEN = 2560
LAYERS = 32

TIER_VRAM_GB = {"T4-16GB": 16.0, "L4-24GB": 24.0, "A100-40GB": 40.0, "A100-80GB": 80.0, "unknown": 16.0}
USABLE_FRACTION = 0.93        # driver + CUDA context + fragmentation reserve
QLORA_BYTES_PER_PARAM = 0.55  # NF4 plus its absmax/quantile doubles
ACT_FACTOR = 4.0              # activation copies kept per layer boundary under gradient checkpointing
RESERVE_GB = 0.6              # allocator slack, cuBLAS workspaces
SEQ_CEILING = 32768

# Hard method ceilings per tier. They are the ceiling, not the finding: the arithmetic below already
# refuses 16-bit LoRA above 3840 on a 16 GB card (4096 projects 15.1 GB against 14.9 GB usable, once
# BOTH logits tensors are priced), and each cap here is the largest 256-step sequence that still fits:
# revalidate_tier_caps() re-derives every one of them and raises when a cap does not fit, so a stale
# cap can never be recommended. The caps also cover what the arithmetic cannot see, such as
# fragmentation growth over a long run and a second process holding a slice of the card.
TIER_METHOD_CAP = {
    "T4-16GB":   {"lora": 3840, "qlora": 8192},
    "L4-24GB":   {"lora": 8192, "qlora": 12288},
    "A100-40GB": {"lora": 16384, "qlora": 16384},
    "A100-80GB": {"lora": 32768, "qlora": 32768},
    "unknown":   {"lora": 3840, "qlora": 8192},
}

# Resolved from verified metadata by run_section_2b() below; never a literal here.
VOCAB, VOCAB_SOURCE, LOSS_CHUNK_TOKENS = None, "unresolved", 0


def resolve_vocab(model_id=MODEL_ID):
    """(vocab, source) from VERIFIED metadata. Raises instead of guessing.

    Precedence: (a) a readable ``config.json`` for a local model path — including
    ``text_config.vocab_size`` for a multimodal wrapper; (b) the repo's own profile registry
    (``resolve_profile(...).preflight_vocab``, which quotes the released text config);
    (c) raise. A guessed vocabulary silently misprices the largest allocation of a training step:
    151936 is the Qwen2/Qwen2.5 number and under-counts Qwen3.5's 248320 by 1.63x.
    """
    from lifemate.models.profiles import resolve_profile  # noqa: PLC0415 - resolved lazily

    candidate = Path(str(model_id))
    config_file = candidate / "config.json" if candidate.is_dir() else None
    if config_file is not None and config_file.is_file():
        payload = json.loads(config_file.read_text(encoding="utf-8"))
        blocks = [payload] + ([payload["text_config"]] if isinstance(payload.get("text_config"), dict) else [])
        for block in blocks:
            value = int(block.get("vocab_size") or 0)
            if value > 0:
                return value, f"local config.json {config_file} vocab_size={value}"
    profile = resolve_profile(model_id)
    if int(profile.preflight_vocab) > 0:
        return int(profile.preflight_vocab), (
            f"lifemate.models.profiles.{profile.name}.preflight_vocab={profile.preflight_vocab} "
            f"({profile.preflight_source})"
        )
    raise RuntimeError(
        f"cannot resolve the vocabulary of {model_id!r} from verified metadata: no readable local "
        "config.json and no repo profile carries preflight_vocab. The logits term is priced from it, "
        "so this estimator refuses to run on a guess. Point MODEL_ID at the local model directory, "
        "or add the family to src/lifemate/models/profiles.py."
    )


def next_multiple(value, multiple=256):
    """Round ``value`` up to the next ``multiple`` (never below one multiple)."""
    return max(multiple, -(-int(value) // multiple) * multiple)


def compute_bytes_per_param(precision=None):
    """Width of the logits the head returns: 4 for fp32, else 2 (bf16/fp16).

    The compute dtype is what the ``[seq, vocab]`` head output costs, so an fp32 run must not be
    priced at bf16 width. ``precision`` defaults to the notebook's own ``PRECISION``.
    """
    try:
        value = PRECISION if precision is None else precision  # noqa: F821 - set by section 1
    except NameError:
        value = "bfloat16"
    return 4 if str(value).lower() in ("float32", "fp32") else 2

def estimate_vram_gb(gpu_tier, method, seq_len, micro_batch=1, gradient_checkpointing=True,
                     vocab=None, loss_chunk_tokens=None, bytes_per_param=None):
    """Projected VRAM for one training step. ESTIMATE — a heuristic, never a measurement.

    The shipped chunked-CE path allocates TWO logits tensors and both are priced here, because
    pricing one of them under-counts the step by ``seq/bpp`` relative to the chunk:

    ``logits_compute_gb``
        the full causal-LM head output ``[micro_batch, seq, vocab]`` in the COMPUTE dtype. With
        ``loss_chunk_tokens > 0`` the trainer calls the model WITHOUT ``labels``, so the head
        returns logits instead of a loss, and that tensor stays alive through ``backward()``
        because the chunked CE reads slices of it: ``seq * micro_batch * vocab * bpp``.
        ``bytes_per_param`` is the compute dtype width — 2 for bf16/fp16, 4 for fp32.
    ``logits_chunked_gb``
        the fp32 upcast of ONE slice, ``min(chunk, seq) * micro_batch * vocab * 4``; the only
        logits term an estimator has, unless it isolates this name from the one above.
    ``logits_unchunked_gb``
        contrast only: the whole-sequence fp32 upcast nobody runs.

    ``logits_gb`` is ``logits_compute_gb + logits_chunked_gb`` — the two terms that actually exist.
    """
    vocab = VOCAB if vocab is None else int(vocab)
    if not vocab:
        raise RuntimeError(
            "no vocabulary resolved: the cross-entropy logits term cannot be priced without one. "
            "Resolve it from a local config.json or the repo profile registry (resolve_vocab); "
            "never guess a vocab_size."
        )
    chunk = LOSS_CHUNK_TOKENS if loss_chunk_tokens is None else int(loss_chunk_tokens)
    seq_len, micro_batch = int(seq_len), int(micro_batch)
    weights = BASE_PARAMS_B * (QLORA_BYTES_PER_PARAM if method == "qlora" else 2.0)
    acts = (LAYERS * seq_len * HIDDEN * 2 * micro_batch
            * (ACT_FACTOR if gradient_checkpointing else ACT_FACTOR * 4) / 1e9)
    bpp = compute_bytes_per_param() if bytes_per_param is None else int(bytes_per_param)
    logits_compute = seq_len * micro_batch * vocab * bpp / 1e9
    logits_chunked = (min(chunk, seq_len) if chunk > 0 else seq_len) * micro_batch * vocab * 4 / 1e9
    logits_unchunked = seq_len * micro_batch * vocab * 4 / 1e9
    usable = TIER_VRAM_GB.get(gpu_tier, 16.0) * USABLE_FRACTION
    total = weights + acts + logits_compute + logits_chunked + RESERVE_GB
    return {
        "weights_gb": round(weights, 2),
        "activations_gb": round(acts, 2),
        "logits_gb": round(logits_compute + logits_chunked, 2),
        "logits_compute_gb": round(logits_compute, 2),
        "logits_chunked_gb": round(logits_chunked, 2),
        "logits_unchunked_gb": round(logits_unchunked, 2),
        "compute_dtype": "fp32" if bpp == 4 else "bf16/fp16",
        "bytes_per_param": bpp,
        "logits_chunk_tokens": min(chunk, seq_len) if chunk > 0 else seq_len,
        "reserve_gb": RESERVE_GB,
        "total_gb": round(total, 2),
        "usable_gb": round(usable, 2),
        "fraction_of_usable": round(total / usable, 3),
        "fits": total <= usable,
        "vocab": vocab,
        "vocab_source": VOCAB_SOURCE if vocab == VOCAB else "caller-supplied",
        "source": "ESTIMATE",
    }


def seq_cap(gpu_tier, method, micro_batch=1):
    """Longest sequence whose ESTIMATE still fits the tier, also bounded by TIER_METHOD_CAP."""
    hard = TIER_METHOD_CAP.get(gpu_tier, TIER_METHOD_CAP["unknown"]).get(method, 4096)
    best = 1024
    for seq in range(1024, min(hard, SEQ_CEILING) + 1, 256):
        if estimate_vram_gb(gpu_tier, method, seq, micro_batch)["fits"]:
            best = seq
        else:
            break
    return min(best, hard)


def choose_method(gpu_tier, method, seq_wanted, micro_batch=1):
    """Tier-driven method decision; returns ``(method, notes)``.

    A T4 cannot hold 4B fp16 weights (8.8 GB) *plus* activations *plus* the logits term at 8192, so
    plain LoRA above the tier's LoRA cap becomes qlora instead of being launched into an OOM. Below
    that cap the user's explicit choice stands.
    """
    notes = []
    other = "qlora" if method == "lora" else "lora"
    est_dtype = "fp32" if compute_bytes_per_param() == 4 else "bf16/fp16"
    cap_here = seq_cap(gpu_tier, method, micro_batch)
    if int(seq_wanted) <= cap_here:
        return method, notes
    cap_other = seq_cap(gpu_tier, other, micro_batch)
    here = estimate_vram_gb(gpu_tier, method, seq_wanted, micro_batch)
    if cap_other >= int(seq_wanted):
        there = estimate_vram_gb(gpu_tier, other, seq_wanted, micro_batch)
        notes.append(
            f"METHOD CHANGED {method} -> {other}: {method} at seq {seq_wanted} needs "
            f"{here['total_gb']} GB but {gpu_tier} has {here['usable_gb']} GB usable "
            f"(ESTIMATE: weights {here['weights_gb']} + activations {here['activations_gb']} + "
            f"lg_compute ({est_dtype}) {here['logits_compute_gb']} + lg_fp32_chunk "
            f"{here['logits_chunked_gb']} + reserve {here['reserve_gb']} GB), so it would OOM before "
            f"the first optimizer step. {other} at the same seq needs {there['total_gb']} GB. "
            f"Set MAX_SEQ_LENGTH <= {cap_here} if you really want {method} on {gpu_tier}."
        )
        return other, notes
    notes.append(
        f"neither {method} nor {other} fits seq {seq_wanted} on {gpu_tier}; keeping {method} and "
        f"clamping to its cap {cap_here}"
    )
    return method, notes


def derive_run_config(gpu_tier, method, token_stats=None, max_seq_length_param=0, micro_batch_param=1,
                      accum_param=8, epochs_param=3, lr_param=2e-4):
    """Concrete, printable config. Every number is an ESTIMATE unless ``token_stats`` is measured.

    Sequence policy, so the user's explicit value is never downgraded silently: the requested
    ``MAX_SEQ_LENGTH`` is used as-is when it fits. When it cannot fit the tier the decision prints a
    ``!! MAX_SEQ_LENGTH`` line naming old -> new, the estimate behind it and how to keep the old
    value (smaller sequence, or a bigger tier). ``MAX_SEQ_LENGTH = 0`` means "take the recommendation".
    """
    notes = []
    requested_method = method
    micro = int(micro_batch_param)
    requested = int(max_seq_length_param)
    wanted = requested if requested > 0 else 8192
    method, method_notes = choose_method(gpu_tier, method, wanted, micro)
    notes.extend(method_notes)
    cap = seq_cap(gpu_tier, method, micro)

    if token_stats and token_stats.get("p99"):
        recommended = min(cap, next_multiple(token_stats["p99"] + 256))
        notes.append(f"seq recommended from measured p99={token_stats['p99']} -> {recommended} "
                     f"(tier cap for {method} = {cap})")
    else:
        recommended = min(cap, 8192)
        notes.append(f"seq from the tier only (no measurement yet) -> {recommended} "
                     f"(tier cap for {method} = {cap})")

    seq = requested if requested > 0 else recommended
    clamped = False
    if seq > cap:
        over = estimate_vram_gb(gpu_tier, method, seq, micro)
        notes.append(
            f"!! MAX_SEQ_LENGTH {seq} -> {cap}: {seq} does not fit {gpu_tier} with {method} "
            f"({over['total_gb']} GB vs {over['usable_gb']} GB usable, ESTIMATE). The requested value "
            f"is NOT used; {cap} is. Raise the tier, or lower MAX_SEQ_LENGTH yourself."
        )
        seq = cap
        clamped = True

    estimate = estimate_vram_gb(gpu_tier, method, seq, micro)
    if not estimate["fits"]:
        notes.append(f"{method} at seq {seq} still projects {estimate['total_gb']} GB: expect an OOM")
    elif estimate["fraction_of_usable"] > 0.85:
        notes.append(f"tight: {estimate['fraction_of_usable']:.0%} of usable VRAM ({method}, seq {seq}); "
                     f"section 7 measures the truth before any long run")
    if micro > 1 and seq > 4096:
        notes.append(f"micro_batch {micro} at seq {seq} is very likely too large for {gpu_tier}")
    accum = int(accum_param)
    if micro * accum != 8:
        notes.append(f"effective batch is {micro * accum} sequences, not the 8 the shipped configs assume")

    return {
        "gpu_tier": gpu_tier,
        "method_requested": requested_method,
        "method": method,                 # lora | qlora; both train LoRA adapters
        "quantized_base": method == "qlora",
        "max_seq_length_requested": requested,
        "max_seq_length": seq,
        "seq_clamped": clamped,
        "seq_cap": cap,
        "micro_batch": micro,
        "grad_accum": accum,
        "effective_batch": micro * accum,
        "epochs": float(epochs_param),
        "learning_rate": float(lr_param),
        "precision": PRECISION,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "gradient_checkpointing": True,
        "vram_estimate": estimate,
        "notes": notes,
        "source": "measured" if token_stats and token_stats.get("method") == "tokenizer" else "estimate",
    }


def print_derived(d):
    label = "measured token lengths" if d["source"] == "measured" else "ESTIMATE, no local dataset yet"
    print(f"Derived effective configuration ({label})")
    for key in ("gpu_tier", "method", "quantized_base", "max_seq_length", "micro_batch", "grad_accum",
                "effective_batch", "epochs", "learning_rate", "precision", "lora_r", "lora_alpha",
                "lora_dropout", "gradient_checkpointing"):
        print(f"  {key:<22}= {d[key]}")
    est = d["vram_estimate"]
    print(f"  {'vram ESTIMATE':<22}= {est['total_gb']} GB of {est['usable_gb']} GB usable "
          f"[weights {est['weights_gb']} + activations {est['activations_gb']} + "
          f"lg_compute ({est['compute_dtype']}) {est['logits_compute_gb']} GB "
          f"(full [seq, vocab] head output held through backward) + "
          f"lg_fp32_chunk {est['logits_chunked_gb']} GB ({est['logits_chunk_tokens']} tokens upcast) + "
          f"lg_unchunked {est['logits_unchunked_gb']} GB (contrast only, not allocated) + "
          f"reserve {est['reserve_gb']} GB] -> "
          f"{'fits' if est['fits'] else 'DOES NOT FIT'} ({est['fraction_of_usable']:.0%}) "
          f"| vocab {est['vocab']} from {est['vocab_source']}")
    if d["method"] != d["method_requested"]:
        print(f"  !! method          {d['method_requested']} -> {d['method']} (reason in the notes)")
    if d["seq_clamped"]:
        print(f"  !! max_seq_length  {d['max_seq_length_requested']} -> {d['max_seq_length']} "
              f"(reason in the notes)")
    for note in d["notes"]:
        print(f"  note: {note}")


def revalidate_tier_caps():
    """Re-derive every (tier, method) cap here, and compare with the superseded 151936 estimate.

    Nothing is carried over from before: the table is computed at run time. A cap the estimator
    says does NOT fit is returned as ``invalid_caps`` (the caller raises) so the notebook can never
    recommend a configuration its own arithmetic overflows.
    """
    print(f"\ntier/method revalidation (ESTIMATE; vocab {VOCAB} from {VOCAB_SOURCE}; "
          f"loss_chunk_tokens={LOSS_CHUNK_TOKENS})")
    print(f"  {'tier':<11} {'method':<6} {'seq':>6} {'weights':>8} {'acts':>6} {'lg_compute':>10} "
          f"{'lg_chunk':>9} {'lg_unchunk':>11} {'total':>7} {'usable':>7}  fits")
    rows, changed = [], []
    for tier, caps in TIER_METHOD_CAP.items():
        for method, seq in caps.items():
            est = estimate_vram_gb(tier, method, seq)
            # Superseded estimator: vocab hardcoded 151936, ONLY an fp32 chunk-priced CE term,
            # so it priced neither the compute-dtype head output nor the real sequence width.
            old_total = (est["weights_gb"] + est["activations_gb"]
                         + min(LOSS_CHUNK_TOKENS or seq, seq) * 151936 * 4 / 1e9 + RESERVE_GB)
            old_fits = old_total <= est["usable_gb"]
            est["old_total_gb"] = round(old_total, 2)
            est["fit_changed"] = old_fits != est["fits"]
            rows.append({"tier": tier, "method": method, "seq": seq, **est})
            print(f"  {tier:<11} {method:<6} {seq:>6} {est['weights_gb']:>8} "
                  f"{est['activations_gb']:>6} {est['logits_compute_gb']:>10} "
                  f"{est['logits_chunked_gb']:>9} "
                  f"{est['logits_unchunked_gb']:>11} {est['total_gb']:>7} {est['usable_gb']:>7}  "
                  f"{'fits' if est['fits'] else 'DOES NOT FIT'}")
            verdict = ("unchanged verdict vs the old fp32-chunk-only 151936 estimate"
                       if not est["fit_changed"] else "CHANGED: fits flipped")
            print(f"  {'':<11} {'':<6} {'':>6} old total {est['old_total_gb']} GB -> {verdict}")
            if est["fit_changed"]:
                changed.append((tier, method, seq))
    overflow = [(r["tier"], r["method"], r["seq"]) for r in rows if not r["fits"]]
    print(f"  caps the estimator refuses (must not be recommended): {overflow or 'none'}")
    print(f"  verdict flips against the old vocabulary: {changed or 'none'}")
    return {"rows": rows, "invalid_caps": overflow, "changed": changed}


def run_section_2b():
    """Resolve the vocabulary, revalidate every cap, derive this GPU's config. Returns (RUN, table)."""
    global VOCAB, VOCAB_SOURCE, LOSS_CHUNK_TOKENS, TRAINING_METHOD
    from lifemate.config import TrainingConfig  # noqa: PLC0415 - resolved lazily

    VOCAB, VOCAB_SOURCE = resolve_vocab()
    LOSS_CHUNK_TOKENS = int(TrainingConfig().loss_chunk_tokens)
    print(f"vocabulary     : {VOCAB} <- {VOCAB_SOURCE}")
    print(f"CE loss        : loss_chunk_tokens={LOSS_CHUNK_TOKENS} "
          f"({'chunked path priced below' if LOSS_CHUNK_TOKENS > 0 else '!! UNCHUNKED path priced below'})")
    table = revalidate_tier_caps()
    if table["invalid_caps"]:
        raise RuntimeError(
            f"TIER_METHOD_CAP recommends a configuration that the estimator says does not fit: "
            f"{table['invalid_caps']}. Lower those caps in cell 2b instead of launching an OOM."
        )
    at8192 = estimate_vram_gb("T4-16GB", "lora", 8192)
    print(f"  for scale at seq 8192 the two REAL logits terms are "
          f"lg_compute ({at8192['compute_dtype']}) {at8192['logits_compute_gb']} GB "
          f"(8192 x {VOCAB} x {at8192['bytes_per_param']} B, the head output held through backward) "
          f"+ lg_fp32_chunk {at8192['logits_chunked_gb']} GB (1024 x {VOCAB} x 4 B); "
          f"lg_unchunked {at8192['logits_unchunked_gb']} GB is the whole-sequence fp32 upcast "
          f"nobody runs — the allocation loss_chunk_tokens exists to avoid")
    RUN = derive_run_config(GPU_TIER, TRAINING_METHOD, None, MAX_SEQ_LENGTH, MICRO_BATCH_SIZE,
                            GRADIENT_ACCUMULATION_STEPS, NUM_EPOCHS, LEARNING_RATE)
    print_derived(RUN)
    if RUN["method"] != TRAINING_METHOD:
        # Sections 3-8 all read these globals, so the run must carry the decision the tier made.
        print(f"applying the tier decision: TRAINING_METHOD {TRAINING_METHOD} -> {RUN['method']}")
        TRAINING_METHOD = RUN["method"]
    return RUN, table


# The vocabulary comes from the repo's profile registry, and this cell runs before the checkout is
# importable (section 3). So: run it now when the package is already bound, otherwise let section 3c
# (the first cell where ``lifemate`` is imported) run it. Nothing between here and 3c reads RUN.
RUN, MEMORY_REVALIDATION = None, None
try:
    import lifemate as _lifemate_probe  # noqa: F401 - availability probe only
except ModuleNotFoundError:
    print("section 2b deferred: the vocabulary must come from the repo's profile registry "
          "(lifemate.models.profiles), and the package is not importable yet. Section 3c runs this "
          "derivation as soon as the checkout is on sys.path.")
else:
    RUN, MEMORY_REVALIDATION = run_section_2b()


## 3. Load the repo

Purpose: get the code that owns the training loop, checkpoint format and evaluation, pin it, and
prove it imports. The notebook never re-implements the loop. Cost: ~1-2 min, network only.

`pip install -e .` alone installs **no** dependencies — the package is stdlib-only by design — so
the ML stack is checked/installed separately, and only what is missing **or below its floor**
(section 1c prints the `installed` / `required` / `action` table).


In [ ]:
# --- 3a. Clone / update / pin the repo --------------------------------------
REPO_URL = "https://github.com/Le-Minh-Nhut/zaloai"
REPO_DIR = Path("/content/zaloai") if IN_COLAB else Path.cwd()


def _git(*args, cwd):
    out = subprocess.run(["git", *args], cwd=str(cwd), capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed in {cwd}:\n{out.stderr.strip()}")
    return out.stdout.strip()


if not (REPO_DIR / "scripts" / "train.py").is_file():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(
            f"{REPO_DIR} exists but does not contain scripts/train.py. Point REPO_DIR at the clone "
            "or delete that directory and re-run this cell."
        )
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"repo already present: {REPO_DIR}")

if BRANCH_OR_COMMIT:
    _git("fetch", "--all", "--tags", "--prune", cwd=REPO_DIR)
    try:
        _git("checkout", BRANCH_OR_COMMIT, cwd=REPO_DIR)
    except RuntimeError as exc:
        raise RuntimeError(
            f"cannot check out {BRANCH_OR_COMMIT!r}. Use a branch name (main) or a full commit sha "
            f"that exists on {REPO_URL}. Original error:\n{exc}"
        ) from None
print("HEAD:", _git("rev-parse", "--short", "HEAD", cwd=REPO_DIR), "| ref:", BRANCH_OR_COMMIT or "(unchanged)")


In [ ]:

# --- 3b. Install the repo itself (editable) --------------------------------
# Dependencies were handled in section 1c. The package declares none (stdlib-only by design), so
# `--no-deps` is correct here: it installs the console script + the src/ layout without dragging in
# a second copy of torch.
def _pip(*args):
    print("pip", " ".join(args), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

_pip("-e", str(REPO_DIR), "--no-deps")
print("installed lifemate from", REPO_DIR)

# QLoRA needs bitsandbytes, VERSION-checked (>=0.43): a wheel that merely imports is not a working
# 4-bit path. Gated here (not in 1c) because only section 2 knows the chosen method. A method that
# cannot be satisfied stops HERE, loudly -- never a silent 16-bit fallback labelled QLoRA.
if TRAINING_METHOD == "qlora":
    _bnb_spec = [f"bitsandbytes>={BITSANDBYTES_FLOOR}"]
    _bnb_rows = dependency_state(_bnb_spec)
    print_state(_bnb_rows, "QLoRA dependency gate:")
    if not _bnb_rows[0]["ok"]:
        _pip(f"bitsandbytes>={BITSANDBYTES_FLOOR}")
        _bnb_rows = dependency_state(_bnb_spec)
        print_state(_bnb_rows, "after pip install:")
    if not _bnb_rows[0]["ok"]:
        raise RuntimeError(
            f"bitsandbytes {_bnb_rows[0]['installed']} does not satisfy >={BITSANDBYTES_FLOOR}, so "
            "TRAINING_METHOD=qlora cannot run; this notebook will not mislabel a 16-bit run as QLoRA. "
            "Install it by hand (`pip install bitsandbytes>=0.43`), then Runtime -> Restart session "
            'and re-run from section 1; or set TRAINING_METHOD="lora" (16-bit adapters) deliberately. '
            "Nothing was trained."
        )


In [ ]:
# --- 3c. Verify imports + bind the repo CLI contract -----------------------
import threading  # noqa: E402 - used by run_cli below

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

import lifemate  # noqa: E402
from lifemate.config import TUNING_MODES, dump_config, load_config  # noqa: E402
from lifemate.errors import LifeMateError  # noqa: E402

if Path.cwd() != REPO_DIR:
    os.chdir(REPO_DIR)  # scripts/ resolve configs/ and outputs/ relative to cwd

print(f"lifemate {lifemate.__version__} from {Path(lifemate.__file__).parent}")
print(f"TUNING_MODES in this checkout: {TUNING_MODES}")
import peft  # noqa: E402
import transformers  # noqa: E402
print(f"transformers {transformers.__version__} | peft {peft.__version__}")

# Section 2b needs this package (the vocabulary comes from lifemate.models.profiles, never a literal),
# and it was skipped there because this checkout was not importable yet. It runs now, once.
if RUN is None:
    RUN, MEMORY_REVALIDATION = run_section_2b()
else:
    print(f"section 2b already derived: vocab {VOCAB} from {VOCAB_SOURCE}")

# Same for the feature switchboard (section 2a): it is built from the REAL dataclasses in
# lifemate.features and validated by the registry, so validation must happen before any GPU work.
if globals().get("FEATURE_CONFIG") is None:
    FEATURE_CONFIG, FEATURE_REPORT = run_feature_section()
else:
    print(f"section 2a already built: mode {FEATURE_REPORT['mode']} | training fingerprint "
          f"{FEATURE_REPORT['training_fingerprint']}")


def script_help(script_name):
    """Raw ``--help`` text of a repo script; fails loudly when it is absent or broken."""
    script = REPO_DIR / "scripts" / script_name
    if not script.is_file():
        raise FileNotFoundError(f"{script} is missing from the checkout at {REPO_DIR}")
    out = subprocess.run([sys.executable, str(script), "--help"], capture_output=True, text=True)
    if not out.stdout and not out.stderr:
        raise RuntimeError(f"{script_name} --help printed nothing; the checkout is broken")
    return out.stdout + out.stderr


def script_flags(script_name):
    """Flags a script actually accepts, parsed from its own help text (never guessed)."""
    return set(re.findall(r"(?<![\w-])(--[a-z0-9][a-z0-9-]*)", script_help(script_name)))


def require_flags(script_name, flags, hint=""):
    """Fail with an actionable message when the checkout lacks a flag this notebook uses."""
    have = script_flags(script_name)
    missing = [flag for flag in flags if flag not in have]
    if missing:
        raise RuntimeError(
            f"{script_name} does not accept {missing}; it accepts {sorted(have)}. "
            f"{hint or 'Pin BRANCH_OR_COMMIT to a commit whose CLI matches this notebook.'}"
        )
    return have


def flag_choices(script_name, flag):
    """Allowed values of a choice flag, from the script's own help text."""
    match = re.search(rf"{re.escape(flag)}\s*\{{([^}}]*)\}}", script_help(script_name))
    return [part.strip() for part in match.group(1).split(",")] if match else []


TUNING_CHOICES = flag_choices("train.py", "--tuning")
QLORA_SUPPORTED = "qlora" in TUNING_CHOICES
print(f"--tuning accepts: {TUNING_CHOICES} (qlora supported: {QLORA_SUPPORTED})")
# The mode the repo will actually run with. Under QLoRA the trained artifact is still a LoRA
# adapter — only the frozen base is 4-bit — so adapter semantics are identical.
CLI_TUNING_MODE = "qlora" if (TRAINING_METHOD == "qlora" and QLORA_SUPPORTED) else "lora"
if TRAINING_METHOD == "qlora" and not QLORA_SUPPORTED:
    print(
        "warning: TRAINING_METHOD=qlora requested, but this checkout has no `--tuning qlora`. "
        "Section 6 still verifies the 4-bit stack (NF4 + prepare_model_for_kbit_training, adapter "
        "gradients flowing), then stops with an actionable error instead of launching a run this "
        "trainer cannot honour."
    )

LAST_CLI_PEAK_MIB = None


def gpu_used_mib():
    """Device memory in use, from nvidia-smi (sees other processes), else torch counters."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        if out.returncode == 0 and out.stdout.strip():
            return str(int(out.stdout.split()[0]))
    except Exception:
        pass
    if torch is not None and torch.cuda.is_available():
        return str(torch.cuda.memory_allocated() // 2**20)
    return "n/a"


def stream_process(cmd, env, *, sample_peak, peak_interval, status_prefix, on_tick=None):
    """Run ``cmd``, stream stdout, sample device memory; return (rc, lines).

    ``on_tick`` runs every ``peak_interval`` seconds while the child is alive — that is the hook the
    training cell uses to mirror finished checkpoints to Drive *during* the run.
    """
    global LAST_CLI_PEAK_MIB
    print("$", " ".join(str(part) for part in cmd), flush=True)
    LAST_CLI_PEAK_MIB = None
    started = time.time()
    lines = []
    proc = subprocess.Popen(cmd, cwd=str(REPO_DIR), env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)

    def _pump():
        for line in proc.stdout:
            text = line.rstrip("\n")
            lines.append(text)
            print(text, flush=True)

    reader = threading.Thread(target=_pump, daemon=True)
    reader.start()
    last_status = 0.0
    while proc.poll() is None:
        if sample_peak:
            used = gpu_used_mib()
            if used.isdigit():
                LAST_CLI_PEAK_MIB = max(LAST_CLI_PEAK_MIB or 0, int(used))
        if on_tick is not None:
            on_tick()
        if time.time() - last_status >= max(peak_interval, 30):
            last_status = time.time()
            print(f"    [{status_prefix}] {int(time.time() - started)}s elapsed, "
                  f"{gpu_used_mib()} MiB used", flush=True)
        time.sleep(peak_interval)
    reader.join(timeout=30)
    rc = proc.wait()
    if sample_peak and LAST_CLI_PEAK_MIB:
        print(f"    [{status_prefix}] sampled peak: {LAST_CLI_PEAK_MIB} MiB", flush=True)
    return rc, lines


def run_cli(script_name, args, env_extra=None, sample_peak=False, peak_interval=2.0):
    """Run a repo script; raise with the script's own exit-code meaning on a non-zero exit."""
    env = dict(os.environ)
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    env.setdefault("PYTHONUNBUFFERED", "1")
    if env_extra:
        env.update({k: str(v) for k, v in env_extra.items()})
    cmd = [sys.executable, str(REPO_DIR / "scripts" / script_name), *[str(a) for a in args]]
    rc, lines = stream_process(cmd, env, sample_peak=sample_peak, peak_interval=peak_interval,
                               status_prefix="cli")
    if rc != 0:
        raise RuntimeError(
            f"{script_name} exited {rc} (0 ok, 1 failure, 2 data missing, 3 environment or weights "
            "unusable, 130 interrupted). The actionable line is in the output above — the scripts "
            "print it immediately before exiting."
        )
    return lines


# Names that can hold a torch module, a tokenizer, a tensor batch or a report dict that carries a
# model. `lora_report` is the trap: `del trainable_model` frees NOTHING while the dict still holds
# the same PeftModel.
RELEASABLE_NAMES = (
    # modules (the notebook never binds a bare ``model``; listed so a future cell cannot leak one)
    "trainable_model", "raw_model", "bundle", "model",
    # tokenizer (+ the section-5 cache that keeps one alive)
    "tokenizer", "TOKENIZER_CACHE",
    # containers that reference the same objects
    "lora_report", "generator", "agent", "agent_env", "_collator",
    # probe intermediates: the batch and every (name, parameter) list hold device tensors
    "_batch", "_examples", "_trainable", "_with_grad", "_nonzero", "_loss", "_probe_records",
    # section 7c's worst-case probe: the model, its adapter report and the tokenizer are all bound
    # by name, and the batch/es held device tensors of the full sequence length
    "_probe_model", "_probe_report", "_probe_tokenizer", "_sample_result",
)


def _is_model_like(value):
    """True for a torch module, a HF tokenizer, or anything exposing a callable ``parameters``."""
    if torch is not None and isinstance(value, torch.nn.Module):
        return True
    try:
        from transformers import PreTrainedTokenizerBase
    except Exception:  # transformers absent: nothing to compare against
        PreTrainedTokenizerBase = None
    if PreTrainedTokenizerBase is not None and isinstance(value, PreTrainedTokenizerBase):
        return True
    return callable(getattr(value, "parameters", None))


def find_model_refs(scope, max_depth=3):
    """Attribute paths inside ``scope`` that still reach a module/tokenizer.

    Walks dicts, sequences, sets and one __dict__ chain per plain object — never module objects,
    classes or functions (walking those would reach into ``torch`` itself).
    """
    import types

    found, seen = [], set()

    def walk(value, path, depth):
        key = id(value)
        if key in seen or depth > max_depth:
            return
        seen.add(key)
        if _is_model_like(value):
            found.append(path)
            return
        if isinstance(value, dict):
            for item_key, item in list(value.items()):
                walk(item, f"{path}[{item_key!r}]", depth + 1)
            return
        if isinstance(value, (list, tuple, set, frozenset)):
            for index, item in enumerate(value):
                walk(item, f"{path}[{index}]", depth + 1)
            return
        if isinstance(value, (types.ModuleType, type, types.FunctionType, types.BuiltinFunctionType,
                               str, bytes, bytearray, int, float, bool, complex, type(None))):
            return
        slots = getattr(value, "__dict__", None)
        if isinstance(slots, dict):
            for attr, item in list(slots.items()):
                walk(item, f"{path}.{attr}", depth + 1)

    for name, value in list(scope.items()):
        walk(value, name, 0)
    return found


def release_model_refs(scope=None, verbose=True):
    """Drop every notebook reference to a torch module / tokenizer and PROVE it is gone.

    ``torch.cuda.empty_cache()`` alone frees nothing that is still referenced — it only returns
    *unreferenced* blocks to the driver, so calling it while a model is alive is a no-op that looks
    like success. The guard is therefore the residual scan: it walks the whole namespace and raises
    naming every attribute path that still reaches a module, a tokenizer or an object with a
    ``parameters`` callable. Only after that scan passes is the cache emptied.

    Returns ``{"allocated_mib", "reserved_mib", "freed_mib", "residual_names"}``.
    """
    import gc

    scope = globals() if scope is None else scope
    cuda = torch is not None and torch.cuda.is_available()
    allocated_before = int(torch.cuda.memory_allocated() // 2**20) if cuda else 0
    dropped = []
    for name in RELEASABLE_NAMES:
        if name in scope:
            dropped.append(name)
            del scope[name]
    gc.collect()
    residual = find_model_refs(scope)
    if cuda:
        # Safe only now that the scan proved nothing is referenced; otherwise this frees nothing.
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    gc.collect()
    allocated = int(torch.cuda.memory_allocated() // 2**20) if cuda else 0
    reserved = int(torch.cuda.memory_reserved() // 2**20) if cuda else 0
    report = {
        "allocated_mib": allocated,
        "reserved_mib": reserved,
        "freed_mib": allocated_before - allocated,
        "residual_names": residual,
    }
    if residual:
        raise RuntimeError(
            f"{len(residual)} reference(s) to a model/tokenizer survived release: {residual}. "
            "Checkpoint sync and the next training subprocess need the whole card; a residual "
            "reference means torch.cuda.empty_cache() just freed nothing. Delete these names (add "
            "them to RELEASABLE_NAMES when they are notebook-owned) before continuing."
        )
    if verbose:
        print(f"release_model_refs: dropped {dropped}; allocated {allocated} MiB, "
              f"reserved {reserved} MiB, freed {report['freed_mib']} MiB; residual refs: none")
        print({"allocated_mib": allocated, "reserved_mib": reserved,
               "freed_mib": report["freed_mib"], "residual_names": residual})
    return report


def free_gpu():
    """Collect garbage, empty the CUDA cache and reset peak counters."""
    import gc

    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    print(f"gpu after free: {gpu_used_mib()} MiB used")


print("helpers ready")


## 4. Google Drive: paths, dataset, atomic artifact sync

Purpose: decide where durable things live. Convention used below:

* `/content/...` (local, fast, ephemeral) — HF cache, training output, probe checkpoints;
* `MyDrive/lifemate_colab/` (durable) — `data/`, `datasets/`, `checkpoints/`, `outputs/`, `logs/`.

A Colab disconnect erases the runtime, not Drive. So checkpoints are written locally by the repo
(atomic staging + rename) and then copied to Drive through a staging directory, `fsync`ed, and
renamed — plus a `SYNC_COMPLETE.json` marker that records a **size and sha256 for every file**.

Three rules make the copy crash-safe:

* **immutable, versioned directories.** A step directory is `checkpoints/<run>/step-000100/` and its
  name is its identity, so a published generation is never rewritten in place; only a re-sync of the
  *same* step replaces one. A marker is written LAST, after every file is fsynced, so `<dir>/SYNC_COMPLETE.json`
  existing means "this whole tree was written by us". A missing or unparsable marker makes the
  directory non-selectable for resume — that is the whole point of the marker.
* the previous generation is never destroyed. When `dst` already exists it is renamed to
  `dst.prev-<n>` before the new tree takes the name; only generations beyond `KEEP_PREVIOUS` are
  pruned, and only after the new copy verified;
* an interrupted copy keeps its `.incomplete-<ts>` name and is never mistaken for a checkpoint.

**Directory rename is probed, not assumed.** `os.replace` of a directory is atomic on a local POSIX
filesystem, but a Drive FUSE mount may implement rename as copy+delete or refuse it outright. Cell 4b
therefore *tests* the capability with two throw-away sibling directories, caches the answer, and the
marker records which mode installed the tree: `rename_mode: "atomic"` or `"copy_only"`. In
`copy_only` mode the sync degrades to copy-then-mark while keeping BOTH generations (`dst.prev-<n>`
stays complete for the whole copy), so a crash mid-copy leaves a `dst` whose marker is absent —
which every reader refuses — and an intact rollback copy. **Not verified against a real Drive mount
on this machine; only the local-filesystem branch was exercised.**

Resume re-hashes every file the marker lists (size and digest), so a torn copy that still carries a
stale marker from an earlier sync is refused with the offending filename. A marker written by an
older notebook version has no `sha256`: it is reported as `legacy marker, cannot be verified` and is
**not** accepted for resume — re-sync the directory to upgrade it.

**Nothing here ever deletes a checkpoint on "looks synced".** Deleting a local step directory
requires, *for that directory*: a Drive counterpart, a marker that verifies (sha256 of every file),
matching run/model/step identity, and a resumable Drive copy. A non-empty local set with an EMPTY
Drive set is an explicit failure (`all([])` being `True` used to delete everything). The newest
local generation is never removed, and `DELETE_LOCAL_AFTER_VERIFIED_SYNC` defaults to `False`, so
the normal outcome is a printed dry run naming what *would* be deleted.

Cost: ~1 CU-minute, plus transfer time proportional to checkpoint size (~35 MB per LoRA step dir).

No personal path is hardcoded: `DRIVE_ROOT` is resolved from the mounted Drive root at runtime.


In [ ]:
# --- 4a. Mount Drive (optional) and define every path in one place ----------
DRIVE_ROOT = None
if USE_GOOGLE_DRIVE:
    if not IN_COLAB:
        print("note: USE_GOOGLE_DRIVE is on but this is not Colab; falling back to local paths")
    else:
        from google.colab import drive  # noqa: PLC0415 - Colab only
        drive.mount("/content/drive")
        DRIVE_ROOT = Path("/content/drive/MyDrive/lifemate_colab")

LOCAL_WORK = Path("/content/lifemate_work") if IN_COLAB else Path(REPO_DIR) / "outputs" / "colab_work"
DATA_ROOT = LOCAL_WORK / "data"   # outside the git clone: the repo tree stays clean
LOCAL_DATASET_ROOT = LOCAL_WORK / "datasets"
LOCAL_CKPT_ROOT = LOCAL_WORK / "checkpoints"
LOCAL_OUTPUT_ROOT = LOCAL_WORK / "outputs"
HF_CACHE = LOCAL_WORK / "hf_cache"

for _p in (LOCAL_WORK, LOCAL_DATASET_ROOT, LOCAL_CKPT_ROOT, LOCAL_OUTPUT_ROOT, HF_CACHE):
    _p.mkdir(parents=True, exist_ok=True)

# One HF cache for the whole session: weights are downloaded once, then reused.
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

DRIVE_PATHS = {}
if DRIVE_ROOT is not None:
    DRIVE_PATHS = {
        "data": DRIVE_ROOT / "data",
        "datasets": DRIVE_ROOT / "datasets",
        "checkpoints": DRIVE_ROOT / "checkpoints",
        "outputs": DRIVE_ROOT / "outputs",
        "logs": DRIVE_ROOT / "logs",
    }
    for _p in DRIVE_PATHS.values():
        _p.mkdir(parents=True, exist_ok=True)
    print("drive root:", DRIVE_ROOT)

print(f"local work : {LOCAL_WORK}")
print(f"repo       : {REPO_DIR}")
print(f"dataset in : {DATA_ROOT} (provisioned in 4c; test splits and test labels are never read)")


In [ ]:
# --- 4b. Atomic artifact sync + completion markers --------------------------
import hashlib  # noqa: E402 - per-file digests in the completion marker

SYNC_MARKER = "SYNC_COMPLETE.json"
SYNC_SCHEMA = 2                 # 2 adds per-file sha256; 1 (legacy) had sizes only
KEEP_PREVIOUS = 1               # how many `dst.prev-<n>` generations stay on Drive
_PROHIBITED = re.compile(r"(^|/)[^/]+_environment(/|$)")


def _reject_prohibited(path):
    """Refuse any path under an organizers' ``*_environment`` directory."""
    if _PROHIBITED.search(str(Path(path)).replace("\\", "/")):
        raise PermissionError(
            f"{path} is inside a *_environment directory: off-limits for reading, copying or "
            "uploading (docs/COMPETITION_COMPLIANCE.md section 1)."
        )
    return Path(path)


def safe_unzip(archive, dest):
    """Extract an archive, skipping any member under an organizers' ``*_environment`` dir."""
    import zipfile

    archive, dest = _reject_prohibited(archive), Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    skipped = 0
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            name = member.filename.replace("\\", "/")
            if _PROHIBITED.search(name) or name.startswith("/") or ".." in Path(name).parts:
                skipped += 1
                continue
            target = dest / name
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with bundle.open(member) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)
    print(f"extracted {archive} -> {dest} (skipped {skipped} prohibited/unsafe member(s))")
    return dest


def _fsync_tree(root):
    """fsync every regular file in the tree; returns the number synced."""
    synced = 0
    for item in Path(root).rglob("*"):
        if item.is_file():
            with open(item, "rb") as handle:
                os.fsync(handle.fileno())
            synced += 1
    return synced


def _sha256(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def file_manifest(root):
    """``{relative path: size}`` for every file under ``root`` (marker excluded)."""
    root = Path(root)
    return {
        str(p.relative_to(root)): p.stat().st_size
        for p in sorted(root.rglob("*")) if p.is_file() and p.name != SYNC_MARKER
    }


def write_sync_marker(root, extra=None):
    """Completion marker: per-file size AND sha256, itself written last and atomically.

    Sizes alone cannot detect a copy that was truncated at a block boundary but kept its length;
    the digest list can. The marker is renamed into place only after every other file is durable,
    so ``<root>/SYNC_COMPLETE.json`` existing means "this whole tree was written by us".
    """
    root = _reject_prohibited(root)
    files = {}
    for relative in file_manifest(root):
        path = root / relative
        files[relative] = {"size": path.stat().st_size, "sha256": _sha256(path)}
    payload = {
        "schema": SYNC_SCHEMA,
        "written_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "host": "colab" if IN_COLAB else "local",
        "files": files,
        "total_bytes": sum(entry["size"] for entry in files.values()),
        "n_files": len(files),
    }
    payload.update(extra or {})
    tmp = root / (SYNC_MARKER + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    with open(tmp, "rb") as handle:
        os.fsync(handle.fileno())
    os.replace(tmp, root / SYNC_MARKER)
    return root / SYNC_MARKER


def verify_sync_marker(root):
    """``(ok, reason)``: re-hash every file the marker lists and catch every drift.

    Detects a missing marker, a schema the notebook does not know, a marker without digests
    (schema 1, "legacy": sizes alone cannot be verified), a missing file, an extra file, a wrong
    size and a wrong sha256 — each reason names the offending file.
    """
    root = Path(root)
    marker = root / SYNC_MARKER
    if not marker.is_file():
        return False, f"{root} has no {SYNC_MARKER}; the copy never completed"
    try:
        payload = json.loads(marker.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as exc:
        return False, f"{SYNC_MARKER} under {root} is unreadable: {exc}"
    if not isinstance(payload, dict):
        return False, f"{SYNC_MARKER} under {root} is not a JSON object"
    listed = payload.get("files")
    if not isinstance(listed, dict) or not listed:
        return False, f"{SYNC_MARKER} under {root} lists no files"
    # A marker from the previous notebook version recorded sizes only. Sizes cannot prove content,
    # so it is reported verbatim and refused: the directory must be re-synced to be resumable.
    for relative, entry in sorted(listed.items()):
        if not isinstance(entry, dict) or "sha256" not in entry:
            return False, (
                f"legacy marker, cannot be verified: {SYNC_MARKER} under {root} records no sha256 "
                f"for {relative} (schema {payload.get('schema')!r}). Re-sync this directory to "
                "upgrade the marker; it cannot be trusted for resume."
            )
    if payload.get("schema") != SYNC_SCHEMA:
        return False, (
            f"{SYNC_MARKER} under {root} has schema {payload.get('schema')!r}, this notebook writes "
            f"{SYNC_SCHEMA}: the marker was written by an incompatible version"
        )
    for relative, entry in sorted(listed.items()):
        path = root / relative
        if not path.is_file():
            return False, f"{relative} is listed in {SYNC_MARKER} but missing from {root}"
        size = path.stat().st_size
        if int(entry.get("size", -1)) != size:
            return False, (f"{relative} is {size} bytes on disk, {entry.get('size')} in {SYNC_MARKER}")
        actual = _sha256(path)
        if actual != entry["sha256"]:
            return False, f"{relative} sha256 {actual[:16]}... != {entry['sha256'][:16]}... (torn copy)"
    present = {name for name in file_manifest(root)}
    extra = sorted(present - set(listed))
    if extra:
        return False, f"{extra} present under {root} but absent from {SYNC_MARKER}"
    missing = sorted(set(listed) - present)
    if missing:
        return False, f"{missing} listed in {SYNC_MARKER} but absent from {root}"
    declared_total = payload.get("total_bytes")
    if declared_total is not None and int(declared_total) != sum(e["size"] for e in listed.values()):
        return False, f"{SYNC_MARKER} total_bytes={declared_total} disagrees with its own file list"
    return True, f"ok ({len(listed)} files, {payload.get('total_bytes')} bytes, schema {SYNC_SCHEMA})"


def checkpoint_ready(path, require_marker=False, run_name=None):
    """Can the repo's trainer genuinely resume from this directory?

    Returns ``(ok, reason)``. ``require_marker`` must be True for anything that crossed a network
    copy (Drive), because only the marker proves the copy finished — and it is now *verified*:
    every listed file is re-hashed, so a directory that only looks complete is refused with the
    name of the file that disagrees. ``run_name`` additionally requires the marker's ``run_name``
    (when recorded) to match this session, so a resume cannot silently adopt another run.
    """
    path = Path(path)
    if not path.is_dir():
        gens = [p.name for p in _previous_generations(path)] if path.parent.is_dir() else []
        hint = (f" Generations that DID survive: {gens} — point RESUME_FROM_CHECKPOINT at one of them."
                if gens else "")
        return False, f"{path} is not a directory.{hint}"
    if ".incomplete-" in path.name:
        return False, (
            f"{path} is an interrupted staging directory (.incomplete-*), not a finished copy. Use "
            f"the finished generations next to it: {[p.name for p in _previous_generations(path)] or 'none'}"
        )
    if require_marker:
        ok, why = verify_sync_marker(path)
        if not ok:
            return False, f"{path} is not a verified sync: {why}"
        marker = json.loads((path / SYNC_MARKER).read_text(encoding="utf-8"))
        if run_name and marker.get("run_name") and marker["run_name"] != run_name:
            return False, (f"{path} was synced for run {marker['run_name']!r}, not {run_name!r}")
    state = path / "training_state.json"
    if not state.is_file():
        return False, f"{path} has no training_state.json (not a step-<n> checkpoint)"
    try:
        payload = json.loads(state.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        return False, f"training_state.json is corrupt: {exc}"
    if int(payload.get("step") or 0) <= 0:
        return False, f"training_state.json records step {payload.get('step')!r}"
    weights = list(path.glob("adapter_model.*")) + list(path.glob("*.safetensors")) + list(path.glob("pytorch_model*.bin"))
    if not weights:
        return False, f"{path} has no weights (adapter_model.* / *.safetensors / pytorch_model*.bin)"
    if not (path / "optimizer_state.pt").is_file():
        return False, (
            f"{path} has no optimizer_state.pt: resuming would restart AdamW moments from zero. "
            "That is adapter-only loading, not optimizer-state resumption."
        )
    return True, "ok"


def _previous_generations(dst):
    """Existing ``dst.prev-<n>`` siblings, lowest generation first."""
    dst = Path(dst)
    gens = []
    for sibling in dst.parent.glob(f"{dst.name}.prev-*"):
        try:
            gens.append((int(sibling.name.rsplit("-", 1)[-1]), sibling))
        except ValueError:
            continue
    return [sibling for _, sibling in sorted(gens)]


# --- 4b-ii. Directory-rename probe: an atomic rename is a POSIX fade, not a filesystem promise ---
_RENAME_PROBE = {}

def rename_mode_for(parent):
    """``(mode, reason)``, mode "atomic" | "copy_only", cached per parent directory.

    ``os.replace`` of a directory is only atomic on a local POSIX filesystem. Google Drive's FUSE
    mount implements rename as copy+delete and can refuse it outright (as do some NFS/SMB mounts),
    so the capability is PROBED with two throw-away sibling directories instead of assumed. The
    answer is cached per parent, and ``atomic_sync`` records it in the marker it writes, so a later
    reader can tell which install path produced the directory.
    """
    parent = Path(parent)
    key = str(parent.resolve()) if parent.exists() else str(parent)
    if key in _RENAME_PROBE:
        return _RENAME_PROBE[key]
    if not parent.is_dir():
        return ("copy_only", f"{key} is not a directory: a directory rename cannot be probed")
    probe_a = parent / f".rename-probe-{os.getpid()}-a"
    probe_b = parent / f".rename-probe-{os.getpid()}-b"
    mode, reason = "copy_only", f"directory rename unsupported on {key}"
    try:
        for leftover in (probe_a, probe_b):
            shutil.rmtree(leftover, ignore_errors=True)
        probe_a.mkdir()
        (probe_a / "probe.txt").write_text("probe", encoding="utf-8")
        os.replace(probe_a, probe_b)
        if (probe_b / "probe.txt").is_file() and not probe_a.exists():
            mode, reason = "atomic", f"os.replace of a directory works on {key}"
        else:
            reason = f"os.replace returned without moving the directory on {key}"
    except OSError as exc:
        reason = f"os.replace of a directory failed on {key}: {exc}"
    finally:
        for leftover in (probe_a, probe_b):
            shutil.rmtree(leftover, ignore_errors=True)
    _RENAME_PROBE[key] = (mode, reason)
    return mode, reason

def _next_generation_name(dst):
    """``dst.prev-<n>`` with ``n`` one past the highest existing generation."""
    generations = _previous_generations(dst)
    next_gen = (int(generations[-1].name.rsplit("-", 1)[-1]) + 1) if generations else 1
    return dst.with_name(f"{dst.name}.prev-{next_gen}")

def _copy_generation_aside(dst):
    """COPY ``dst`` to ``dst.prev-<n>``; returns ``(target, ok, why)``. Used when rename fails.

    The copy happens BEFORE anything is removed, so the "copy-then-mark, keep both" degradation
    never trades the last complete generation for a torn one. A source that already failed
    verification is still preserved (it is evidence, and its bytes may be the only copy) and the
    caller warns instead of aborting — otherwise a corrupt Drive directory could never be re-synced.
    """
    target = _next_generation_name(dst)
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(dst, target)
    _fsync_tree(target)
    ok, why = verify_sync_marker(target)
    return target, ok, why

def atomic_sync(src, dst, extra=None, keep_previous=KEEP_PREVIOUS):
    """Copy ``src`` to ``dst`` without ever deleting the only durable copy.

    Order, and why: copy -> fsync tree -> verified marker (per-file sha256, written LAST) -> keep the
    CURRENT ``dst`` as ``dst.prev-<n>`` -> install the new tree. Nothing is removed before a copy of
    the old generation exists, so a crash at ANY point leaves one complete directory under ``dst`` or
    ``dst.prev-<n>``, and an interrupted copy is refused because its marker is missing (atomic path:
    the ``.incomplete-*`` staging name) or unparsable.

    ``dst`` is versioned by the step number in its name, so a step-N directory is immutable once
    published; only a re-sync of the SAME step replaces one. Whether the install uses ``os.replace``
    (``rename_mode: "atomic"``) or copy-then-mark (``"copy_only"``, for a Drive mount whose directory
    rename is not atomic) is PROBED by :func:`rename_mode_for` and recorded in the marker.
    """
    src, dst = _reject_prohibited(src), _reject_prohibited(dst)
    if not src.is_dir():
        raise FileNotFoundError(f"nothing to sync: {src} is not a directory")
    dst.parent.mkdir(parents=True, exist_ok=True)
    mode, mode_reason = rename_mode_for(dst.parent)
    print(f"rename mode    : {mode} ({mode_reason})")
    # The staging copy always gets a fresh marker: a marker copied in from an already-synced source
    # describes the OLD tree, so it is never trusted here. The marker is written after _fsync_tree,
    # so its presence means "every file in this tree is durable".
    marker_extra = {**(extra or {}), "source": str(src), "rename_mode": mode}
    staging = None
    if mode == "atomic":
        staging = dst.with_name(f"{dst.name}.incomplete-{int(time.time())}")
        if staging.exists():
            shutil.rmtree(staging)
        shutil.copytree(src, staging)
        _fsync_tree(staging)
        write_sync_marker(staging, extra=marker_extra)
        ok, why = verify_sync_marker(staging)
        if not ok:
            raise RuntimeError(f"the staged copy at {staging} does not verify ({why}); {dst} untouched")

    previous = None
    if dst.exists():
        if mode == "atomic":
            previous = _next_generation_name(dst)
            os.replace(dst, previous)      # rename, never rmtree: dst keeps every byte it had
            print(f"kept previous generation: {previous.name}")
        else:
            previous, kept_ok, kept_why = _copy_generation_aside(dst)
            print(f"kept previous generation: {previous.name} (copy_only; verifies: {kept_ok}"
                  + ("" if kept_ok else f" — {kept_why}") + ")")
            shutil.rmtree(dst)             # every byte is already under `previous`
    if mode == "atomic":
        os.replace(staging, dst)           # now dst is complete AND verified
    else:
        # "copy-then-mark, keep both": no atomic install exists on this mount. A crash mid-copy
        # leaves a dst whose marker is absent, which every reader refuses; the previous verified
        # generation is untouched under `previous` for the whole duration.
        shutil.copytree(src, dst)
        _fsync_tree(dst)
        write_sync_marker(dst, extra=marker_extra)

    durable = verify_sync_marker(dst)
    if not durable[0]:
        raise RuntimeError(f"{dst} does not verify right after the rename: {durable[1]}")
    kept = _previous_generations(dst)[-keep_previous:] if keep_previous > 0 else []
    for stale in _previous_generations(dst):
        if stale not in kept:
            shutil.rmtree(stale, ignore_errors=True)
    print(f"synced {src} -> {dst} ({len(file_manifest(dst))} files, verified"
          + (f"; rollback copy {previous.name}" if previous is not None else "") + ")")
    return dst

# --- 4b-iii. Discovery, resume selection, cleanup safety --------------------
_STEP_RE = re.compile(r"^step-(\d+)$")

def step_of(name):
    """Step number of a ``step-<n>`` name, or None."""
    match = _STEP_RE.match(Path(str(name)).name)
    return int(match.group(1)) if match else None

def _read_json_file(path):
    """``(payload, reason)``; payload is None (and reason says why) when absent or unparsable."""
    path = Path(path)
    if not path.is_file():
        return None, f"{path.name} is missing"
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as exc:
        return None, f"{path.name} is unreadable: {exc}"
    if not isinstance(payload, dict):
        return None, f"{path.name} is not a JSON object"
    return payload, "ok"

def _default_require_marker(parent):
    """A marker is mandatory for anything that crossed the Drive copy, optional locally.

    Local checkpoint directories are written by the trainer itself (no marker); Drive directories
    are written by ``atomic_sync`` and only a marker proves the copy finished. ``DRIVE_ROOT`` is
    what tells the two apart, so this stays correct if the work dir moves.
    """
    drive = globals().get("DRIVE_ROOT")
    return bool(drive is not None and str(parent).startswith(str(drive)))

def optimizer_state_facts(path):
    """``(ok, reason, facts)`` for ``optimizer_state.pt``. Never raises.

    "Loads" is checked by actually loading: a truncated ``torch.save`` file parses as garbage much
    later otherwise, and the failure would surface as a broken first optimizer step instead of a
    refused resume. Without torch the file is size-checked and reported as not loadable here, so a
    stdlib-only machine can still inspect a checkpoint's layout.
    """
    path = Path(path)
    if not path.is_file():
        return False, f"{path.name} is missing: resuming would restart AdamW moments from zero", {}
    size = int(path.stat().st_size)
    if size <= 0:
        return False, f"{path.name} is empty (0 bytes)", {"size": 0}
    # A modern torch.save payload is a zip archive. Checking that the archive is COMPLETE needs no
    # torch, so a copy truncated in transit is refused even on a machine without torch.
    with open(path, "rb") as handle:
        head = handle.read(4)
    if head[:2] == b"PK":
        import zipfile  # noqa: PLC0415 - stdlib, only for this structural check

        try:
            with zipfile.ZipFile(path) as archive:
                names = archive.namelist()
        except (zipfile.BadZipFile, OSError) as exc:
            return False, (f"{path.name} is not a complete torch.save archive ({exc}): the file was "
                           "truncated, so the optimizer state cannot be restored"), {"size": size}
        if not any(name.endswith(("data.pkl", "byteorder")) for name in names):
            return False, (f"{path.name} is a zip archive but holds no torch data entries "
                           f"({names[:4]}), so it is not optimizer state"), {"size": size}
    try:
        import torch  # noqa: PLC0415 - lazy: the notebook's stdlib parts must import without it
    except Exception:
        torch = None
    if torch is None:
        # No torch here. A torch.save container built from plain Python objects can still be
        # unpickled without torch, so the KEYS are read when possible; a payload that references
        # torch storage classes cannot be, and is reported as structurally intact but unverified.
        facts = {"size": size, "loaded": False}
        if head[:2] == b"PK":
            import pickle  # noqa: PLC0415
            import zipfile  # noqa: PLC0415

            try:
                with zipfile.ZipFile(path) as archive:
                    members = [name for name in archive.namelist() if name.endswith("data.pkl")]
                if members:
                    payload = pickle.loads(zipfile.ZipFile(path).read(members[0]))
            except Exception:
                payload = None
            if isinstance(payload, dict):
                facts = {**facts, "keys": sorted(str(key) for key in payload),
                         "optimizer": payload.get("optimizer") is not None,
                         "scheduler": payload.get("scheduler") is not None,
                         "grad_scaler": payload.get("grad_scaler") is not None}
                if not facts["optimizer"]:
                    return False, (f"{path.name} carries no optimizer state (key 'optimizer' is "
                                   "empty)"), facts
                return True, (f"keys read without torch ({size} B); scheduler "
                              f"{'present' if facts['scheduler'] else 'None'} — the payload was NOT "
                              "deserialised, so restoring it is unproven here"), facts
        return True, (f"present ({size} B) and structurally intact; keys not readable without torch"), \
            {**facts, "loaded": False}
    try:
        payload = torch.load(str(path), map_location="cpu", weights_only=False)
    except Exception as exc:  # truncated / corrupt / mismatched pickle
        return False, f"{path.name} does not load: {type(exc).__name__}: {exc}", {"size": size}
    if not isinstance(payload, dict):
        return False, (f"{path.name} loaded as {type(payload).__name__}, not an optimizer-state "
                       "dict"), {"size": size, "loaded": True}
    facts = {
        "size": size,
        "loaded": True,
        "keys": sorted(str(key) for key in payload),
        "optimizer": payload.get("optimizer") is not None,
        "scheduler": payload.get("scheduler") is not None,
        "grad_scaler": payload.get("grad_scaler") is not None,
    }
    if not facts["optimizer"]:
        return False, f"{path.name} carries no optimizer state (key 'optimizer' is empty)", facts
    missing = [key for key in ("scheduler", "grad_scaler") if key not in payload]
    if missing:
        return False, (f"{path.name} has no {missing} key: it was written by an older trainer and "
                       "cannot restore the LR schedule / loss scaling"), facts
    return True, (f"loads ({size} B); scheduler {'restored' if facts['scheduler'] else 'None'}, "
                  f"grad_scaler {'present' if facts['grad_scaler'] else 'None'}"), facts

def verify_resume_candidate(path, *, run_name=None, model_id=None, profile=None, config_hash=None,
                            dataset_manifest_hash=None, require_marker=None, load_optimizer=True):
    """Everything that must hold before a directory may be resumed from. Returns a dict.

    ``{"path", "name", "step", "ok", "rejections", "warnings", "exact", "exact_missing", "state",
    "progress", "optimizer"}``. A rejection is fatal (the directory is not a resume target); a
    missing exact-resume block is NOT a rejection — schema-1 checkpoints legitimately have no
    ``progress``, so they resume with ``exact=False`` and a stated reason.
    """
    path = Path(path)
    state, progress, optimizer = {}, None, {}
    rejections, warnings = [], []
    name = path.name
    step = step_of(name)
    if not path.is_dir():
        rejections.append(f"{name} is not a directory")
    elif ".incomplete-" in name:
        rejections.append(f"{name} is an interrupted staging directory (.incomplete-*), never a "
                          "checkpoint")
    elif ".prev-" in name:
        rejections.append(f"{name} is a superseded rollback generation (.prev-*): its step number is "
                          "not the newest copy of that step")
    elif step is None:
        rejections.append(f"{name} is not a step-<n> directory")
    if rejections:
        return {"path": path, "name": name, "step": step, "ok": False, "rejections": rejections,
                "warnings": warnings, "exact": False, "exact_missing": ["not a step directory"],
                "state": state, "progress": progress, "optimizer": optimizer}

    marker_required = _default_require_marker(path.parent) if require_marker is None else bool(require_marker)
    marker = {}
    if marker_required:
        ok, why = verify_sync_marker(path)
        if not ok:
            rejections.append(f"sync marker unusable: {why}")
        else:
            marker, _ = _read_json_file(path / SYNC_MARKER)
    elif (path / SYNC_MARKER).is_file():
        ok, why = verify_sync_marker(path)
        if not ok:
            rejections.append(f"sync marker present but unusable: {why}")
        else:
            marker, _ = _read_json_file(path / SYNC_MARKER)

    payload, why = _read_json_file(path / "training_state.json")
    if payload is None:
        rejections.append(f"no usable training_state.json: {why}")
    else:
        state = payload
        recorded = int(state.get("step") or 0)
        if recorded <= 0:
            rejections.append(f"training_state.json records step {state.get('step')!r}")
        elif recorded != step:
            rejections.append(f"training_state.json records step {recorded} but the directory is "
                              f"{name}")
        progress = state.get("progress") if isinstance(state.get("progress"), dict) else None

    weights = (list(path.glob("adapter_model.*")) + list(path.glob("*.safetensors"))
               + list(path.glob("pytorch_model*.bin")))
    if not weights:
        rejections.append("no weights (adapter_model.* / *.safetensors / pytorch_model*.bin)")

    meta, meta_why = _read_json_file(path / "lifemate_meta.json")
    recorded_ids = []
    if meta is None:
        rejections.append(f"model identity cannot be verified: {meta_why}")
    else:
        recorded_ids = [value for value in (meta.get("model_name"), meta.get("model_id")) if value]
        if model_id:
            if not recorded_ids:
                rejections.append("lifemate_meta.json records no model_name: identity unverifiable")
            elif str(model_id) not in [str(value) for value in recorded_ids]:
                rejections.append(f"model identity mismatch: checkpoint was trained for "
                                  f"{recorded_ids}, this session is {model_id}")
    if profile:
        recorded_profile = (meta or {}).get("model_profile") or marker.get("model_profile")
        if not recorded_profile:
            warnings.append("lifemate_meta.json records no model_profile: the architecture family "
                            "cannot be compared with this session")
        elif str(recorded_profile) != str(profile):
            rejections.append(f"model profile mismatch: checkpoint family {recorded_profile!r}, this "
                              f"session {profile!r} (adapter/base family differs)")
    marker_ids = [value for value in (marker.get("model_id"), marker.get("model_name")) if value]
    if model_id and marker_ids and str(model_id) not in [str(v) for v in marker_ids]:
        rejections.append(f"sync marker says model {marker_ids}, this session is {model_id}")
    if run_name:
        marker_run = marker.get("run_name")
        if marker_run and marker_run != run_name:
            rejections.append(f"synced for run {marker_run!r}, not {run_name!r}")
        meta_run = (meta or {}).get("run_name")
        if meta_run and meta_run != run_name:
            rejections.append(f"written by run {meta_run!r}, not {run_name!r}")

    stored_config = state.get("config_hash") if state else None
    if config_hash is not None:
        if not stored_config:
            rejections.append("checkpoint records no config_hash: the config it was trained under "
                              "cannot be compared")
        elif str(stored_config) != str(config_hash):
            rejections.append(f"config_hash mismatch: checkpoint {stored_config}, this session "
                              f"{config_hash}")
    stored_data = state.get("dataset_manifest_hash") if state else None
    if dataset_manifest_hash is not None:
        if not stored_data:
            warnings.append("checkpoint records no dataset_manifest_hash: the split cannot be compared")
        elif str(stored_data) != str(dataset_manifest_hash):
            rejections.append(f"dataset_manifest_hash mismatch: checkpoint {stored_data}, this "
                              f"session {dataset_manifest_hash}")

    opt_ok, opt_why, optimizer = optimizer_state_facts(path / "optimizer_state.pt")
    if not opt_ok:
        rejections.append(opt_why)
    elif not load_optimizer:
        warnings.append(opt_why)

    exact_missing = []
    if progress is None:
        exact_missing.append("training_state.json has no `progress` block (schema 1): the next "
                             "batch position and sampler permutation were never recorded")
    else:
        if int(progress.get("global_step") or 0) <= 0:
            exact_missing.append("progress.global_step is 0")
        if not progress.get("permutation"):
            exact_missing.append("progress.permutation is empty: the epoch order is unknown")
        if int(progress.get("planned_total_steps") or 0) <= 0:
            exact_missing.append("progress.planned_total_steps is 0: the schedule horizon is unknown")
        if not progress.get("sampler_rng_state"):
            exact_missing.append("progress.sampler_rng_state is empty")
    rng_recorded = (state.get("torch_rng_state") is not None
                    and state.get("python_rng_state") is not None)
    if not rng_recorded:
        exact_missing.append("no torch/python RNG state in training_state.json")

    return {
        "path": path,
        "name": name,
        "step": step,
        "ok": not rejections,
        "rejections": rejections,
        "warnings": warnings,
        "exact": not exact_missing,
        "exact_missing": exact_missing,
        "state": state,
        "progress": progress,
        "optimizer": optimizer,
        "marker": marker,
    }

def discover_checkpoints(root, run_name, *, model_id=None, profile=None, config_hash=None,
                         dataset_manifest_hash=None, require_marker=None):
    """Every ``step-*`` generation of ``run_name`` under ``root``, each with its verdict.

    Returns ``{"root", "run_dir", "candidates", "accepted", "rejected", "reason"}``. Ordering is by
    step, highest last. ``.incomplete-*`` staging dirs and ``.prev-*`` rollback generations are
    enumerated so their rejection is VISIBLE (silently skipping them is how a torn copy used to be
    mistaken for "nothing to resume"). An absent run directory is not an error here — the caller
    decides whether that means "fresh start" or "nothing verifies".
    """
    root, run_name = _reject_prohibited(root), str(run_name)
    run_dir = Path(root) / run_name
    result = {"root": Path(root), "run_dir": run_dir, "candidates": [], "accepted": [],
              "rejected": [], "reason": "ok"}
    if not run_dir.is_dir():
        result["reason"] = f"{run_dir} does not exist"
        return result
    entries = []
    for child in sorted(run_dir.iterdir()):
        if not child.is_dir() or child.name.startswith("."):
            continue
        entries.append(child)
    for child in entries:
        verdict = verify_resume_candidate(
            child, run_name=run_name, model_id=model_id, profile=profile,
            config_hash=config_hash, dataset_manifest_hash=dataset_manifest_hash,
            require_marker=require_marker)
        result["candidates"].append(verdict)
        if verdict["ok"]:
            result["accepted"].append(verdict)
        else:
            result["rejected"].append(verdict)
    result["candidates"].sort(key=lambda item: (item["step"] is not None, item["step"] or -1))
    result["accepted"].sort(key=lambda item: item["step"])
    if not entries:
        result["reason"] = f"{run_dir} holds no step-* directory"
    return result

def select_resume_checkpoint(root, run_name, *, model_id=None, profile=None, config_hash=None,
                             dataset_manifest_hash=None, require_marker=None, verbose=True):
    """The HIGHEST step that passes every check, plus why every newer candidate was rejected.

    Falls back to an older valid generation when the newest is corrupt, and prints the rejected
    candidates with their reasons so the fallback is never silent. Returns the discovery dict plus
    ``selected`` (a candidate dict or None) and ``newer_rejected``.
    """
    found = discover_checkpoints(root, run_name, model_id=model_id, profile=profile,
                                config_hash=config_hash,
                                dataset_manifest_hash=dataset_manifest_hash,
                                require_marker=require_marker)
    selected = found["accepted"][-1] if found["accepted"] else None
    found["selected"] = selected
    found["newer_rejected"] = ([c for c in found["rejected"]
                                if selected is not None and (c["step"] or 0) > selected["step"]]
                               if selected else found["rejected"])
    if verbose:
        print(f"checkpoint scan: {found['run_dir']} -> {len(found['accepted'])} usable, "
              f"{len(found['rejected'])} rejected")
        for verdict in found["rejected"]:
            print(f"  [rejected] {verdict['name']}: " + "; ".join(verdict["rejections"]))
        for verdict in found["accepted"]:
            print(f"  [usable]   {verdict['name']} (exact={verdict['exact']}"
                  + ("" if verdict["exact"] else f", {verdict['exact_missing'][0]}") + ")")
        if selected is not None and found["newer_rejected"]:
            print(f"  -> falling back to {selected['name']}: newer candidates failed verification")
    return found

def plan_local_cleanup(local_run_dir, drive_run_dir, *, run_name=None, model_id=None,
                       delete=False, verbose=True):
    """Which local step dirs may be deleted, checked PAIRWISE against the Drive copy.

    Deletion requires, for THAT checkpoint: a Drive counterpart directory, a marker that verifies
    (sha256 of every file), matching identity (run name / step / model id) and a resumable Drive
    copy. The newest local generation and the newest verified Drive generation are never removed.

    The failure that matters: a non-empty local set with an EMPTY drive set used to pass
    ``all([]) is True`` and deleted everything. Here it is an explicit failure, and with
    ``delete=False`` (the notebook default) nothing is removed at all — a dry run states what would
    be.
    """
    local_run_dir = Path(local_run_dir)
    local_dirs = sorted((p for p in local_run_dir.glob("step-*")
                         if p.is_dir() and step_of(p) is not None and ".prev-" not in p.name
                         and ".incomplete-" not in p.name), key=step_of) if local_run_dir.is_dir() else []
    plan = {"local_run_dir": str(local_run_dir), "drive_run_dir": None if drive_run_dir is None
            else str(drive_run_dir), "local": [p.name for p in local_dirs], "pairs": [],
            "deletable": [], "would_delete": [], "performed": [], "ok": True, "reason": "",
            "delete": bool(delete)}
    if not local_dirs:
        plan["reason"] = f"{local_run_dir} holds no step-* checkpoint; nothing to clean up"
        if verbose:
            print(f"cleanup       : {plan['reason']}")
        return plan
    if drive_run_dir is None or not Path(drive_run_dir).is_dir():
        plan["ok"] = False
        plan["reason"] = (f"REFUSING to delete: {len(local_dirs)} local checkpoint(s) but the Drive "
                          f"counterpart {drive_run_dir} is missing/empty. A local copy is deleted "
                          "only when its own Drive copy verifies.")
        if verbose:
            print(f"cleanup       : {plan['reason']}")
        return plan
    drive_dirs = {p.name: p for p in Path(drive_run_dir).glob("step-*") if p.is_dir()}
    if not drive_dirs:
        plan["ok"] = False
        plan["reason"] = (f"REFUSING to delete: {len(local_dirs)} local checkpoint(s) but "
                          f"{drive_run_dir} holds no step-* directory. An empty Drive set is never "
                          "evidence that a sync succeeded.")
        if verbose:
            print(f"cleanup       : {plan['reason']}")
        return plan
    newest_local = max(local_dirs, key=step_of)
    newest_step = step_of(newest_local)
    for local_dir in local_dirs:
        step = step_of(local_dir)
        pair = {"step": step, "local": local_dir.name, "drive": None, "ok": False, "reasons": []}
        drive_dir = drive_dirs.get(local_dir.name)
        pair["drive"] = None if drive_dir is None else str(drive_dir)
        if drive_dir is None:
            pair["reasons"].append(f"no Drive counterpart {local_dir.name} under {drive_run_dir}")
        else:
            ok, why = verify_sync_marker(drive_dir)
            if not ok:
                pair["reasons"].append(f"Drive marker does not verify: {why}")
            marker, marker_why = _read_json_file(drive_dir / SYNC_MARKER)
            if marker is None:
                pair["reasons"].append(f"Drive marker unreadable: {marker_why}")
            else:
                if run_name and marker.get("run_name") and marker["run_name"] != run_name:
                    pair["reasons"].append(f"Drive copy belongs to run {marker['run_name']!r}")
                recorded_step = marker.get("step")
                if recorded_step is not None and int(recorded_step) != int(step):
                    pair["reasons"].append(f"Drive marker records step {recorded_step}, directory "
                                           f"is {local_dir.name}")
                if model_id and marker.get("model_id") and marker["model_id"] != model_id:
                    pair["reasons"].append(f"Drive copy is for model {marker['model_id']!r}")
            resumable, why = checkpoint_ready(drive_dir, require_marker=True, run_name=run_name)
            if not resumable:
                pair["reasons"].append(f"Drive copy is not resumable: {why}")
            local_ok, local_why = checkpoint_ready(local_dir)
            if not local_ok:
                pair["reasons"].append(f"local copy is not resumable: {local_why}")
        if step == newest_step:
            pair["reasons"].append("newest local generation: never removed, it is the one the "
                                   "session would resume from")
        pair["ok"] = not pair["reasons"]
        plan["pairs"].append(pair)
    # ALL OR NOTHING: one local generation without a verified Drive counterpart means the sync did
    # not finish, and a partial delete would leave the session unable to resume from what remains.
    failures = [pair for pair in plan["pairs"] if not pair["ok"] and pair["step"] != newest_step]
    plan["deletable"] = ([] if failures else [pair["local"] for pair in plan["pairs"] if pair["ok"]])
    plan["would_delete"] = list(plan["deletable"])
    if failures:
        plan["ok"] = False
        plan["reason"] = (
            f"REFUSING to delete: {len(failures)} local checkpoint(s) have no verified Drive "
            f"counterpart ({[pair['local'] for pair in failures]}); deleting the others would "
            "destroy the run's only complete history. Resume/clean up after a full sync.")
    elif not plan["deletable"]:
        plan["ok"] = False
        plan["reason"] = ("nothing may be deleted: every local generation is either the newest or "
                          "already verified and pruned")
    elif delete:
        for name in plan["deletable"]:
            target = local_run_dir / name
            shutil.rmtree(target, ignore_errors=True)
            plan["performed"].append(name)
        plan["reason"] = f"deleted {len(plan['performed'])} local checkpoint(s), each verified on Drive"
    else:
        plan["reason"] = (f"dry run (DELETE_LOCAL_AFTER_VERIFIED_SYNC=False): would delete "
                          f"{plan['would_delete']}, kept the newest {newest_local.name}")
    if verbose:
        for pair in plan["pairs"]:
            status = "delete" if pair["ok"] else "keep"
            print(f"  [cleanup] {pair['local']}: {status}"
                  + ("" if pair["ok"] else " — " + "; ".join(pair["reasons"])))
        print(f"cleanup       : {plan['reason']}")
    return plan

# Report the rename capability of the paths this session will actually install into, once each.
# A `copy_only` answer is not an error: it is the honest description of a non-atomic mount, and every
# marker written from here on records which mode produced it.
for _sync_parent in ([DRIVE_PATHS["checkpoints"]] if DRIVE_ROOT is not None else [LOCAL_CKPT_ROOT]):
    _mode, _why = rename_mode_for(_sync_parent)
    print(f"rename probe   : {_sync_parent} -> {_mode} ({_why})")
    if _mode == "copy_only":
        print("                 every install keeps BOTH generations (copy-then-mark): a crash leaves a "
              "marker-less dst, which no reader accepts, next to an intact .prev-<n> copy")


In [ ]:

# --- 4c. Provide the dataset: reuse, copy from Drive, unzip, or upload -----
DRIVE_DATA_ZIP = ""  # optional: path to the organizers' archive already on Drive
DATASET_UPLOAD_NAME = "dataset.zip"   # filename the upload widget will ask for
SPLIT_FILES = ("train.jsonl", "train_tools_catalog.json")


def _visible_subdirs(path):
    """Child directories of ``path``, skipping any ``*_environment`` name (never entered)."""
    try:
        names = sorted(os.listdir(path))
    except OSError:
        return []
    children = []
    for name in names:
        if _PROHIBITED.search(f"{path}/{name}/"):
            continue
        child = Path(path) / name
        if child.is_dir():
            children.append(child)
    return children


def locate_data_root(base, max_depth=2):
    """Directory holding ``train/train.jsonl``, at most ``max_depth`` levels below ``base``.

    Breadth-first and bounded: it never recurses, and a ``*_environment`` directory is filtered out
    before ``is_dir()`` is even called on it.
    """
    level = [_reject_prohibited(base)]
    for _depth in range(max_depth + 1):
        for candidate in level:
            if (candidate / "train" / "train.jsonl").is_file():
                return candidate
        level = [child for parent in level for child in _visible_subdirs(parent)]
    return None


def copy_train_split(src_root, dst_root):
    """Copy only the named files/dirs of the ``train`` split — never list a *_environment dir."""
    src_split, dst_split = Path(src_root) / "train", Path(dst_root) / "train"
    dst_split.mkdir(parents=True, exist_ok=True)
    for name in SPLIT_FILES:
        if (src_split / name).is_file():
            shutil.copy2(src_split / name, dst_split / name)
    for name in ("audio", "zaic_tools"):
        src, dst = src_split / name, dst_split / name
        if src.is_dir() and not dst.exists():
            shutil.copytree(src, dst)
    return dst_split


def provision_dataset():
    """Ensure ``DATA_ROOT/train/train.jsonl`` exists; return its resolved root."""
    found = locate_data_root(DATA_ROOT)
    if found is not None:
        return found
    if DRIVE_ROOT is not None and locate_data_root(DRIVE_PATHS["data"]):
        copy_train_split(locate_data_root(DRIVE_PATHS["data"]), DATA_ROOT)
        print(f"copied the train split from Drive ({DRIVE_PATHS['data']})")
    elif str(DRIVE_DATA_ZIP).strip():
        safe_unzip(Path(str(DRIVE_DATA_ZIP).strip()).expanduser(), DATA_ROOT)
    elif IN_COLAB:
        from google.colab import files  # noqa: PLC0415 - Colab only
        print(f"Upload the LifeMate archive as {DATASET_UPLOAD_NAME}. Only `train/` is used; "
              "the test splits are never read.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("nothing uploaded; re-run this cell and pick the dataset archive")
        name = next(iter(uploaded))
        archive = Path("/content") / name
        if archive.suffix == ".zip":
            safe_unzip(archive, DATA_ROOT)
        else:
            raise RuntimeError(
                f"uploaded {name} is not a .zip archive. Upload the LifeMate data archive (or set "
                "DRIVE_DATA_ZIP to its path on Drive) so the notebook can place train/ under "
                f"{DATA_ROOT}."
            )
    else:
        raise RuntimeError(
            f"{DATA_ROOT}/train/train.jsonl not found and no source configured. Provide the official "
            f"LifeMate data tree (train/train.jsonl, train/train_tools_catalog.json, train/audio/) "
            f"under {DATA_ROOT}, or set DRIVE_DATA_ZIP."
        )
    root = locate_data_root(DATA_ROOT)
    if root is None:
        raise RuntimeError(
            f"the archive was unpacked but no train/train.jsonl was found under {DATA_ROOT}. It "
            "probably nests the split differently; set DATA_ROOT to the directory that contains "
            "`train/`, then re-run this cell."
        )
    return root


DATA_ROOT = provision_dataset()
print("data root      :", DATA_ROOT)
print("train split    :", DATA_ROOT / "train" / "train.jsonl")
print("tool catalog   :", (DATA_ROOT / "train" / "train_tools_catalog.json").is_file())


## 5. Dataset preparation + measured token lengths

Purpose: reuse `scripts/prepare_dataset.py` (it owns the trajectory/prompt build, the target-value
visibility check and the truncation accounting), then **measure** the token-length distribution on a
CPU sample so the sequence budget is a measurement, not a guess. Cost: 1-3 min CPU, no GPU, no CU
beyond the idle runtime.

What is produced, and what is deliberately never produced:

* `train.jsonl` / `val.jsonl` decision points split by the repo's own group-aware splitter;
* `manifest.json` / `stats.json` with the truncation and drop reasons;
* no test labels anywhere: only `<data-root>/train/` is read, `public_test` is never touched.

Rebuilding is cheap and deterministic (same seed → same split), so this cell re-runs the fit check
once the measured p99 has shifted the budget.


In [ ]:
# --- 5a. Build (or reuse) the SFT dataset and measure token lengths ---------
import random  # deterministic sampling of records for the token measurement

TRAIN_DATASET_DIR = LOCAL_DATASET_ROOT / "train"
MANIFEST_PATH = TRAIN_DATASET_DIR / "manifest.json"
# One CPU tokenizer for the whole measurement pass (~10 MB, no CUDA context). It outlives this cell
# on purpose; section 6b's release_model_refs() deletes TOKENIZER_CACHE with the model, so nothing
# measured here can keep a tokenizer (or anything else) alive into the training subprocess.
TOKENIZER_CACHE = {}


def prepare_dataset(seq_budget):
    """Run the repo's data builder at one sequence budget; returns the manifest dict.

    Feature flags reach the builder through its OWN augmentation switches, and only when the
    owning feature is enabled: with every flag off `feature_prepare_args()` is `[]`, so the command
    below is byte-identical to a feature-free run. The translation lives in cell 2a
    (`feature_prepare_args`), never here, so there is exactly one mapping to keep correct.
    """
    args = [
        "--data-root", DATA_ROOT,
        "--split", "train",
        "--output", LOCAL_DATASET_ROOT,
        "--tokenizer", MODEL_ID,
        "--max-seq-length", int(seq_budget),
        "--val-ratio", 0.1,
        "--seed", SEED,
    ]
    require_flags("prepare_dataset.py", ["--data-root", "--split", "--output", "--tokenizer",
                                         "--max-seq-length", "--val-ratio", "--seed"])
    feature_args = feature_prepare_args(FEATURE_CONFIG)
    if feature_args:
        # Feature-detect every augmentation flag this checkout would receive; a pin without them
        # must say so instead of dying inside argparse.
        require_flags(
            "prepare_dataset.py",
            [flag for flag in feature_args if flag.startswith("--")],
            hint="This checkout's prepare_dataset.py has no augmentation flag for the enabled "
                 "feature. Pin BRANCH_OR_COMMIT to a commit that ships it, or disable the feature "
                 "in cell 2a.",
        )
        args += feature_args
        print(f"prepare_dataset.py feature args: {' '.join(feature_args)}")
    print("prepare argv   :", " ".join(str(part) for part in
                                       ["prepare_dataset.py"] + [str(a) for a in args]))
    run_cli("prepare_dataset.py", args)
    if not MANIFEST_PATH.is_file():
        raise RuntimeError(
            f"{MANIFEST_PATH} was not written; inspect the builder output above. It refuses to "
            "silently continue when the split has no tool catalog."
        )
    return json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))


def measured_token_stats(max_records=300):
    """Tokenise a fixed sample of records and report the real length distribution.

    Uses the repo's own ``build_supervised_example`` at a 32k budget so the numbers are
    *untruncated* lengths in the exact tokenisation training will use. CPU only.
    """
    from lifemate.training.collator import build_supervised_example
    from transformers import AutoTokenizer

    path = TRAIN_DATASET_DIR / "train.jsonl"
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not records:
        raise RuntimeError(f"{path} holds no SFT records")
    if MODEL_ID not in TOKENIZER_CACHE:
        TOKENIZER_CACHE[MODEL_ID] = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer = TOKENIZER_CACHE[MODEL_ID]

    rng = random.Random(SEED)
    sample = records if len(records) <= max_records else rng.sample(records, max_records)
    lengths, dropped = [], {}
    for record in sample:
        example = build_supervised_example(
            tokenizer, list(record.get("messages") or []), str(record.get("target") or ""), 32768
        )
        if not example.get("kept"):
            dropped[example.get("reason") or "unknown"] = dropped.get(example.get("reason") or "unknown", 0) + 1
            continue
        lengths.append(example["n_prompt_tokens"] + example["n_target_tokens"])
    if not lengths:
        raise RuntimeError(f"no tokenisable record in the {len(sample)}-record sample: {dropped}")
    lengths.sort()

    def pct(q):
        return int(lengths[min(len(lengths) - 1, int(q * (len(lengths) - 1)))])

    return {
        "method": "tokenizer",
        "tokenizer": MODEL_ID,
        "n_sampled": len(sample),
        "n_tokenised": len(lengths),
        "n_unusable": len(sample) - len(lengths),
        "unusable_reasons": dropped,
        "p50": pct(0.50),
        "p90": pct(0.90),
        "p99": pct(0.99),
        "max": lengths[-1],
        "mean": int(sum(lengths) / len(lengths)),
    }


def dataset_augmentation_matches(payload):
    """Does the manifest's own augmentation block describe the flags selected in cell 2a?

    The builder records ``augmentation`` (enabled set, ratio, seed, transcript digests) exactly when
    a transform ran, so this is the dataset's real identity — not a second fingerprint the builder
    never writes. With every flag off the notebook expects NO block, and the manifest has none.
    """
    wanted = feature_prepare_args(FEATURE_CONFIG)
    block = payload.get("augmentation") if isinstance(payload, dict) else None
    if not wanted:
        return block is None
    if not isinstance(block, dict):
        return False
    enabled = block.get("enabled") or {}
    expected = {
        "noisy_asr": bool(FEATURE_CONFIG.training.noisy_asr_training_enabled),
        "nbest_format": bool(FEATURE_CONFIG.training.nbest_training_format_enabled),
        "slot_masking": bool(FEATURE_CONFIG.training.slot_masking_enabled),
        "tool_dropout": bool(FEATURE_CONFIG.training.tool_dropout_enabled),
    }
    return all(bool(enabled.get(name)) == value for name, value in expected.items())

def existing_manifest():
    """The manifest already on disk, when it was built by this model at this budget."""
    if not MANIFEST_PATH.is_file():
        return None
    try:
        payload = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return None
    if not (TRAIN_DATASET_DIR / "train.jsonl").is_file():
        return None
    return payload


def manifest_is_reusable(payload, budget):
    """True only when the existing manifest describes *this* tokenizer at *this* budget.

    The truncation counts are a function of both, so a manifest from another budget would make
    the pre-training report describe a different run.
    """
    if not payload or DATASET_MODE == "rebuild":
        return False
    fit = payload.get("fit") or {}
    same_budget = fit.get("max_seq_length") == int(budget)
    same_tokenizer = MODEL_ID.split("/")[-1] in str(payload.get("tokenization") or "")
    same_data = str(payload.get("data_root") or "").rstrip("/") == str(Path(DATA_ROOT).resolve()).rstrip("/")
    # Feature flags change the supervised records, so the augmentation block the BUILDER writes is
    # part of the dataset's identity: it appears only when an augmentation was active. Comparing it
    # to the enabled set makes "flags changed -> rebuild" automatic, and an all-off run matches a
    # pre-feature manifest (neither side has the block), so nothing is rebuilt for nothing.
    same_features = dataset_augmentation_matches(payload)
    return bool(same_budget and same_tokenizer and same_data and same_features
                and fit.get("method") == "tokenizer")


# The tier-derived budget from 2b: it already honours the requested MAX_SEQ_LENGTH or the clamp
# that cell printed. The measured p99 below may still shrink it, which rebuilds the fit numbers.
FIRST_BUDGET = int(RUN["max_seq_length"])
manifest = existing_manifest()
if manifest_is_reusable(manifest, FIRST_BUDGET):
    print(f"reusing the prepared dataset at seq {FIRST_BUDGET} (DATASET_MODE={DATASET_MODE})")
else:
    manifest = prepare_dataset(FIRST_BUDGET)

TOKEN_STATS = measured_token_stats()
RUN = derive_run_config(GPU_TIER, TRAINING_METHOD, TOKEN_STATS, MAX_SEQ_LENGTH, MICRO_BATCH_SIZE,
                        GRADIENT_ACCUMULATION_STEPS, NUM_EPOCHS, LEARNING_RATE)
if int(RUN["max_seq_length"]) != int(manifest["fit"]["max_seq_length"]):
    # The measured p99 moved the budget: the truncated/dropped counts must be recomputed at the
    # budget we will actually train with, or the report describes a different run.
    print(f"measured p99 moved the budget to {RUN['max_seq_length']}: rebuilding the fit numbers")
    manifest = prepare_dataset(RUN["max_seq_length"])
print_derived(RUN)


In [ ]:
# --- 5b. Dataset report printed before any training ------------------------
fit = manifest.get("fit") or {}
fit_val = manifest.get("fit_val") or {}
n_train = len((TRAIN_DATASET_DIR / "train.jsonl").read_text(encoding="utf-8").splitlines())
n_val = len((TRAIN_DATASET_DIR / "val.jsonl").read_text(encoding="utf-8").splitlines())
if fit.get("method") != "tokenizer":
    raise RuntimeError(
        "the manifest's fit block came from the chars/4 ESTIMATE, not a tokenizer. Re-run section 5a "
        "with a reachable tokenizer before trusting the truncation numbers."
    )

DATASET_REPORT = {
    "dataset_dir": str(TRAIN_DATASET_DIR),
    "n_train_records": n_train,
    "n_val_records": n_val,
    "effective_max_seq_length": int(manifest["fit"]["max_seq_length"]),
    "train": {
        "n_truncated": fit.get("n_truncated"),
        "n_truncated_fraction": fit.get("n_truncated_fraction"),
        "n_dropped": fit.get("n_dropped"),
        "dropped_reasons": fit.get("dropped_reasons"),
    },
    "val": {
        "n_truncated": fit_val.get("n_truncated"),
        "n_dropped": fit_val.get("n_dropped"),
        "dropped_reasons": fit_val.get("dropped_reasons"),
    },
    "token_lengths_sampled": {k: TOKEN_STATS[k] for k in ("n_sampled", "n_tokenised", "p50", "p90", "p99", "max", "mean")},
    "unusable_in_sample": TOKEN_STATS["unusable_reasons"],
    "split_manifest": {
        "group_hash": manifest.get("group_hash"),
        "n_train_samples": manifest.get("n_train"),
        "n_val_samples": manifest.get("n_val"),
        "seed": manifest.get("seed"),
        "ratio": manifest.get("ratio"),
    },
}

if n_train == 0:
    raise RuntimeError(f"{TRAIN_DATASET_DIR}/train.jsonl is empty")
if n_train + n_val == 0:
    raise RuntimeError(
        f"no usable record at seq {DATASET_REPORT['effective_max_seq_length']}. Read the dropped "
        f"reasons above ({DATASET_REPORT['train']['dropped_reasons']}); raising the sequence budget is "
        "the fix when targets overflow it, not lowering it."
    )
if n_train == 0 or n_val == 0:
    raise RuntimeError(
        f"split produced {n_train} train / {n_val} val records; both must be non-empty for a "
        "validation-loss signal and a meaningful resume check."
    )

print(json.dumps(DATASET_REPORT, indent=2, ensure_ascii=False))
print(
    "\nreading of the numbers: 'truncated' records are still trained on (middle-truncated prompt, "
    "request and tool catalog preserved); 'dropped' records were refused because the target could "
    "not be supervised — a dropped target is never replaced by a synthesised label."
)


## 6. Load Qwen3.5-4B, verify compatibility, attach the adapter

Purpose: prove the checkpoint is real, that this `transformers` release can build it, that the LoRA
targets are discovered **from the architecture** (Qwen's hybrid linear-attention projections
included — never a hardcoded Llama q/k/v/o/gate/up/down list), and that gradients actually reach the
adapter. Cost: 3-10 min on GPU (weights ~8.7 GB from the HF hub, cached afterwards) — the single
largest fixed cost after training itself.

QLoRA specifics: the base goes into 4-bit NF4 with the **compute** dtype matching the platform
(bf16 everywhere except T4, which uses fp16), `prepare_model_for_kbit_training` is applied so the
frozen 4-bit base still propagates adapter gradients, and a missing `bitsandbytes` is a hard error.

The model is loaded here for verification and then released: the trainer runs as a subprocess in
section 8, so holding a second copy would just halve the VRAM available to it.

The gradient probe uses the **repo's chunked cross-entropy** (`forward_loss`, the same call the
trainer's loop makes): `labels` never reach the model, so `transformers` never upcasts a whole
`[seq, vocab]` logits tensor to fp32 (8192 × 248320 × 4 B = 8.1 GiB at Qwen3.5's vocabulary). The cell
prints the chunk size it used, the loss, how many adapter tensors received a gradient, and the peak
allocated MiB during the probe. With `training.loss_chunk_tokens <= 0` the old single-shot path comes
back, so the probe **refuses to run unchunked** on a real GPU and names the config knob instead.

Release is explicit and verified: `release_model_refs()` drops every name that can hold a module,
a tokenizer or a batch — including `lora_report`, whose `["model"]` **is** the `PeftModel`, which is
why `del trainable_model` alone freed nothing — then re-scans the namespace for residual references
and raises naming them. `torch.cuda.empty_cache()` runs only after that scan passes; on its own it
frees nothing that is still referenced.

On a GPU without bf16 (T4) this section prints the fp16 warning again, next to the config it is
about: fp16 relies on `torch.amp.GradScaler` for loss scaling in the repo trainer, a T4 user should
prefer `TRAINING_METHOD="qlora"`, and `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` (exported in
section 1) is what keeps allocator fragmentation down at long sequence lengths.


In [ ]:

# --- 6a. Effective config: one YAML the repo itself parses ------------------
CONFIG_DIR = LOCAL_OUTPUT_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = CONFIG_DIR / (f"colab_{CLI_TUNING_MODE}" + f"_seq{RUN['max_seq_length']}.yaml")
CKPT_OUTPUT_DIR = LOCAL_CKPT_ROOT
RUN_DIR = CKPT_OUTPUT_DIR / RUN_NAME

import dataclasses  # noqa: E402 - feature detection for the cadence knobs
from lifemate.config import TrainingConfig  # noqa: E402

_TRAINING_FIELDS = {field.name for field in dataclasses.fields(TrainingConfig)}
# BOTH cadences must exist AND the trainer must actually consult save_steps; the current checkout
# reads `save_strategy == "steps"` and gates on `log_every`, so `steps` alone does not guarantee the
# requested cadence — the flag is still passed (it is the authority when it exists) and the README
# records that this trainer's step cadence is coupled to logging_steps.
STEP_CADENCE_SUPPORTED = bool({"save_steps", "eval_steps"} <= _TRAINING_FIELDS)
SAVE_STRATEGY = "steps" if STEP_CADENCE_SUPPORTED else "epoch"

def config_overrides():
    """Repo config overrides for this run, INCLUDING the feature switchboard.

    ``feature_config_overrides()`` is empty when every flag in cell 2a is off, so the YAML stays
    byte-identical to a feature-free run; when a flag is on, the generated config is what carries it
    into ``scripts/train.py`` (``load_config`` applies ``features.*`` with its dotted-override
    engine) and into ``config_fingerprint`` provenance.

    ``training.quantization`` must agree with ``training.tuning_mode`` — the dataclass
    rejects a qlora run whose quantization is "none" — and both are part of the config
    fingerprint, so a resume across a mismatch is refused by the trainer itself.
    """
    return {
        "data.root": str(DATA_ROOT),
        "model.name": MODEL_ID,                     # explicit: the repo default is a different model
        "model.device": "cuda",
        "model.torch_dtype": PRECISION,
        "training.tuning_mode": CLI_TUNING_MODE,    # runtime-owned, never an opaque value
        "training.quantization": "nf4" if CLI_TUNING_MODE == "qlora" else "none",
        "training.precision": PRECISION,
        "training.output_dir": str(CKPT_OUTPUT_DIR),
        "training.max_seq_length": int(RUN["max_seq_length"]),
        "training.per_device_train_batch_size": int(RUN["micro_batch"]),
        "training.gradient_accumulation_steps": int(RUN["grad_accum"]),
        "training.num_train_epochs": float(RUN["epochs"]),
        "training.learning_rate": float(RUN["learning_rate"]),
        "training.weight_decay": 0.01,
        "training.warmup_ratio": 0.03,
        "training.lr_scheduler_type": "cosine",
        "training.gradient_checkpointing": True,
        "training.gradient_clip_norm": 1.0,
        "training.logging_steps": 10,
        # Cadence: step-based when this checkout knows `training.save_steps` (0 = fall back to
        # logging_steps), otherwise the shipped per-epoch strategy. Feature-detected from the
        # dataclass, not assumed, so an older checkout keeps working.
        "training.save_strategy": SAVE_STRATEGY,
        "training.eval_strategy": SAVE_STRATEGY,
        **({"training.save_steps": int(SAVE_EVERY_N_STEPS),
            "training.eval_steps": int(SAVE_EVERY_N_STEPS)} if STEP_CADENCE_SUPPORTED else {}),
        "training.save_total_limit": 2,
        "training.seed": int(SEED),
        "training.dataloader_num_workers": 0,
        "training.report_to": [],                   # W&B is opt-in, never required
        "lora.r": int(RUN["lora_r"]),
        "lora.alpha": int(RUN["lora_alpha"]),
        "lora.dropout": float(RUN["lora_dropout"]),
        "lora.target_modules": None,                # None => discovery from the real module names
        "lora.bias": "none",
        **feature_config_overrides(FEATURE_CONFIG),
    }


def build_config():
    """Load the repo base config + overrides, prove the invariants, dump for the CLIs."""
    cfg = load_config(str(REPO_DIR / "configs" / "base.yaml"), config_overrides())
    if cfg.model.name != MODEL_ID:
        raise RuntimeError(f"config model.name is {cfg.model.name!r}, expected {MODEL_ID!r}")
    if cfg.training.tuning_mode != CLI_TUNING_MODE:
        raise RuntimeError(
            f"config tuning_mode is {cfg.training.tuning_mode!r}, expected {CLI_TUNING_MODE!r}"
        )
    # The features this run claims are the features the config FILE carries: a mismatch would mean
    # the subprocess trains a different experiment than the report describes.
    if FEATURE_CONFIG is not None:
        from lifemate.features.config import training_feature_fingerprint  # noqa: PLC0415

        file_features = getattr(cfg, "features", None)
        if file_features is None:
            raise RuntimeError(
                "the loaded config has no `features` section, so cell 2a's flags cannot reach the "
                "trainer. This checkout's lifemate.config has no AppConfig.features; pin "
                "BRANCH_OR_COMMIT to a commit that has it, or keep every flag OFF."
            )
        expected = training_feature_fingerprint(FEATURE_CONFIG)
        actual = training_feature_fingerprint(file_features)
        if expected != actual:
            raise RuntimeError(
                f"the generated config's training-feature fingerprint ({actual}) differs from cell "
                f"2a's ({expected}). The YAML did not receive the overrides; refusing to launch a "
                "run whose report would describe different flags than its config."
            )
        print(f"features in cfg: training fingerprint {actual} matches cell 2a")
    dump_config(cfg, CONFIG_PATH)
    return cfg


import math  # noqa: E402 - total-step estimate below

from lifemate.training.trainer import config_fingerprint, optimizer_steps_per_epoch  # noqa: E402

cfg = build_config()
CONFIG_HASH = config_fingerprint(cfg)
print(f"config written : {CONFIG_PATH}")
print(f"config hash    : {CONFIG_HASH}  (a resume must match this)")
if FEATURE_REPORT is not None:
    # RUN is the run's own report dict (section 2b) and it is synced as run_report.json; the
    # feature report belongs to it, so a downstream reader never has to guess which flags ran.
    RUN["features"] = FEATURE_REPORT
    print(f"feature mode   : {FEATURE_REPORT['mode']} | enabled "
          f"{FEATURE_REPORT['enabled'] or 'none'} | training fingerprint "
          f"{FEATURE_REPORT['training_fingerprint']}")
print(f"tuning_mode    : {cfg.training.tuning_mode} | quantization {cfg.training.quantization} "
      f"| precision {cfg.training.precision} | seq {cfg.training.max_seq_length}")
print(f"batch          : micro {cfg.training.per_device_train_batch_size} x accum "
      f"{cfg.training.gradient_accumulation_steps} = "
      f"{cfg.training.per_device_train_batch_size * cfg.training.gradient_accumulation_steps} sequences")
if PRECISION == "float16":
    print(
        "\n!! PRECISION=float16 (no bf16 on this GPU): fp16 training relies on torch.amp.GradScaler\n"
        "   inside the repo trainer for loss scaling, and that fp16+scaler path is NOT verified here.\n"
        f"   This run is tuning_mode={CLI_TUNING_MODE} at seq {cfg.training.max_seq_length}. On a T4\n"
        "   prefer TRAINING_METHOD=\"qlora\" (cell 2b derives it when 16-bit LoRA cannot fit); the\n"
        "   4-bit base is what makes 8192 tokens fit in 16 GB.\n"
    )

STEPS_PER_EPOCH = optimizer_steps_per_epoch(
    DATASET_REPORT["n_train_records"],
    cfg.training.per_device_train_batch_size,
    cfg.training.gradient_accumulation_steps,
)
TOTAL_STEPS = int(math.ceil(STEPS_PER_EPOCH * cfg.training.num_train_epochs))
print(f"steps/epoch    : {STEPS_PER_EPOCH} -> horizon {TOTAL_STEPS} "
      f"(MAX_STEPS={MAX_STEPS or 'unset'} caps it)")


In [ ]:
# --- 6b. Load the checkpoint through the repo loader, verify adapter + grads --
import inspect  # noqa: E402
import math  # noqa: E402 - isfinite() gates the loss-equivalence verdict

from lifemate.models.loader import count_parameters, load_model_and_tokenizer  # noqa: E402
from lifemate.training.lora_finetuning import discover_target_modules  # noqa: E402
from lifemate.training.trainer import chunked_causal_lm_loss, forward_loss  # noqa: E402
import torch.nn.functional  # noqa: E402 - the reference CE in chunk_equivalence

COMPUTE_DTYPE = torch.bfloat16 if PRECISION == "bfloat16" else torch.float16

LOADER_KWARGS = set(inspect.signature(load_model_and_tokenizer).parameters)
if "quantization" not in LOADER_KWARGS:
    raise RuntimeError(
        "this checkout's load_model_and_tokenizer has no `quantization` argument, so the repo's "
        f"4-bit path is absent (it accepts: {sorted(LOADER_KWARGS)}). Pin BRANCH_OR_COMMIT to a "
        "commit that has the QLoRA path, or set TRAINING_METHOD=lora (16-bit adapters). Section 6 "
        "will not hand-roll a BitsAndBytesConfig that the trainer's own loader would never see."
    )
if CLI_TUNING_MODE == "qlora":
    from lifemate.models.loader import require_bitsandbytes  # noqa: PLC0415 - QLoRA only

    bnb = require_bitsandbytes()   # raises LifeMateError with the pip command when absent
    print(f"bitsandbytes   : {getattr(bnb, '__version__', 'unknown')} (NF4 4-bit available)")


def load_for_verification():
    """Load the training model exactly as scripts/train.py will (same loader, same kwargs)."""
    bundle = load_model_and_tokenizer(
        MODEL_ID,
        torch_dtype=PRECISION,
        device="cuda",
        tuning_mode=CLI_TUNING_MODE,
        checkpoint=None,
        attn_implementation=cfg.model.attn_implementation,
        trust_remote_code=cfg.model.trust_remote_code,
        trainable=True,
        for_training=True,
        max_new_tokens=cfg.agent.max_new_tokens,
        quantization=cfg.training.quantization,
    )
    if bundle.model is None or bundle.tokenizer is None:
        raise RuntimeError("the loader returned no model/tokenizer")
    return bundle.model, bundle.tokenizer


def probe_loss(model, batch, chunk_tokens):
    """One forward WITHOUT labels, then the repo's own chunked cross-entropy.

    ``labels`` must never reach the model: ``transformers``' ``ForCausalLMLoss`` upcasts the whole
    ``[seq, vocab]`` logits tensor to fp32 before the fused CE — 8192 x 248320 x 4 B ≈ 8.1 GB in a
    single allocation at Qwen3.5's vocabulary. ``forward_loss`` strips ``labels``, asks for logits
    and slices the loss over the sequence, exactly like the trainer's own loop, so the notebook
    duplicates no loss maths and the chunk size is the one the run itself will use.
    """
    chunk = int(chunk_tokens)
    if chunk <= 0:
        # With ``training.loss_chunk_tokens <= 0`` the trainer falls back to the single-shot path,
        # so the probe would silently re-introduce the 8.1 GB upcast. Refuse that on a real GPU.
        raise RuntimeError(
            f"training.loss_chunk_tokens={chunk}: the gradient probe refuses to run UNCHUNKED on a "
            "real GPU. Passing labels into the model makes transformers upcast the whole "
            f"[{cfg.training.max_seq_length}, {VOCAB}] logits tensor to fp32 = "
            f"{cfg.training.max_seq_length * VOCAB * 4 / 2**30:.2f} GiB in one allocation. Set "
            "`training.loss_chunk_tokens` (configs/base.yaml, shipped default 1024; also settable "
            "through config_overrides() in 6a) to a positive value and re-run section 6b."
        )
    if "labels" not in batch:
        raise RuntimeError("probe batch carries no labels; supervised CE cannot be computed")
    return forward_loss(model, batch, chunk)


def _supervised_shift_counts(labels):
    """Per-position count of supervised (``label != -100``) targets AFTER the causal shift.

    Index ``i`` of the result is the target predicted by logits position ``i``; the CE both paths
    compute pairs ``logits[..., :-1, :]`` with ``labels[..., 1:]``, so position 0 of a batch can
    never be a target and is excluded here instead of being silently dropped later.
    """
    shifted = labels[..., 1:]
    return (shifted != -100).sum(dim=0)

def chunk_equivalence(model, batch, chunk_tokens, max_tokens=256, tolerance=1e-4):
    """Prove the chunked CE equals the full single-shot CE on the SAME slice and the SAME weights.

    The slice is chosen by where the SUPERVISION is, not by where the batch starts: LifeMate masks
    prompt tokens with ``-100``, so taking the first ``max_tokens`` positions of a long prompt could
    hold zero supervised tokens. The reference CE over an all-ignored slice is ``0/0`` = NaN, and
    ``NaN > 1e-4`` is False — the old check therefore PASSED on exactly the batch it could not
    score. So:

    1. the window (up to ``max_tokens`` compared labels) is the contiguous run with the most
       supervised targets, located with a sliding sum;
    2. ``n_supervised_tokens`` is counted and a zero count is INVALID, never success;
    3. the reference (whole-slice fp32 ``cross_entropy``) and the chunked path both use
       ``logits[..., :-1, :]`` against ``labels[..., 1:]`` — the same causal shift;
    4. ``math.isfinite`` is asserted on the reference, the chunked value AND the delta, so NaN/Inf
       becomes an error instead of a comparison that is false;
    5. a delta above ``tolerance`` raises here, not in the caller.

    The reference is deliberately bounded by ``max_tokens``: at the full 8192-token budget it IS the
    8.1 GB allocation this probe exists to avoid. The chunked path stays the production loss — this
    function only ever compares.

    Returns ``{"seq_tokens", "window_start", "n_supervised_tokens", "chunk_used", "single_shot",
    "chunked", "delta", "tolerance"}``.
    """
    if "labels" not in batch:
        raise RuntimeError("no labels in the batch: supervised CE cannot be computed")
    labels = batch["labels"]
    width = int(labels.shape[-1])
    if width < 2:
        raise RuntimeError(f"the batch holds {width} position(s); the causal shift needs at least 2")
    counts = _supervised_shift_counts(labels)
    span = max(1, min(int(max_tokens), width - 1))
    per_position = [int(value) for value in counts.tolist()]
    # Sliding sum over `span` compared labels; the first best start wins, so the choice is stable.
    window_sum = sum(per_position[:span])
    start, best = 0, window_sum
    for candidate in range(1, len(per_position) - span + 1):
        window_sum += per_position[candidate + span - 1] - per_position[candidate - 1]
        if window_sum > best:
            start, best = candidate, window_sum
    # Slice one extra leading position: `chunked_causal_lm_loss` shifts internally, so the compared
    # labels are `slice[..., 1:]` — the same shift the reference applies.
    stop = start + span + 1
    sliced = {key: value[..., start:stop] for key, value in batch.items()
              if hasattr(value, "shape") and value.shape[-1] == width}
    sliced_labels = sliced["labels"]
    n_supervised = int((sliced_labels[..., 1:] != -100).sum().item())
    if n_supervised <= 0:
        raise RuntimeError(
            f"INVALID equivalence test: the {span}-position window at offset {start} of a "
            f"{width}-position batch has NO supervised tokens (every label is -100). A prompt-only "
            "slice scores 0/0 and would make the check pass on a NaN; widen max_tokens or probe a "
            "trained record whose target survives truncation."
        )
    seq_tokens = int(sliced_labels.shape[-1]) - 1
    test_chunk = max(1, seq_tokens // 4)      # strictly smaller than the window, so slicing happens
    with torch.no_grad():
        logits = model(**{k: v for k, v in sliced.items() if k != "labels"}).logits
        single = float(torch.nn.functional.cross_entropy(
            logits[..., :-1, :].float().reshape(-1, logits.shape[-1]),
            sliced_labels[..., 1:].reshape(-1), ignore_index=-100))
        chunked = float(chunked_causal_lm_loss(logits, sliced_labels, test_chunk))
    for label, value in (("single-shot", single), ("chunked", chunked)):
        if not math.isfinite(value):
            raise RuntimeError(
                f"the {label} CE is {value} on the {seq_tokens}-position window at offset {start} "
                f"({n_supervised} supervised tokens): a non-finite loss cannot be compared, and "
                "`NaN > tolerance` is False, which is how this check used to pass silently. The "
                "forward pass or the label shift is wrong; do not train through it."
            )
    delta = abs(single - chunked)
    if not math.isfinite(delta):
        raise RuntimeError(f"the loss delta is {delta}; refusing to compare non-finite losses")
    if delta > float(tolerance):
        raise RuntimeError(
            f"the chunked CE ({chunked}) disagrees with the single-shot CE ({single}) by {delta:.2e} "
            f"on {seq_tokens} positions ({n_supervised} supervised, {test_chunk}-token chunks): the "
            "chunked loss is not the loss the trainer computes. Fix chunked_causal_lm_loss; do not "
            "train through it."
        )
    return {"seq_tokens": seq_tokens, "window_start": start, "n_supervised_tokens": n_supervised,
            "chunk_used": test_chunk, "single_shot": round(single, 6), "chunked": round(chunked, 6),
            "delta": delta, "tolerance": float(tolerance)}

raw_model, tokenizer = load_for_verification()
print(f"base loaded    : {MODEL_ID} | mode={cfg.training.tuning_mode} quant={cfg.training.quantization} "
      f"| dtype {PRECISION}")
print(f"architecture   : {type(raw_model).__name__} | "
      f"config.architectures={getattr(getattr(raw_model, 'config', None), 'architectures', None)}")
if CLI_TUNING_MODE == "qlora":
    _qcfg = getattr(getattr(raw_model, "config", None), "quantization_config", None)
    if not _qcfg:
        raise RuntimeError(
            "tuning_mode=qlora but the loaded model carries no quantization_config: the base is NOT "
            "4-bit, so this run would be LoRA mislabelled as QLoRA. Stop and inspect the loader."
        )
    print(f"quantization   : {_qcfg if isinstance(_qcfg, dict) else _qcfg.to_dict()}")
before = count_parameters(raw_model)

# Qwen's hybrid projections come from the model's own module names, never a Llama-shaped list.
discovered = discover_target_modules(raw_model)
if not discovered:
    raise RuntimeError(
        "LoRA target discovery found no attention/MLP projection. Do not hand-write "
        "q_proj/k_proj/... for a different architecture: extend "
        "lifemate.training.lora_finetuning.HYBRID_SUFFIXES or set lora.target_modules deliberately."
    )
print(f"discovered     : {len(discovered)} projection suffixes -> {', '.join(discovered)}")

if CLI_TUNING_MODE == "qlora":
    from lifemate.training.lora_finetuning import configure_qlora  # noqa: PLC0415

    lora_report = configure_qlora(raw_model, tokenizer, cfg)
else:
    from lifemate.training.lora_finetuning import configure_lora  # noqa: PLC0415

    lora_report = configure_lora(raw_model, cfg)
trainable_model = lora_report["model"]
if lora_report["trainable_params"] <= 0:
    raise RuntimeError("the adapter was attached but zero parameters are trainable: nothing would train")
print(f"adapter        : r={lora_report['r']} alpha={lora_report['alpha']} "
      f"targets={lora_report['n_target_modules']} tensors={lora_report['n_adapter_tensors']}")
print(f"trainable      : {lora_report['trainable_params']:,} / {lora_report['total_params']:,} "
      f"({lora_report['trainable_percentage']}%) grad_ckpt={lora_report['gradient_checkpointing']}")

# Real gradient-flow probe: forward + backward on one record, then inspect the adapter grads.
from lifemate.training.collator import SupervisedCollator, build_supervised_example, to_torch_batch  # noqa: E402

_probe_records = [
    json.loads(line)
    for line in (TRAIN_DATASET_DIR / "train.jsonl").read_text(encoding="utf-8").splitlines()[:4]
]
_collator = SupervisedCollator.from_tokenizer(tokenizer)
_examples = [
    build_supervised_example(tokenizer, list(r["messages"]), str(r["target"]), cfg.training.max_seq_length)
    for r in _probe_records
]
_examples = [ex for ex in _examples if ex.get("kept")][:1]
if not _examples:
    raise RuntimeError("the first records cannot be tokenised at this sequence length; see section 5")
trainable_model.train()
_batch = to_torch_batch(_collator(_examples), "cuda")
# Peak as loaded, BEFORE the probe resets the counter: section 7a quotes this as the measured
# weight cost, so it must stay the load-time number, not a probe-local one.
_PEAK_LOAD_MIB = int(torch.cuda.max_memory_allocated() // 2**20)
torch.cuda.reset_peak_memory_stats()
_PROBE_BASELINE_MIB = int(torch.cuda.memory_allocated() // 2**20)
with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
    _loss = probe_loss(trainable_model, _batch, cfg.training.loss_chunk_tokens)
    _loss_value = float(_loss.detach())
_loss.backward()
_PROBE_PEAK_MIB = int(torch.cuda.max_memory_allocated() // 2**20)
_trainable = [(n, p) for n, p in trainable_model.named_parameters() if p.requires_grad]
_with_grad = [n for n, p in _trainable if p.grad is not None]
_nonzero = [n for n, p in _trainable if p.grad is not None and bool(p.grad.abs().sum() > 0)]
print(f"loss chunk     : loss_chunk_tokens={cfg.training.loss_chunk_tokens} "
      f"-> {min(int(cfg.training.loss_chunk_tokens), int(_batch['input_ids'].shape[-1]))} sequence "
      f"positions upcast to fp32 per step (unchunked would be "
      f"{_batch['input_ids'].shape[-1]})")
# The equivalence check locates the supervision itself (LifeMate masks prompt tokens with -100, so a
# fixed first-N window can hold zero targets and score a NaN that compares False — i.e. silently
# PASSES). It raises on non-finite values, on an all-masked window and on a real disagreement.
_EQUIV = chunk_equivalence(trainable_model, _batch, cfg.training.loss_chunk_tokens)
print(f"loss check     : {_EQUIV['n_supervised_tokens']} supervised of {_EQUIV['seq_tokens']} compared "
      f"positions (window offset {_EQUIV['window_start']}): chunked CE ({_EQUIV['chunk_used']}-token "
      f"chunks) {_EQUIV['chunked']:.6f} vs single-shot {_EQUIV['single_shot']:.6f} "
      f"(delta {_EQUIV['delta']:.2e}, tolerance {_EQUIV['tolerance']:.0e})")
print(f"gradient probe : loss={_loss_value:.4f} | {len(_with_grad)}/{len(_trainable)} trainable "
      f"tensors got a grad, {len(_nonzero)} non-zero "
      f"| peak allocated during probe {_PROBE_PEAK_MIB} MiB "
      f"(+{_PROBE_PEAK_MIB - _PROBE_BASELINE_MIB} MiB over the loaded model)")
if not _nonzero:
    raise RuntimeError(
        "no non-zero adapter gradient after one backward pass. With gradient checkpointing a frozen "
        "base needs enable_input_require_grads(); configure_lora/configure_qlora call it, so check the "
        "peft version before spending GPU hours."
    )
if _loss_value != _loss_value:
    raise RuntimeError("the probe loss is NaN; fp16 + this batch would poison every adapter weight")
# A single fp32 [seq, vocab] temporary for the whole sequence would cost this much; the chunked path
# must stay far below it or the probe is not actually chunking.
_UNCHUNKED_MIB = int(_batch["input_ids"].shape[-1]) * VOCAB * 4 // 2**20
if _PROBE_PEAK_MIB - _PROBE_BASELINE_MIB >= _UNCHUNKED_MIB:
    print(f"warning: the probe's peak grew {_PROBE_PEAK_MIB - _PROBE_BASELINE_MIB} MiB, at or above "
          f"the {_UNCHUNKED_MIB} MiB a single unchunked fp32 loss temporary would need — inspect "
          f"loss_chunk_tokens ({cfg.training.loss_chunk_tokens}) before trusting this run.")

# Plain values only: ints/strings, nothing that could hold the module alive past the release below.
MODEL_FACTS = {
    "model_id": MODEL_ID,
    "tuning_mode": cfg.training.tuning_mode,
    "quantization": cfg.training.quantization,
    "total_params": before["total"],
    "trainable_params": lora_report["trainable_params"],
    "trainable_pct": lora_report["trainable_percentage"],
    "targets": list(discovered),
    "gradient_checkpointing": bool(lora_report["gradient_checkpointing"]),
    "compute_dtype": str(COMPUTE_DTYPE).replace("torch.", ""),
    "peak_load_mib": _PEAK_LOAD_MIB,
    "grad_tensors_nonzero": len(_nonzero),
    "loss_chunk_tokens": int(cfg.training.loss_chunk_tokens),
    "probe_loss": round(_loss_value, 6),
    "probe_peak_mib": _PROBE_PEAK_MIB,
    "probe_equivalence": _EQUIV,
}
if find_model_refs({"MODEL_FACTS": MODEL_FACTS}):
    raise RuntimeError(f"MODEL_FACTS holds a live module reference: {find_model_refs({'MODEL_FACTS': MODEL_FACTS})}")
print(f"peak during verification: {MODEL_FACTS['peak_load_mib']} MiB allocated")

# Section 7b launches a training subprocess that expects the whole card. `del trainable_model` alone
# freed nothing: lora_report["model"] IS the PeftModel, and the tokenizer/batch were still bound.
RELEASE = release_model_refs()


In [ ]:
# --- 6c. Bind the CLI contract: flags, choices, resume form ----------------
TRAIN_FLAGS = require_flags(
    "train.py",
    ["--config", "--dataset-dir", "--model", "--output-dir", "--seed", "--max-steps", "--tuning", "--run-name"],
    hint="These are the flags section 8 drives the trainer with.",
)

if RUN["quantized_base"] and not QLORA_SUPPORTED:
    # The 4-bit stack itself was verified above; what is missing is the trainer's own quantized
    # weight-loading path. Launching anyway would train a bf16/fp16 base and silently mislabel the
    # run, so stop here with the exact options.
    raise RuntimeError(
        f"TRAINING_METHOD=qlora requested, but this checkout's train.py accepts only {TUNING_CHOICES}. "
        "The NF4 + prepare_model_for_kbit_training stack above is verified (bitsandbytes imported, "
        "adapter gradients flowed), but scripts/train.py owns weight loading and has no quantized "
        "branch, so the notebook cannot inject 4-bit loading into the subprocess it must use.\n"
        "Options:\n"
        f"  1. TRAINING_METHOD=lora (works today; on {GPU_TIER} use MAX_SEQ_LENGTH<="
        f"{seq_cap(GPU_TIER, 'lora')}), or\n"
        "  2. pin BRANCH_OR_COMMIT to a commit that ships `--tuning qlora`.\n"
        "Required repo change: a `--tuning qlora` value + BitsAndBytesConfig in the loader path."
    )


def resolve_flag(script_name, candidates, meaning):
    """First existing spelling of a flag, so the notebook survives a rename."""
    have = script_flags(script_name)
    for name in candidates:
        if name in have:
            return name
    raise RuntimeError(
        f"{script_name} exposes none of {candidates} (needed for the {meaning}). It accepts: "
        f"{', '.join(sorted(have))}. Update the notebook's candidate list to match the checkout."
    )


SEQ_FLAG = resolve_flag("train.py", ["--max-seq-length", "--seq-len"], "sequence budget")
RESUME_FLAG = resolve_flag("train.py", ["--resume-from", "--resume"], "checkpoint resume")

# The feature switchboard is an OPTIONAL CLI slice too. `--enable-feature <id>` is how a TRAINING
# run learns about a flag; a pin without it would silently train with every feature OFF while cell
# 2a claimed otherwise, so the flags this session needs are required and a missing one stops here
# with the exact remedy.
FEATURE_IDS_NEEDED = enabled_training_ids(FEATURE_CONFIG) if FEATURE_CONFIG is not None else []
FEATURE_CLI_SUPPORTED = "--enable-feature" in TRAIN_FLAGS
FEATURE_OVERRIDE_SUPPORTED = "--override" in TRAIN_FLAGS
if FEATURE_IDS_NEEDED and not FEATURE_CLI_SUPPORTED:
    raise RuntimeError(
        f"cell 2a enabled {FEATURE_IDS_NEEDED}, but this checkout's train.py has no "
        "`--enable-feature`. Launching would train with every feature OFF while the run report "
        "claimed otherwise. Options:\n"
        "  1. pin BRANCH_OR_COMMIT to a commit whose train.py ships `--enable-feature` (the "
        "lifemate.features slice), or\n"
        "  2. set every flag in cell 2a to False and re-run it.\n"
        f"train.py accepts: {', '.join(sorted(TRAIN_FLAGS))}"
    )
if not FEATURE_CLI_SUPPORTED:
    print("note: train.py has no --enable-feature; cell 2a must keep every flag OFF on this pin")
if FEATURE_IDS_NEEDED and not FEATURE_OVERRIDE_SUPPORTED:
    print(f"note: train.py has no --override, so the numeric feature parameters "
          f"({FEATURE_IDS_NEEDED} is on) reach the run only through the generated YAML config — "
          "which scripts/train.py loads with the same dotted-override engine")

# Step-based checkpointing is an OPTIONAL CLI affordance: a checkout without it saves per epoch. Ask
# the script's own --help, never assume, and name the missing flag instead of crashing on argparse.
OPTIONAL_FLAGS = {}
for _flag in ("--save-steps", "--eval-steps", "--planned-steps", "--extend-horizon"):
    OPTIONAL_FLAGS[_flag] = _flag in TRAIN_FLAGS
if not OPTIONAL_FLAGS["--save-steps"]:
    print(
        "note: this checkout's train.py has no --save-steps, so SAVE_EVERY_N_STEPS=%d cannot be "
        "honoured and checkpoints follow training.save_strategy=%r (=per epoch). Nothing is silently "
        "rewritten: the run still saves, just less often than the param asks."
        % (SAVE_EVERY_N_STEPS, cfg.training.save_strategy)
    )
print(f"seq flag: {SEQ_FLAG} | resume flag: {RESUME_FLAG} | optional step flags: "
      f"{[flag for flag, present in OPTIONAL_FLAGS.items() if present] or 'none'}")


def train_args(*, max_steps=None, resume=None, output_dir=None):
    """The exact argv section 7/8/9 use; one builder so probe and real run cannot drift."""
    args = [
        "--config", CONFIG_PATH,
        "--dataset-dir", TRAIN_DATASET_DIR,
        "--model", MODEL_ID,
        "--output-dir", output_dir or CKPT_OUTPUT_DIR,
        "--run-name", RUN_NAME,
        "--tuning", CLI_TUNING_MODE,
        "--seed", int(SEED),
        SEQ_FLAG, int(RUN["max_seq_length"]),
    ]
    if max_steps:
        args += ["--max-steps", int(max_steps)]
    # SAVE_EVERY_N_STEPS is driven through the CLI when the checkout exposes it; the config's
    # save_strategy is switched to "steps" only then, so a checkout without the flag keeps its
    # epoch-based default instead of saving never.
    if OPTIONAL_FLAGS["--save-steps"]:
        args += ["--save-steps", int(SAVE_EVERY_N_STEPS)]
        if OPTIONAL_FLAGS["--eval-steps"]:
            args += ["--eval-steps", int(SAVE_EVERY_N_STEPS)]
        if OPTIONAL_FLAGS["--planned-steps"] and TOTAL_STEPS:
            args += ["--planned-steps", int(TOTAL_STEPS)]
            if max_steps and int(max_steps) > int(TOTAL_STEPS):
                args += ["--extend-horizon"]
    # Feature flags: `--enable-feature <id>` for every enabled training|both id, plus the numeric
    # parameters when the checkout accepts --override. With every flag off this is exactly `[]`, so
    # the command is byte-identical to a feature-free run.
    args += feature_train_args(FEATURE_CONFIG, override_supported=FEATURE_OVERRIDE_SUPPORTED)
    if resume:
        args += [RESUME_FLAG, str(resume)]
    return args


print("train argv (no max-steps):", " ".join(str(a) for a in train_args()))


In [ ]:
# --- 6d. Resume target: discovery, verification, report (fresh / auto / explicit) ------
# Everything below reads ARTIFACTS, never a variable that merely exists in this kernel: the ten
# report lines are derived from training_state.json, optimizer_state.pt and the sync marker.
RESUME_REPORT_LABELS = (
    "Resume checkpoint",
    "Previous global step",
    "Previous epoch",
    "Next batch",
    "Optimizer restored",
    "Scheduler restored",
    "Sampler restored",
    "RNG restored",
    "Expected total steps",
)

def resume_mode_from_cli(lines):
    """``(mode, detail)`` parsed from ``train.py``'s own ``resume mode : exact|best-effort`` line.

    The CLI is the authority when it says something: it reports the decision the trainer made after
    inspecting the checkpoint, so the notebook does not compute a second, possibly disagreeing,
    verdict. ``(None, reason)`` means the line is absent (a legacy CLI) and the caller falls back to
    reading ``training_state.json``.
    """
    for line in lines or []:
        text = str(line).strip()
        if text.startswith("resume mode"):
            value = text.split(":", 1)[1].strip() if ":" in text else ""
            if value in ("exact", "best-effort"):
                return value, f"train.py printed '{text}'"
            return None, f"train.py printed an unparsable resume-mode line: {text!r}"
    return None, "train.py printed no 'resume mode :' line (older CLI)"

def feature_resume_comparison(state):
    """``(match, checkpoint_fp, current_fp, differing_ids, detail)`` for a checkpoint's features.

    Reads the two values the TRAINER stores (``training_feature_fingerprint`` and
    ``training_features`` in ``training_state.json``, mirrored inside ``progress``) and compares them
    with cell 2a's own config computed by the same functions. A schema-1 checkpoint from a
    feature-free trainer records neither: the checkpoint is then feature-free, so the comparison is
    against "" and no flags can differ.
    """
    progress = (state or {}).get("progress") if isinstance((state or {}).get("progress"), dict) else {}
    stored_fp = ((state or {}).get("training_feature_fingerprint")
                 or (progress or {}).get("training_feature_fingerprint") or "")
    stored_ids = list((state or {}).get("training_features")
                      or (progress or {}).get("training_features") or [])
    current = feature_resume_identity(FEATURE_CONFIG)
    current_fp = current.get("training_feature_fingerprint") or ""
    current_ids = list(current.get("training_features") or [])
    differing = sorted(set(stored_ids) ^ set(current_ids))
    match = str(stored_fp) == str(current_fp)
    if match:
        detail = (f"checkpoint {stored_fp or '(none recorded)'} == this session "
                  f"{current_fp}; enabled training ids {current_ids or 'none'}")
    else:
        detail = (f"checkpoint {stored_fp or '(none recorded)'} != this session {current_fp}; "
                  f"differing flags: {differing or 'none (only a parameter changed)'}")
    return {"match": match, "checkpoint_fingerprint": str(stored_fp),
            "current_fingerprint": str(current_fp), "differing_ids": differing,
            "checkpoint_ids": stored_ids, "current_ids": current_ids, "detail": detail}

def resume_report(candidate, *, expected_total_steps=None, cli_lines=None):
    """The ten resume facts, from the checkpoint's own contents. Returns a dict.

    Booleans come from verification, not from notebook state: ``optimizer_state.pt`` must have
    LOADED and carry an ``optimizer`` key, its ``scheduler`` key must be non-null, the sampler needs
    a recorded permutation, and the RNG needs recorded torch+python states. A checkpoint written by
    a schema-1 trainer has no ``progress`` block, so sampler/exact resume is honestly ``False``.
    """
    state = candidate.get("state") or {}
    progress = candidate.get("progress") or {}
    optimizer = candidate.get("optimizer") or {}
    cli_mode, cli_detail = resume_mode_from_cli(cli_lines)
    exact_missing = list(candidate.get("exact_missing") or [])
    permutation = list(progress.get("permutation") or [])
    sampler_restored = bool(permutation) and int(progress.get("next_micro_index") or 0) >= 0
    rng_recorded = {
        "torch": state.get("torch_rng_state") is not None,
        "python": state.get("python_rng_state") is not None,
        "cuda": state.get("cuda_rng_state") is not None,
    }
    rng_restored = bool(rng_recorded["torch"] and rng_recorded["python"])
    previous_step = int(progress.get("global_step") or state.get("step") or 0)
    previous_epoch = int(progress.get("epoch_index") or state.get("epoch") or 0)
    next_micro = progress.get("next_micro_index")
    planned = int(progress.get("planned_total_steps") or 0)
    if progress:
        next_batch = (f"{next_micro} (micro-batch index in epoch {previous_epoch}; "
                      f"permutation of {len(permutation)} examples recorded)")
    else:
        next_batch = "unknown: schema-1 checkpoint records no permutation or micro-batch cursor"
    report = {
        "checkpoint": str(candidate.get("path")),
        "previous_global_step": previous_step,
        "previous_epoch": previous_epoch,
        "next_batch": next_batch,
        "optimizer_restored": optimizer.get("optimizer") is True,
        "scheduler_restored": optimizer.get("scheduler") is True,
        "sampler_restored": sampler_restored,
        "rng_restored": rng_restored,
        "rng_recorded": rng_recorded,
        "expected_total_steps": planned or (int(expected_total_steps) if expected_total_steps else 0),
        "expected_total_steps_source": ("training_state.json progress.planned_total_steps" if planned
                                       else "this session's computed horizon"),
        "exact": bool(candidate.get("exact")),
        "exact_missing": exact_missing,
        # One verdict, not two: the CLI's own line wins when it exists (it reports what the trainer
        # decided after inspecting the checkpoint). Without it, exact/best-effort is recomputed from
        # the progress block and the source is stated explicitly.
        "mode": cli_mode or ("exact" if candidate.get("exact") else "best-effort"),
        "mode_source": (f"train.py's own line ({cli_detail})" if cli_mode
                        else f"training_state.json progress block ({cli_detail})"),
        "schema": int(progress.get("schema_version") or (0 if progress else 1)),
        "optimizer_detail": optimizer,
        "warnings": list(candidate.get("warnings") or []),
        "rejections": list(candidate.get("rejections") or []),
        # The training-feature fingerprint is part of a checkpoint's identity, so a resume across a
        # change is a different experiment. The trainer already refuses it; the notebook says so
        # instead of papering over it.
        "features": feature_resume_comparison(state),
    }
    if not report["features"]["match"]:
        report["exact"] = False
        report["exact_missing"] = exact_missing + [
            "training-feature fingerprint differs from this session's: " +
            report["features"]["detail"]
        ]
        if report.get("mode") == "exact":
            report["mode"] = "best-effort"
            report["mode_source"] += " (downgraded: training features differ)"
    return report

def print_resume_report(report):
    """Print EXACTLY the ten labelled lines, then the provenance block beneath them."""
    print("Resume checkpoint:")
    print(f"Previous global step: {report['previous_global_step']}")
    print(f"Previous epoch: {report['previous_epoch']}")
    print(f"Next batch: {report['next_batch']}")
    print(f"Optimizer restored: {bool(report['optimizer_restored'])}")
    print(f"Scheduler restored: {bool(report['scheduler_restored'])}")
    print(f"Sampler restored: {bool(report['sampler_restored'])}")
    print(f"RNG restored: {bool(report['rng_restored'])}")
    print(f"Expected total steps: {report['expected_total_steps']}")
    print(f"\nresume detail  : {report['checkpoint']}")
    print(f"  exact resume : {report['exact']}"
          + ("" if report["exact"] else f" -> missing: {report['exact_missing']}"))
    print(f"  mode source  : {report['mode_source']}")
    print(f"  horizon from : {report['expected_total_steps_source']}")
    print(f"  rng states   : {report['rng_recorded']}")
    features = report.get("features") or {}
    print(f"  training features: checkpoint {features.get('checkpoint_fingerprint') or '(none)'} "
          f"| this session {features.get('current_fingerprint') or '(none)'} | "
          f"{'MATCH' if features.get('match') else 'MISMATCH'}")
    if features and not features.get("match"):
        print("  EXACT RESUME REFUSED: the training-feature fingerprint differs "
              f"(differing flags: {features.get('differing_ids') or 'none (a parameter changed)'}). "
              "Training features change the supervised data, so this is a different experiment, not "
              "a continuation. The trainer refuses the same thing; resume with the config the "
              "checkpoint was trained under, or start a fresh run.")
    if report["warnings"]:
        for warning in report["warnings"]:
            print(f"  warning      : {warning}")
    return report

def resolve_resume_target(run_name, *, mode, explicit_path="", root=None, local_root=None,
                          model_id=None, profile=None, config_hash=None,
                          dataset_manifest_hash=None, expected_total_steps=None, verbose=True):
    """Apply RESUME_MODE semantics. Returns ``{"mode", "run_name", "checkpoint", "report", "notes"}``.

    ``fresh``
        a NEW experiment. A populated ``checkpoints/<run_name>/step-*`` is never reused or
        overwritten: the run name is suffixed with a UTC stamp (``<run>-2026-11-14T101500Z``) and the
        change is printed loudly, so two experiments can never share a step numbering.
    ``auto``
        the newest checkpoint that VERIFIES, searched on Drive first and then locally. When
        candidates EXIST but none verifies this RAISES with every rejected candidate and its reason
        — silently starting from scratch over an existing experiment is exactly the failure mode a
        resume mode exists to prevent. When the run has no ``step-*`` directory at all, auto proceeds
        as a FIRST run and prints every root it searched plus the fact that nothing was rejected:
        there is no existing experiment to lose, and refusing would make the default unusable for a
        first launch.
    ``explicit``
        ``RESUME_FROM_CHECKPOINT``, verified; any failure raises.
    """
    search_roots = []
    if root is not None:
        search_roots.append((Path(root), "drive" if DRIVE_ROOT is not None
                             and str(root).startswith(str(DRIVE_ROOT)) else "local"))
    if local_root is not None and Path(local_root) not in [p for p, _ in search_roots]:
        search_roots.append((Path(local_root), "local"))
    notes, target_name, report, checkpoint = [], str(run_name), None, None

    if mode == "explicit":
        path = Path(str(explicit_path).strip())
        verdict = verify_resume_candidate(path, run_name=target_name, model_id=model_id,
                                         profile=profile, config_hash=config_hash,
                                         dataset_manifest_hash=dataset_manifest_hash)
        if not verdict["ok"]:
            raise RuntimeError(
                "RESUME_MODE='explicit' but RESUME_FROM_CHECKPOINT is not resumable:\n  - "
                + "\n  - ".join(verdict["rejections"])
                + "\nNothing was launched. Fix the path, or set RESUME_MODE='fresh' deliberately."
            )
        checkpoint = path
        report = verdict
        notes.append(f"explicit resume target verified: {path}")
        return {"mode": mode, "run_name": target_name, "checkpoint": checkpoint, "report": report,
                "notes": notes, "searched": []}

    if mode == "auto":
        selected, searched = None, []
        for search_root, kind in search_roots:
            found = select_resume_checkpoint(search_root, target_name, model_id=model_id,
                                            profile=profile, config_hash=config_hash,
                                            dataset_manifest_hash=dataset_manifest_hash,
                                            verbose=verbose)
            searched.append({"root": str(search_root), "kind": kind,
                             "accepted": [c["name"] for c in found["accepted"]],
                             "rejected": {c["name"]: c["rejections"] for c in found["rejected"]},
                             "reason": found["reason"]})
            if found["selected"] is not None:
                selected = dict(found["selected"])
                selected["newer_rejected"] = found["newer_rejected"]
                break
        if selected is None and all(not entry["rejected"] for entry in searched):
            # Nothing to reject: no step-* directory exists for this run name anywhere. That is a
            # FIRST run, not a failed resume, so it proceeds — loudly, naming every root it
            # searched, because a run dir that exists but holds only unreadable candidates (below)
            # is a different situation and refuses.
            print(f"\nRESUME_MODE='auto': no existing step-* checkpoint for run {target_name!r} under "
                  f"{[entry['root'] for entry in searched]} -> this is a first run. Nothing was "
                  "rejected, and no existing experiment could be overwritten.\n")
            return {"mode": "fresh", "run_name": target_name, "checkpoint": None, "report": None,
                    "notes": [f"auto: no existing checkpoint under "
                              f"{[entry['root'] for entry in searched]}; first run"],
                    "searched": searched}
        if selected is None:
            rejected_lines = [f"{entry['root']}: {entry['reason'] or 'no usable candidate'}"
                              for entry in searched]
            for entry in searched:
                for name, reasons in entry["rejected"].items():
                    rejected_lines.append(f"  [rejected] {name}: " + "; ".join(reasons))
            raise RuntimeError(
                "RESUME_MODE='auto' found no checkpoint that verifies, and this cell will NOT "
                "silently start a fresh run (that would mix two experiments' step numbering):\n  - "
                + "\n  - ".join(rejected_lines)
                + "\n\nDo one of:\n"
                "  * set RESUME_MODE='fresh' in section 2 to start a NEW experiment deliberately, or\n"
                "  * set RESUME_MODE='explicit' + RESUME_FROM_CHECKPOINT=<a step-<n> dir>, or\n"
                "  * re-sync the run's checkpoints from Drive and re-run this cell."
            )
        checkpoint = selected["path"]
        report = selected
        notes.append(f"auto resume: newest verified generation is {selected['name']}")
        if report.get("newer_rejected"):
            notes.append(f"older fallback used; newer rejected: "
                         f"{[c['name'] for c in report['newer_rejected']]}")
        return {"mode": mode, "run_name": target_name, "checkpoint": checkpoint, "report": report,
                "notes": notes, "searched": searched}

    if mode != "fresh":
        raise ValueError(f"unknown RESUME_MODE {mode!r}")
    populated = []
    for search_root, kind in search_roots:
        candidate_dir = Path(search_root) / target_name
        if candidate_dir.is_dir():
            steps = sorted((p.name for p in candidate_dir.glob("step-*")), key=step_of)
            if steps:
                populated.append((candidate_dir, kind, steps))
    if populated:
        stamp = time.strftime("%Y-%m-%dT%H%M%SZ", time.gmtime())
        distinct = f"{target_name}-{stamp}"
        for candidate_dir, kind, steps in populated:
            notes.append(f"{kind} {candidate_dir} already holds {len(steps)} checkpoint(s) "
                         f"(latest {steps[-1]})")
        print(f"\n!! RESUME_MODE='fresh' but a populated experiment exists:\n   "
              + "\n   ".join(notes) + f"\n!! starting a DISTINCT experiment: RUN_NAME "
              f"{target_name!r} -> {distinct!r}. Existing checkpoints are untouched; resume them "
              "with RESUME_MODE='auto' if that was the intent.\n")
        target_name = distinct
    else:
        notes.append(f"fresh run: no step-* exists under {[str(r / target_name) for r, _ in search_roots]}")
    return {"mode": mode, "run_name": target_name, "checkpoint": None, "report": None,
            "notes": notes, "searched": [str(r) for r, _ in search_roots]}

def model_profile_name(model_id):
    """Architecture family of ``model_id`` from the repo's own registry, or None when unresolved.

    The checkpoint records ``model_profile`` in ``lifemate_meta.json``; comparing it catches an
    adapter being resumed into a different architecture family, which a name check alone would miss
    when two ids happen to share a tokenizer.
    """
    try:
        from lifemate.models.profiles import resolve_profile  # noqa: PLC0415 - resolved lazily
    except Exception:  # pragma: no cover - profiles module absent
        return None
    profile = resolve_profile(model_id)
    name = str(getattr(profile, "name", "") or "")
    return name or None

MODEL_PROFILE = model_profile_name(MODEL_ID)

def apply_resume_plan(plan):
    """Bind the plan's run name and print the report; called by section 8."""
    global RUN_NAME, RUN_DIR
    if plan["run_name"] != RUN_NAME:
        print(f"RUN_NAME       : {RUN_NAME!r} -> {plan['run_name']!r} (fresh mode, populated run)")
        RUN_NAME = plan["run_name"]
        RUN_DIR = CKPT_OUTPUT_DIR / RUN_NAME
    for note in plan["notes"]:
        print(f"resume note    : {note}")
    if plan["report"] is not None:
        print_resume_report(resume_report(plan["report"], expected_total_steps=TOTAL_STEPS))
    return plan

RESUME_PLAN = resolve_resume_target(
    RUN_NAME,
    mode=RESUME_MODE,
    explicit_path=RESUME_FROM_CHECKPOINT,
    root=DRIVE_PATHS["checkpoints"] if DRIVE_ROOT is not None else None,
    local_root=CKPT_OUTPUT_DIR,
    model_id=MODEL_ID,
    profile=MODEL_PROFILE,
    config_hash=CONFIG_HASH,
    dataset_manifest_hash=manifest.get("group_hash"),
    expected_total_steps=TOTAL_STEPS,
)
RESUME_PLAN = apply_resume_plan(RESUME_PLAN)
if "train_args" in globals():
    # Printed again because a `fresh` decision may have renamed the run: the argv quoted by 6c was
    # built before that, and two different names in the log would be a false trail.
    print("train argv (resume decision applied):", " ".join(str(a) for a in train_args()))


## 7. Memory preflight: estimate, then measure, then the worst case

Purpose: three independent numbers before spending GPU hours.

1. **Estimate** from the repo's `run_memory_preflight` (labelled as a heuristic projection, not a
   measurement) — weights, gradients, optimizer state, activations, total.
2. **Measurement** from a real 2-step training run of the real trainer on the real model, sampled
   from `nvidia-smi` while the child process runs.
3. **Worst case** (cell 7c): the two steps above are only as long as the *first batch* happened to
   tokenise to, which says nothing about the step that will eventually hit the configured ceiling.
   The worst-case probe therefore forces a synthetic batch padded to the ACTUAL configured
   `MAX_SEQ_LENGTH` and runs the whole step: forward → chunked CE → backward → gradient
   accumulation over `GRADIENT_ACCUMULATION_STEPS` micro-batches → gradient clipping → optimizer
   step → one eval batch → one checkpoint save. `reset_peak_memory_stats()` runs immediately before,
   so `max_memory_allocated()` / `max_memory_reserved()` describe exactly that sequence.

The longest **real** prepared sample is probed separately and reported as its own row: the configured
maximum is the allocator ceiling, the longest sample is what most steps actually cost. The two are
never averaged into one number, and the report prints `Estimated VRAM` (heuristic) and
`Observed VRAM` (measured) as different quantities with different labels. A probe that does not fit
STOPS with options — the notebook never lowers `MAX_SEQ_LENGTH` for you, because that would silently
train a different context than the one you configured.

Refusal rule: an estimate that cannot fit the card stops the notebook here, and a measured peak
above 92% of free VRAM stops it before the long run. Cost: 1-2 optimizer steps ≈ 1-4 min GPU; the
checkpoints it writes are deleted immediately (`--max-steps 2` → `step-1`, `step-2`).

**Everything in this section is a no-op on CPU**, with a printed reason: the machine that wrote this
notebook has no CUDA device, so no VRAM number in it was ever measured.


In [ ]:
# --- 7a. Heuristic estimate + explicit refusal ------------------------------
from lifemate.training.memory_preflight import format_report, run_memory_preflight  # noqa: E402

ESTIMATE = run_memory_preflight(param_count=MODEL_FACTS["total_params"], cfg=cfg,
                                seq_len=cfg.training.max_seq_length)
print(format_report(ESTIMATE))
print(
    f"\nreality check: the repo estimate sizes the activation term from a heuristic, so it is an "
    f"ESTIMATE. TRAINABLE params are measured ({MODEL_FACTS['trainable_params']:,}); "
    f"weights are measured at load time ({MODEL_FACTS['peak_load_mib']} MiB peak allocated)."
)

if ESTIMATE["verdict"] == "does_not_fit":
    raise RuntimeError(
        f"the estimate ({ESTIMATE['total_gb']} GB) exceeds the {GPU['vram_total_gb']} GB on "
        f"{GPU['name']}. Lower MAX_SEQ_LENGTH (currently {cfg.training.max_seq_length}) and/or "
        f"MICRO_BATCH_SIZE, raise GRADIENT_ACCUMULATION_STEPS to keep the effective batch, or pick a "
        f"bigger runtime. Suggestions: {ESTIMATE['suggestions']}"
    )
if ESTIMATE["verdict"] == "tight":
    print("warning: the estimate is 'tight' — the measured probe below decides, not this number")


In [ ]:
# --- 7b. Measured preflight: one 2-step run of the real trainer ------------
PROBE_DIR = LOCAL_OUTPUT_ROOT / "probe"
if PROBE_DIR.exists():
    shutil.rmtree(PROBE_DIR)

print("running a 2-step probe with the real trainer (this loads the weights again, on purpose):")
run_cli("train.py", train_args(max_steps=2, output_dir=PROBE_DIR), sample_peak=True, peak_interval=2.0)

_peak_mib = LAST_CLI_PEAK_MIB
PROBE = {
    "sampled_peak_mib": _peak_mib,
    "sampled_peak_gb": None if _peak_mib is None else round(_peak_mib * 2**20 / 1e9, 2),
    "device_total_gb": GPU.get("vram_total_gb"),
    "estimate_total_gb": ESTIMATE["total_gb"],
    "estimate_verdict": ESTIMATE["verdict"],
    "probe_output": str(PROBE_DIR),
}
if _peak_mib is None:
    raise RuntimeError(
        "could not sample device memory during the probe (nvidia-smi unavailable). The probe's own "
        "torch peak line in the output above is the number to use; refusing to guess."
    )
HEADROOM_FRACTION = 0.92
_used_fraction = PROBE["sampled_peak_gb"] / float(GPU["vram_total_gb"])
PROBE["used_fraction_of_total"] = round(_used_fraction, 4)
print(json.dumps(PROBE, indent=2))
if _used_fraction > HEADROOM_FRACTION:
    raise RuntimeError(
        f"the probe peaked at {PROBE['sampled_peak_gb']} GB = {_used_fraction:.0%} of "
        f"{GPU['vram_total_gb']} GB. A full epoch will fragment past that and OOM. Lower "
        "MAX_SEQ_LENGTH (halving it cuts the activation term roughly in half) or MICRO_BATCH_SIZE=1 "
        "with a larger GRADIENT_ACCUMULATION_STEPS, then re-run this section."
    )
print(f"probe fits: {PROBE['sampled_peak_gb']} GB peak of {GPU['vram_total_gb']} GB "
      f"({_used_fraction:.0%}) — real measurement, not the heuristic")

# The probe checkpoints are throwaway: keeping them would collide with the real run's step numbering.
shutil.rmtree(PROBE_DIR, ignore_errors=True)
free_gpu()


In [ ]:
# --- 7c. Worst-case memory probe: the CONFIGURED sequence length, measured ------------
# Section 7b runs two real optimizer steps, but their padded length is whatever the first batch
# happened to tokenise to — a 900-token sample says nothing about the 8192-token step the run will
# eventually take. This probe forces the worst case on purpose:
#
#   forward -> chunked CE -> backward -> accumulate `grad_accum` micro-batches -> clip -> optimizer
#   step -> one eval batch -> one checkpoint save,
#
# with torch's own counters, reset immediately before, so `max_memory_allocated()` /
# `max_memory_reserved()` describe exactly this sequence. Two batches are probed and reported
# SEPARATELY, never as one number: the configured MAX_SEQ_LENGTH (the ceiling the allocator must
# survive) and the longest real prepared sample (what most steps will actually cost).
PROBE_LABELS = (
    "GPU", "CUDA", "PyTorch", "Model", "Precision", "Quantization", "LoRA rank", "Sequence length",
    "Microbatch size", "Gradient accumulation", "Peak allocated VRAM", "Peak reserved VRAM",
    "Free VRAM", "Estimated VRAM", "Observed VRAM",
)

def _probe_batch(tokenizer, seq_len, micro_batch, supervised_tail=64):
    """A synthetic batch padded to ``seq_len`` with a supervised tail and a -100 prompt.

    Built here rather than tokenised from a record so the length is EXACT: any padding of a real
    sample would silently measure a shorter step than the configured maximum. Prompt positions are
    masked like LifeMate's collator does, so the chunked CE sees the same label structure the
    trainer produces.
    """
    seq_len, micro_batch = int(seq_len), int(micro_batch)
    tail = max(1, min(int(supervised_tail), seq_len - 1))
    ids_row = [(i * 7919) % 1000 + 1 for i in range(seq_len)]
    labels_row = [-100] * (seq_len - tail) + ids_row[-tail:]
    batch = {
        "input_ids": torch.tensor([ids_row] * micro_batch, dtype=torch.long),
        "attention_mask": torch.ones((micro_batch, seq_len), dtype=torch.long),
        "labels": torch.tensor([labels_row] * micro_batch, dtype=torch.long),
    }
    return batch

def longest_prepared_sample_length(path):
    """Longest ``input_ids`` length among the prepared SFT records (measured, not assumed).

    Reported next to the configured maximum so the two can never be mistaken for each other: the
    configured value is the allocator ceiling, this one is the typical step.
    """
    path = Path(path)
    if not path.is_file():
        return None
    longest = 0
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            longest = max(longest, len(record.get("input_ids") or []))
    return longest or None

def run_worst_case_probe(model, batch, *, cfg, chunk_tokens, grad_accum, micro_batch,
                         tokenizer=None, checkpoint_dir=None, require_cuda=True,
                         supervised_tail=64):
    """Drive the full worst-case step and return measured VRAM, or a no-op verdict on CPU.

    Returns ``{"ran", "reason", "estimated_gb", "observed_gb", "observed_reserved_gb", "seq_len",
    "source", "steps", "free_vram_gb", "fits"}``. ``source`` is ``"configured_max"`` or
    ``"longest_sample"`` — the caller must never average them.
    """
    cuda = bool(torch is not None and torch.cuda.is_available())
    ids = batch["input_ids"] if hasattr(batch, "get") else {}
    width = int(ids.shape[-1]) if hasattr(ids, "shape") else 0
    if require_cuda and not cuda:
        return {"ran": False, "source": "configured_max",
                "reason": ("no CUDA device: the worst-case VRAM probe is skipped with a clear "
                           "message instead of reporting CPU host memory as GPU memory. Nothing was "
                           "measured here — run it on a GPU runtime."),
                "seq_len": width or None, "estimated_gb": None,
                "observed_gb": None, "observed_reserved_gb": None, "free_vram_gb": None,
                "fits": None, "steps": []}
    if not cuda:
        raise RuntimeError(
            "run_worst_case_probe(require_cuda=False) is not supported: peak VRAM is a CUDA "
            "counter, and host RSS is not a substitute for it. Run this on a GPU runtime."
        )
    device = "cuda"
    seq_len = width
    chunk = int(chunk_tokens)
    if chunk <= 0:
        raise RuntimeError(
            f"loss_chunk_tokens={chunk}: the worst-case probe refuses to run unchunked. Passing "
            "labels into the model makes transformers upcast the whole [seq, vocab] logits tensor to "
            f"fp32 = {seq_len * VOCAB * 4 / 2**30:.2f} GiB in one allocation, which is not the path "
            "the trainer takes."
        )
    micro_batch = max(1, int(micro_batch))
    grad_accum = max(1, int(grad_accum))
    if cuda:
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()      # BEFORE anything this probe allocates
        free_before, _total = torch.cuda.mem_get_info(0)
    else:
        free_before = None

    steps = []
    model.train()
    from lifemate.training.full_finetuning import build_optimizer_and_scheduler  # noqa: PLC0415

    optimizer, scheduler = build_optimizer_and_scheduler(model, cfg, num_training_steps=1)
    device_batch = {key: value.to(device) for key, value in batch.items()}
    optimizer.zero_grad(set_to_none=True)
    for micro in range(grad_accum):
        with torch.autocast(device_type="cuda" if cuda else "cpu",
                            dtype=COMPUTE_DTYPE if cuda else torch.float32, enabled=cuda):
            # Same call the trainer's loop makes: no labels reach the model, the chunked CE is the
            # production loss. Scaled by the accumulation count, as the loop does.
            loss = forward_loss(model, device_batch, chunk) / grad_accum
        loss.backward()
        steps.append(f"micro {micro + 1}/{grad_accum}: forward+chunked CE+backward")
    torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg.training.gradient_clip_norm))
    steps.append(f"clip_grad_norm({cfg.training.gradient_clip_norm})")
    optimizer.step()
    if scheduler is not None:
        scheduler.step()
    steps.append("optimizer.step + scheduler.step")

    eval_batch = {key: value.clone() for key, value in device_batch.items()}
    model.eval()
    with torch.no_grad():
        with torch.autocast(device_type="cuda" if cuda else "cpu",
                            dtype=COMPUTE_DTYPE if cuda else torch.float32, enabled=cuda):
            eval_loss = float(forward_loss(model, eval_batch, chunk).detach())
    steps.append(f"eval batch: chunked CE = {eval_loss:.4f}")

    saved = None
    if checkpoint_dir is not None:
        from lifemate.models.checkpoint import save_checkpoint  # noqa: PLC0415

        if tokenizer is None:
            raise RuntimeError("a checkpoint save was requested with no tokenizer; the probe writes "
                               "a real checkpoint, it does not fake one")
        saved = save_checkpoint(
            model, tokenizer, Path(checkpoint_dir),
            meta={"model_name": MODEL_ID, "model_mode": "lora", "step": 0,
                  "config": {"probe": True}, "seed": int(SEED)},
            is_lora=True, step=0, force=True,
        )
        steps.append(f"checkpoint save -> {saved}")

    observed = observed_reserved = free_after = None
    if cuda:
        torch.cuda.synchronize()
        observed = int(torch.cuda.max_memory_allocated(0))
        observed_reserved = int(torch.cuda.max_memory_reserved(0))
        free_after, _ = torch.cuda.mem_get_info(0)
    estimate = estimate_vram_gb(GPU_TIER, RUN["method"], seq_len, micro_batch)
    fits = None
    if cuda:
        fits = observed_reserved <= int(free_before or 0)
    return {
        "ran": True,
        "source": "configured_max",
        "reason": "measured",
        "seq_len": seq_len,
        "estimated_gb": float(estimate["total_gb"]),
        "observed_gb": None if observed is None else round(observed / 1e9, 2),
        "observed_reserved_gb": None if observed_reserved is None else round(observed_reserved / 1e9, 2),
        "free_vram_gb": None if free_after is None else round(free_after / 1e9, 2),
        "free_before_gb": None if free_before is None else round(free_before / 1e9, 2),
        "device_total_gb": GPU.get("vram_total_gb"),
        "fits": fits,
        "chunk_tokens": min(chunk, seq_len),
        "micro_batch": micro_batch,
        "grad_accum": grad_accum,
        "steps": steps,
        "saved_checkpoint": None if saved is None else str(saved),
    }

def print_worst_case_report(result):
    """Print the 15 required labels, then the OBSERVED-vs-ESTIMATE reading.

    Every VRAM number is labelled: ``Estimated VRAM`` is `estimate_vram_gb` (a heuristic),
    ``Observed VRAM`` is ``max_memory_allocated`` (measured). They are different quantities and are
    never merged into one figure.
    """
    def gb(value):
        return "not measured" if value is None else f"{value} GB"

    observed = result.get("observed_gb")
    estimate = result.get("estimated_gb")
    reserved = result.get("observed_reserved_gb")
    print(f"GPU                    : {GPU.get('name') or 'none'}")
    print(f"CUDA                   : {GPU.get('cuda') or 'n/a'}")
    print(f"PyTorch                : {GPU.get('torch') or (torch.__version__ if torch else 'n/a')}")
    print(f"Model                  : {MODEL_ID}")
    print(f"Precision              : {PRECISION}")
    print(f"Quantization           : {RUN['method']} (base {'nf4 4-bit' if RUN['quantized_base'] else '16-bit'})")
    print(f"LoRA rank              : {RUN['lora_r']}")
    print(f"Sequence length        : {result.get('seq_len')} ({result.get('source')})")
    print(f"Microbatch size        : {result.get('micro_batch')}")
    print(f"Gradient accumulation  : {result.get('grad_accum')}")
    print(f"Peak allocated VRAM    : {gb(observed)} (OBSERVED, torch.cuda.max_memory_allocated)")
    print(f"Peak reserved VRAM     : {gb(reserved)} (OBSERVED, torch.cuda.max_memory_reserved)")
    print(f"Free VRAM              : {gb(result.get('free_vram_gb'))} after the probe "
          f"(before: {gb(result.get('free_before_gb'))})")
    print(f"Estimated VRAM         : {gb(estimate)} (ESTIMATE from estimate_vram_gb, heuristic)")
    print(f"Observed VRAM          : {gb(observed)} (OBSERVED peak of the full "
          f"accumulate+clip+step+eval+save sequence)")
    if result.get("ran") and observed is not None and estimate:
        delta = observed - estimate
        print(f"  ESTIMATE vs OBSERVED: {estimate} GB projected, {observed} GB measured "
              f"({'under' if delta > 0 else 'over'}-estimated by {abs(delta):.2f} GB). The ESTIMATE "
              "sizes the activation term heuristically; only OBSERVED can refuse a run.")
    if not result.get("ran"):
        print(f"  probe not run: {result.get('reason')}")
    return result

PROBE_RESULT = None
if not GPU.get("available"):
    PROBE_RESULT = {"ran": False, "source": "configured_max", "seq_len": int(RUN["max_seq_length"]),
                    "reason": "no CUDA GPU visible; the worst-case VRAM probe is a no-op here",
                    "estimated_gb": None, "observed_gb": None, "observed_reserved_gb": None,
                    "free_vram_gb": None, "fits": None,
                    "steps": ["skipped: no CUDA device"]}
    print_worst_case_report(PROBE_RESULT)
else:
    WORST_CASE_DIR = LOCAL_OUTPUT_ROOT / "probe_worst_case"
    if WORST_CASE_DIR.exists():
        shutil.rmtree(WORST_CASE_DIR)
    WORST_SAMPLE = longest_prepared_sample_length(TRAIN_DATASET_DIR / "train.jsonl")
    print(f"batch budgets  : configured MAX_SEQ_LENGTH={RUN['max_seq_length']} | longest prepared "
          f"sample={WORST_SAMPLE} tokens -> these are DIFFERENT quantities and are probed separately")
    _probe_model, _probe_tokenizer = load_for_verification()
    if CLI_TUNING_MODE == "qlora":
        from lifemate.training.lora_finetuning import configure_qlora  # noqa: PLC0415

        _probe_report = configure_qlora(_probe_model, _probe_tokenizer, cfg)
    else:
        from lifemate.training.lora_finetuning import configure_lora  # noqa: PLC0415

        _probe_report = configure_lora(_probe_model, cfg)
    PROBE_RESULT = run_worst_case_probe(
        _probe_report["model"],
        _probe_batch(_probe_tokenizer, RUN["max_seq_length"], RUN["micro_batch"]),
        cfg=cfg,
        chunk_tokens=cfg.training.loss_chunk_tokens,
        grad_accum=RUN["grad_accum"],
        micro_batch=RUN["micro_batch"],
        tokenizer=_probe_tokenizer,
        checkpoint_dir=WORST_CASE_DIR / f"step-0",
    )
    print_worst_case_report(PROBE_RESULT)
    if WORST_SAMPLE and WORST_SAMPLE < int(RUN["max_seq_length"]):
        _sample_result = run_worst_case_probe(
            _probe_report["model"],
            _probe_batch(_probe_tokenizer, WORST_SAMPLE, RUN["micro_batch"]),
            cfg=cfg,
            chunk_tokens=cfg.training.loss_chunk_tokens,
            grad_accum=RUN["grad_accum"],
            micro_batch=RUN["micro_batch"],
            tokenizer=_probe_tokenizer,
            checkpoint_dir=WORST_CASE_DIR / "step-sample",
            supervised_tail=min(64, max(1, WORST_SAMPLE // 4)),
        )
        _sample_result["source"] = "longest_sample"
        print(f"\nlongest prepared sample ({WORST_SAMPLE} tokens), measured separately:")
        print(f"  Peak allocated VRAM    : {_sample_result['observed_gb']} GB")
        print(f"  Peak reserved VRAM     : {_sample_result['observed_reserved_gb']} GB")
        print(f"  Estimated VRAM         : {_sample_result['estimated_gb']} GB (ESTIMATE)")
        PROBE_RESULT["longest_sample"] = _sample_result
    # A probe that does not fit STOPS here. Lowering MAX_SEQ_LENGTH silently is exactly the failure
    # this cell exists to prevent: the run would then train on a different context than configured.
    if PROBE_RESULT.get("fits") is False:
        raise RuntimeError(
            f"the worst-case probe at MAX_SEQ_LENGTH={PROBE_RESULT['seq_len']} reserved "
            f"{PROBE_RESULT['observed_reserved_gb']} GB of the {PROBE_RESULT['free_before_gb']} GB "
            "free before it started: this configuration does not fit and the notebook will NOT lower "
            "MAX_SEQ_LENGTH for you. Options, in order of preference:\n"
            f"  1. raise the GPU tier (L4 24 GB / A100 40-80 GB) and keep MAX_SEQ_LENGTH="
            f"{PROBE_RESULT['seq_len']};\n"
            "  2. set TRAINING_METHOD='qlora' (4-bit frozen base frees ~6 GB);\n"
            f"  3. lower MAX_SEQ_LENGTH yourself to <= {seq_cap(GPU_TIER, RUN['method'], RUN['micro_batch'])} "
            "and re-run section 5 so the dataset is rebuilt at the new budget;\n"
            "  4. keep MICRO_BATCH_SIZE=1 and raise GRADIENT_ACCUMULATION_STEPS to hold the "
            "effective batch."
        )
    RELEASE_WORST_CASE = release_model_refs()
    shutil.rmtree(WORST_CASE_DIR, ignore_errors=True)
    free_gpu()


## 8. Training

Purpose: run the repo's trainer (its loop, its gradient accumulation, checkpointing, LR schedule
and validation) for the configured epochs. The notebook only assembles argv and watches the output.

**Resume is already decided before this cell runs** (cell 6d applied `RESUME_MODE`): this cell only
carries the decision out. It passes `--save-steps/--eval-steps` when the checkout exposes them
(feature-detected from the script's own `--help`), and it quotes the trainer's *own* resume evidence —
`resumed`, `resume mode : exact|best-effort`, `epoch`, `scheduler`, `rng` — instead of recomputing a
second verdict that could disagree with the one the trainer made.

While it runs, this cell **syncs each completed checkpoint to Drive** as soon as it appears
(`checkpoint_ready()` proves `training_state.json`, weights and `optimizer_state.pt` are all
present) through a staging directory + fsync + verified marker, and the copy is re-verified after the
install. The previous generation is kept, not deleted, so a crash between the two renames still
leaves a complete, verified `step-<n>` under one of the two names — that is what makes a disconnect
survivable. A `step-<n>` directory is immutable once published: the step number is its identity, so
two runs can never interleave inside one directory.

`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` (exported in section 1) is inherited by the
trainer subprocess and measurably reduces allocator fragmentation at long sequence lengths. It
changes no number in the run, only how the driver hands memory back.

Cost: the dominant term. Rough ESTIMATE for 4368 records, 3 epochs, effective batch 8,
~1650 steps: T4 16 GB ≈ 4-7 h; L4 ≈ 2.5-4 h; A100-40/80 ≈ 1-2 h. A Colab runtime that idles while
you read logs bills exactly the same — stop it when done (section 13).


In [ ]:
# --- 8. Train, streaming output and mirroring finished checkpoints --------
SYNCED_CHECKPOINTS = {}
TRAIN_LOG_PATH = LOCAL_OUTPUT_ROOT / "train_stdout.log"
TRAIN_RESUME_REPORT = None

def latest_step():
    """Highest step number present locally, or 0."""
    steps = [int(p.name.split("-")[-1]) for p in RUN_DIR.glob("step-*") if p.is_dir()] if RUN_DIR.is_dir() else []
    return max(steps) if steps else 0

def sync_finished_checkpoints(verbose=True):
    """Copy every complete, not-yet-synced step dir to Drive (atomic + verified marker).

    ``checkpoint_ready`` is the gate: a directory whose ``training_state.json``, weights or
    ``optimizer_state.pt`` is incomplete is skipped with its reason, never copied as if finished.
    """
    run_root = RUN_DIR
    if not run_root.is_dir():
        return []
    done = []
    for step_dir in sorted((p for p in run_root.glob("step-*") if p.is_dir()),
                           key=lambda p: int(p.name.split("-")[-1])):
        if step_dir.name in SYNCED_CHECKPOINTS:
            continue
        ok, reason = checkpoint_ready(step_dir)
        if not ok:
            if verbose:
                print(f"  [sync] {step_dir.name} not ready: {reason}")
            continue
        if DRIVE_ROOT is not None:
            target = DRIVE_PATHS["checkpoints"] / RUN_NAME / step_dir.name
            atomic_sync(step_dir, target, extra={"step": int(step_dir.name.split("-")[-1]),
                                                 "run_name": RUN_NAME, "model_id": MODEL_ID,
                                                 "kind": "checkpoint"})
            verified, why = verify_sync_marker(target)
            if not verified:
                raise RuntimeError(
                    f"the Drive copy {target} does not verify right after the sync ({why}); the "
                    "local copy is kept and nothing else is mirrored."
                )
        SYNCED_CHECKPOINTS[step_dir.name] = str(time.strftime("%H:%M:%S"))
        done.append(step_dir.name)
        if verbose:
            print(f"  [sync] {step_dir.name} durable ({SYNCED_CHECKPOINTS[step_dir.name]})")
    return done

def run_training(*, max_steps=None, resume=None):
    """Run the trainer in a subprocess; mirror checkpoints; raise on a non-zero exit.

    Returns ``(lines, resume_lines)``: the child's stdout plus the resume-evidence lines it printed
    (``resumed`` / ``resume mode`` / ``epoch`` / ``scheduler`` / ``rng``), so section 9 can quote what
    the trainer itself decided instead of recomputing it.
    """
    global LAST_CLI_PEAK_MIB
    env = dict(os.environ)
    # 1a exported expandable_segments already; restated so a re-run of this cell alone keeps it.
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    env.setdefault("PYTHONUNBUFFERED", "1")
    env.setdefault("WANDB_MODE", "offline")   # only used if training.report_to asks for wandb
    cmd = [sys.executable, str(REPO_DIR / "scripts" / "train.py"),
           *[str(a) for a in train_args(max_steps=max_steps, resume=resume)]]
    def _tick():
        # Live progress: finished checkpoints (mirrored immediately) and the step counter.
        before = len(SYNCED_CHECKPOINTS)
        sync_finished_checkpoints(verbose=False)
        if len(SYNCED_CHECKPOINTS) != before:
            print(f"    [run] step {latest_step()} checkpoint ready "
                  f"({len(SYNCED_CHECKPOINTS)} synced)", flush=True)

    rc, lines = stream_process(cmd, env, sample_peak=True, peak_interval=15, status_prefix="run",
                               on_tick=_tick)
    TRAIN_LOG_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
    sync_finished_checkpoints()
    if rc != 0:
        raise RuntimeError(
            f"train.py exited {rc}. Full output saved to {TRAIN_LOG_PATH}. 130 means it was "
            "interrupted after writing an interrupt-<step> checkpoint — resume with "
            "RESUME_MODE='auto' in section 2."
        )
    resume_lines = [line for line in lines
                    if line.startswith(("resumed", "resume mode", "epoch  ", "scheduler", "rng "))]
    return lines, resume_lines

# Section 6d already applied RESUME_MODE: fresh may have renamed the run, auto/explicit verified a
# checkpoint. This cell only carries the decision out — it never re-decides.
resume_target = RESUME_PLAN["checkpoint"]
if resume_target is not None:
    print(f"resuming from {resume_target} (verified by section 6d: "
          f"exact={bool((RESUME_PLAN['report'] or {}).get('exact'))})")
else:
    print(f"fresh run: RUN_NAME={RUN_NAME!r}, no checkpoint resumed")
if resume_target is None and RUN_DIR.is_dir() and any(p.is_dir() for p in RUN_DIR.glob("step-*")):
    _existing = sorted((p for p in RUN_DIR.glob("step-*") if p.is_dir()),
                       key=lambda p: int(p.name.split("-")[-1]))
    raise RuntimeError(
        f"{RUN_DIR} already holds {len(_existing)} checkpoint(s) (latest {_existing[-1].name}) and "
        "RESUME_MODE='fresh' produced this name anyway. The repo's CheckpointManager refuses to "
        "overwrite a populated step directory, and two runs must never share step numbering. Set "
        "RESUME_MODE='auto' to continue it, or change RUN_NAME in section 2. Nothing was launched."
    )

if FEATURE_REPORT is not None:
    print(f"features       : {FEATURE_REPORT['mode']} | "
          f"{FEATURE_REPORT['enabled'] or 'none enabled'} | training fingerprint "
          f"{FEATURE_REPORT['training_fingerprint']}")
print("CU cost is billed for wall-clock GPU time; a slow epoch is a bigger bill.")
train_output, TRAIN_RESUME_LINES = run_training(max_steps=int(MAX_STEPS) or None,
                                                resume=resume_target)
if TRAIN_RESUME_LINES:
    print("\ntrainer's own resume evidence:")
    for _line in TRAIN_RESUME_LINES:
        print(f"  {_line}")

summary_path = Path(cfg.training.output_dir) / RUN_NAME / "train_summary.json"
if not summary_path.is_file():
    summary_path = Path(cfg.training.output_dir) / "train_summary.json"
if summary_path.is_file():
    TRAIN_SUMMARY = json.loads(summary_path.read_text(encoding="utf-8"))
    _history = (TRAIN_SUMMARY.get("result") or {}).get("metrics_history") or []
    print("\nstep   loss      val_loss  lr")
    for row in _history:
        loss, val = row.get("train_loss"), row.get("val_loss")
        lr = row.get("learning_rate")
        print(f"{str(row.get('step')):<6} "
              f"{(f'{loss:.4f}' if isinstance(loss, float) else '-'):<9} "
              f"{(f'{val:.4f}' if isinstance(val, float) else '-'):<9} "
              f"{(f'{lr:.3e}' if isinstance(lr, float) else '-')}")
    print(f"\nsteps={TRAIN_SUMMARY.get('result', {}).get('steps')} "
          f"epochs={TRAIN_SUMMARY.get('result', {}).get('epochs')} "
          f"final_loss={TRAIN_SUMMARY.get('result', {}).get('final_loss')} "
          f"val_loss={TRAIN_SUMMARY.get('result', {}).get('val_loss')}")
    print(f"tuning_mode={TRAIN_SUMMARY.get('tuning_mode')} model={TRAIN_SUMMARY.get('model')}")
else:
    TRAIN_SUMMARY = None
    print(f"warning: {summary_path} missing; the run's stdout is in {TRAIN_LOG_PATH}")

# The resume report, re-derived from the ARTIFACT the trainer actually continued from (never from a
# variable that merely exists in this kernel), with its own line quoted when the CLI printed one.
if resume_target is not None and RESUME_PLAN.get("report") is not None:
    TRAIN_RESUME_REPORT = resume_report(RESUME_PLAN["report"], expected_total_steps=TOTAL_STEPS,
                                       cli_lines=TRAIN_RESUME_LINES)
    print("\nafter training, the resumed position was:")
    print_resume_report(TRAIN_RESUME_REPORT)
LAST_STEP_DIR = max((p for p in RUN_DIR.glob("step-*") if p.is_dir()),
                    key=lambda p: int(p.name.split("-")[-1])) if RUN_DIR.is_dir() else None
print(f"latest checkpoint: {LAST_STEP_DIR}")


## 9. Checkpoint contents + an opt-in resume proof

Purpose: show that the saved directory is enough to *continue the same optimisation*. A checkpoint
from this trainer holds:

`adapter_model.safetensors` + `adapter_config.json` (or full weights), the tokenizer,
`training_state.json` (step, epoch, seed, `config_hash`, dataset manifest hash, is_lora, precision,
torch/CUDA/python RNG state, and — in schema 2 — a `progress` block with `global_step`,
`epoch_index`, `next_micro_index`, `permutation`, `planned_total_steps`, `dataset_order_hash`),
`optimizer_state.pt` (AdamW moments + scheduler + grad scaler), `lifemate_meta.json` (model id,
tuning mode, config), and — after a Drive sync — a verified `SYNC_COMPLETE.json`.

**A schema-1 checkpoint is not corrupt, it is merely inexact.** Without a `progress` block there is
no recorded permutation or micro-batch cursor, so the reader reports `exact=False` with the reason
and the trainer resumes in `best-effort` mode. The reader accepts both schemas; the notebook never
writes schema 2 itself.

Loading only `adapter_model.safetensors` is **adapter-only loading**; it restores weights and nothing
about the optimizer. This notebook refuses to call that a resume: `checkpoint_ready()` requires
`training_state.json` **and** a loadable `optimizer_state.pt` (with its `scheduler`/`grad_scaler`
keys), plus a marker whose per-file sha256 re-hashes cleanly for a Drive copy, and the trainer itself
refuses a config-hash or dataset-hash mismatch.

Two halves, and the second one is **opt-in**:

1. **Always** (read-only): print the directory, run `checkpoint_ready`, compare `config_hash` with
   this session. No subprocess, no new step dirs, no Drive write.
2. **Only with `RUN_RESUME_PROOF = True`** in section 2: run two more optimizer steps from the
   checkpoint and assert the step counter advanced. That **mutates** the experiment — it prints which
   `step-<n>` directories it created and records `resume_proof.json` with `"isolated": false`, so the
   extension is never mistaken for a read-only demonstration.

Cost: half 1 is seconds; half 2 is 2 optimizer steps ≈ 1-3 min of GPU.


In [ ]:
# --- 9a. Inspect the checkpoint, then (opt-in) prove continuation -----------
if LAST_STEP_DIR is None:
    raise RuntimeError(f"no step-* checkpoint under {RUN_DIR}; run section 8 first")

state = json.loads((LAST_STEP_DIR / "training_state.json").read_text(encoding="utf-8"))
meta = json.loads((LAST_STEP_DIR / "lifemate_meta.json").read_text(encoding="utf-8"))
ok, why = checkpoint_ready(LAST_STEP_DIR)
CONTENTS = {
    "dir": str(LAST_STEP_DIR),
    "step": state.get("step"),
    "epoch": state.get("epoch"),
    "seed": state.get("seed"),
    "config_hash": state.get("config_hash"),
    "dataset_manifest_hash": state.get("dataset_manifest_hash"),
    "is_lora": state.get("is_lora"),
    "precision": state.get("precision"),
    "rng_state_saved": {"torch": state.get("torch_rng_state") is not None,
                        "cuda": state.get("cuda_rng_state") is not None,
                        "python": state.get("python_rng_state") is not None},
    "optimizer_state_present": (LAST_STEP_DIR / "optimizer_state.pt").is_file(),
    "model_identity": meta.get("model_name"),
    "tuning_mode": meta.get("tuning_mode"),
    "files": sorted(p.name for p in LAST_STEP_DIR.iterdir()),
    "size_mb": round(sum(p.stat().st_size for p in LAST_STEP_DIR.rglob("*") if p.is_file()) / 2**20, 1),
    "resumable": {"ok": ok, "reason": why},
}
print(json.dumps(CONTENTS, indent=2, ensure_ascii=False))

# Refusing here is the point: a directory that cannot continue the SAME optimisation must not be
# presented as a resume target.
if not ok:
    raise RuntimeError(f"the checkpoint is not resumable: {why}")
if CONTENTS["config_hash"] != CONFIG_HASH:
    raise RuntimeError(
        f"the checkpoint was trained under config hash {CONTENTS['config_hash']}, this session's "
        f"config hashes to {CONFIG_HASH}. Something changed between the run and now (lr, seed, seq "
        "len, batch, epochs, precision, tuning mode); the trainer would refuse it for the same reason."
    )
print(f"resume checks  : resumable, config_hash matches this session ({CONFIG_HASH})")

RESUME_PROOF = {"from_step": int(CONTENTS["step"]), "to_step": None, "continued": False,
                "isolated": None, "ran": bool(RUN_RESUME_PROOF)}
if not RUN_RESUME_PROOF:
    # Read-only by default. The demonstration below is a real training launch: it extends a FINISHED
    # experiment by two optimizer steps and writes new step dirs (and, with Drive on, syncs them), so
    # it must be asked for explicitly.
    print(
        f"resume proof   : SKIPPED (RUN_RESUME_PROOF=False in section 2). The checkpoint above is "
        f"verified resumable at step {RESUME_PROOF['from_step']}. Set RUN_RESUME_PROOF=True in section "
        "2 and re-run this cell to launch the 2-step continuation; be aware it MUTATES the run "
        "(two new step-<n> directories, mirrored to Drive) rather than only reading it."
    )
else:
    _resume_limit = int(CONTENTS["step"]) + 2
    print(f"resume proof   : RUN_RESUME_PROOF=True -> launching {RESUME_PROOF['from_step']} -> "
          f"{_resume_limit} optimizer steps in {RUN_DIR}. This MUTATES the experiment: two new "
          f"step-<n> directories appear next to {LAST_STEP_DIR.name} and are synced to Drive.")
    resume_log, resume_evidence = run_training(max_steps=_resume_limit, resume=LAST_STEP_DIR)
    resumed_lines = [line for line in resume_evidence if line.startswith("resumed")]
    if not resumed_lines:
        raise RuntimeError(
            f"the trainer did not print a `resumed` line; output saved to {TRAIN_LOG_PATH}. "
            "That line is the only evidence that step continuity was restored."
        )
    print(resumed_lines[0])
    _new_dirs = sorted((p for p in RUN_DIR.glob("step-*") if int(p.name.split("-")[-1]) > int(CONTENTS["step"])),
                       key=lambda p: int(p.name.split("-")[-1]))
    if not _new_dirs:
        raise RuntimeError(
            f"no step dir above {CONTENTS['step']} appeared: the run did not continue. Check that "
            "MAX_STEPS / NUM_EPOCHS leave room after the checkpoint's step (total steps must exceed it)."
        )
    RESUME_PROOF["to_step"] = int(_new_dirs[-1].name.split("-")[-1])
    RESUME_PROOF["continued"] = RESUME_PROOF["to_step"] > RESUME_PROOF["from_step"]
    RESUME_PROOF["new_step_dirs"] = [p.name for p in _new_dirs]
    RESUME_PROOF["synced"] = sorted(SYNCED_CHECKPOINTS)
    # Not isolated: the new steps live in the same run directory as the experiment they extend.
    RESUME_PROOF["isolated"] = False
    (LOCAL_OUTPUT_ROOT / "resume_proof.json").write_text(json.dumps(RESUME_PROOF, indent=2), encoding="utf-8")
    if DRIVE_ROOT is not None:
        atomic_sync(_new_dirs[-1], DRIVE_PATHS["checkpoints"] / RUN_NAME / _new_dirs[-1].name,
                    extra={"step": RESUME_PROOF["to_step"], "resume_proof": True, "run_name": RUN_NAME})
    print(json.dumps(RESUME_PROOF, indent=2))
    if not RESUME_PROOF["continued"]:
        raise RuntimeError("resume did not advance the step counter; this is not a valid resume")
    LAST_STEP_DIR = _new_dirs[-1]
free_gpu()


## 10. Evaluation

Purpose: score the trained checkpoint with the repo's evaluator. Teacher-forced metrics (validation
loss, tool-name accuracy, argument accuracy, action exact match) are available whenever the model
loads; the closed loop needs the organizers' `zaic_tools` SDK and is attempted only when that import
succeeds.

**Compliance.** Teacher-forced numbers on the held-out split of `train` are an *offline* signal.
Mock-environment numbers are printed with `MOCK ENV / SYNTHETIC` and are never presented as an
official score; nothing here writes a submission file.

Cost: validation-only over 482 records ≈ 3-8 min on T4; a 20-sample closed loop adds a few minutes.


In [ ]:

# --- 10. Evaluate: teacher-forced (always), closed loop (only if possible) --
require_flags("evaluate.py", ["--validation-only", "--closed-loop", "--checkpoint", "--model-mode",
                              "--config", "--predictions", "--output", "--limit", "--split"])


def eval_args(output_dir, *, validation_only=True, closed_loop=False, limit=None):
    """argv for the evaluator; teacher-forced scoring uses the val decision records."""
    args = [
        "--config", CONFIG_PATH,
        "--checkpoint", LAST_STEP_DIR,
        "--model-mode", "lora",
        "--split", "train",
        "--output", output_dir,
    ]
    if validation_only:
        args += ["--validation-only", "--predictions", TRAIN_DATASET_DIR / "val.jsonl"]
    if closed_loop:
        args += ["--closed-loop"]
        if limit:
            args += ["--limit", int(limit)]
    return args


EVAL_DIR = LOCAL_OUTPUT_ROOT / "evaluation"
if EVAL_DIR.exists():
    shutil.rmtree(EVAL_DIR)

run_cli("evaluate.py", eval_args(EVAL_DIR))
metrics_path = EVAL_DIR / "metrics.json"
if not metrics_path.is_file():
    raise RuntimeError(f"evaluate.py wrote no metrics.json under {EVAL_DIR}; see its output above")
EVAL_PAYLOAD = json.loads(metrics_path.read_text(encoding="utf-8"))
TEACHER_FORCED = {
    "n_records": (EVAL_PAYLOAD.get("teacher_forced") or {}).get("n_records"),
    "metrics": EVAL_PAYLOAD.get("metrics"),
    "mock_env": EVAL_PAYLOAD.get("mock_env"),
    "provenance": EVAL_PAYLOAD.get("provenance"),
}
print("teacher-forced metrics — offline, on the held-out split of `train`. NOT a leaderboard score:")
print(json.dumps(TEACHER_FORCED["metrics"], indent=2)[:3000])
if EVAL_PAYLOAD.get("mock_env"):
    raise RuntimeError(
        "this report is marked MOCK ENV / SYNTHETIC; something routed evaluation through the mock "
        "environment. It must never be quoted as a score."
    )

# Closed loop: only with the organizers' SDK present. Never silently upgraded to the mock.
CLOSED_LOOP = {"attempted": False, "reason": "data/train/zaic_tools is absent"}
if (DATA_ROOT / "train" / "zaic_tools").is_dir():
    CLOSED_LOOP["attempted"] = True
    CLOSED_DIR = LOCAL_OUTPUT_ROOT / "evaluation_closed_loop"
    if CLOSED_DIR.exists():
        shutil.rmtree(CLOSED_DIR)
    try:
        run_cli("evaluate.py", eval_args(CLOSED_DIR, validation_only=False, closed_loop=True, limit=20),
                sample_peak=True)
        CLOSED_LOOP["result"] = "ran against the official environment, train split, first 20 samples"
        _closed = json.loads((CLOSED_DIR / "metrics.json").read_text(encoding="utf-8"))
        CLOSED_LOOP["status_counts"] = ((_closed.get("closed_loop") or {}).get("status_counts"))
        CLOSED_LOOP["mock_env"] = _closed.get("mock_env")
    except RuntimeError as exc:
        CLOSED_LOOP["result"] = f"not run: {exc}"
else:
    print(
        "closed loop skipped: data/train/zaic_tools is absent, and the notebook will not substitute "
        "the mock for it. With the SDK present, the command is:\n"
        f"  python scripts/evaluate.py --closed-loop --split train --limit 20 --model-mode lora "
        f"--checkpoint {LAST_STEP_DIR} --config {CONFIG_PATH} --output {LOCAL_OUTPUT_ROOT}/evaluation_closed_loop\n"
        "Without it, the only closed-loop option is the mock, whose every artifact is stamped "
        "MOCK ENV / SYNTHETIC and which is NEVER a score:\n"
        f"  python scripts/evaluate.py --closed-loop --use-mock-env --split train --limit 5 "
        f"--model-mode lora --checkpoint {LAST_STEP_DIR} --output {LOCAL_OUTPUT_ROOT}/evaluation_mock"
    )
print(json.dumps(CLOSED_LOOP, indent=2))
if DRIVE_ROOT is not None and EVAL_DIR.is_dir():
    atomic_sync(EVAL_DIR, DRIVE_PATHS["outputs"] / f"evaluation_{RUN_NAME}", extra={"kind": "evaluation"})
free_gpu()


## 11. Export

Purpose: produce a portable artifact. The default is the **adapter** — a few tens of MB, loadable at
inference with `--model-mode lora --checkpoint <dir>`, no merge required. A merge into full weights
is available when the GPU can hold the fp16 base during the merge; it is off by default because it
costs VRAM, disk and time for no training benefit.

Exported: adapter weights, tokenizer, `export_meta.json` (reproducibility: model id, config hash,
dataset hash, git commit, libraries, step) and the effective YAML. Then synced to Drive.

Cost: <1 min for the adapter copy; a merge on a 4B model ≈ 2-6 min plus ~9 GB of disk.


In [ ]:
# --- 11. Export the adapter (+ optional merge) ----------------------------
require_flags("export_model.py", ["--ckpt", "--output", "--merge-lora", "--base-model", "--dtype", "--config"])
MERGE_LORA = False  # set True to also produce merged fp16 weights (needs ~20 GB free VRAM+disk)

# Re-read the state: section 9 may have resumed past the checkpoint inspected there.
EXPORT_STEP = int(json.loads((LAST_STEP_DIR / "training_state.json").read_text(encoding="utf-8"))["step"])
EXPORT_DIR = LOCAL_OUTPUT_ROOT / "export" / f"{RUN_NAME}-step{EXPORT_STEP}"
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
ADAPTER_DIR = EXPORT_DIR / "adapter"
ADAPTER_DIR.mkdir(parents=True)

# Optimizer state is not part of a portable adapter; everything else in the step dir is.
KEEP = ("adapter_model.safetensors", "adapter_model.bin", "adapter_config.json", "tokenizer.json",
        "tokenizer_config.json", "special_tokens_map.json", "vocab.json", "merges.txt",
        "generation_config.json", "chat_template.jinja", "config.json")
copied = []
for name in KEEP:
    src = LAST_STEP_DIR / name
    if src.is_file():
        shutil.copy2(src, ADAPTER_DIR / name)
        copied.append(name)
if not any(name.startswith("adapter_model") for name in copied):
    raise RuntimeError(
        f"{LAST_STEP_DIR} holds no adapter weights (found: {sorted(p.name for p in LAST_STEP_DIR.iterdir())}). "
        "This notebook only exports LoRA/QLoRA adapters; a full-finetune checkpoint would need "
        "scripts/export_model.py without the adapter copy."
    )
shutil.copy2(CONFIG_PATH, EXPORT_DIR / "effective_config.yaml")
_export_meta = {
    "model_id": MODEL_ID,
    "base_model": MODEL_ID,
    "kind": "lora_adapter",
    "source_checkpoint": str(LAST_STEP_DIR),
    "step": EXPORT_STEP,
    "training_state": json.loads((LAST_STEP_DIR / "training_state.json").read_text(encoding="utf-8")),
    "config_hash": CONFIG_HASH,
    "dataset_manifest_hash": DATASET_REPORT["split_manifest"]["group_hash"],
    "dataset": DATASET_REPORT,
    "git_commit": subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR),
                                 capture_output=True, text=True).stdout.strip(),
    "libraries": {"torch": torch.__version__, "transformers": transformers.__version__,
                  "peft": peft.__version__, "python": sys.version.split()[0]},
    "precision": PRECISION,
    "trainable_params": MODEL_FACTS["trainable_params"],
    "files": copied,
    "how_to_run": (f"python scripts/infer.py --model-mode lora --checkpoint {ADAPTER_DIR} "
                   f"--model {MODEL_ID} --split public_test --data-root ./data "
                   "--output submission.jsonl"),
    "caveat": "An adapter reproduces the weights; it does not carry optimizer state. For resuming "
              "training use the step-<n> directory, not this export.",
}
(EXPORT_DIR / "export_meta.json").write_text(json.dumps(_export_meta, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"exported adapter -> {ADAPTER_DIR} ({sum((ADAPTER_DIR / n).stat().st_size for n in copied) / 2**20:.1f} MB)")

if MERGE_LORA:
    _free_gb = float(GPU.get("vram_free_gb") or 0)
    if _free_gb < 20:
        print(f"merge skipped: {_free_gb} GB free VRAM, a 4B fp16 merge needs ~20 GB. The adapter is "
              "the deliverable; the repo merges it at load time when you pass --model-mode lora.")
    else:
        run_cli("export_model.py", ["--ckpt", LAST_STEP_DIR, "--output", EXPORT_DIR / "merged",
                                    "--merge-lora", "--base-model", MODEL_ID,
                                    "--config", CONFIG_PATH, "--dtype", PRECISION,
                                    "--force"], sample_peak=True)
        print("merged weights written; verify them with scripts/infer.py --model-mode full")

if DRIVE_ROOT is not None:
    atomic_sync(EXPORT_DIR, DRIVE_PATHS["outputs"] / EXPORT_DIR.name, extra={"kind": "export"})
    print(f"on Drive: {DRIVE_PATHS['outputs'] / EXPORT_DIR.name}")
print("download it from Drive, or copy to your machine, then:")
print(f"  python scripts/infer.py --model-mode lora --checkpoint <downloaded>/adapter \\\n"
      f"      --split public_test --data-root ./data --output submission.jsonl \\\n"
      f"      --model {MODEL_ID}")


## 12. Inference smoke test

Purpose: load the exported adapter through the repo's loader, generate a real completion, and push
it through the agent's parser + validator + one agent step. This catches adapter/tokenizer/chat-
template mismatch — the failure mode that makes "the model loaded fine" worthless.

**A successful load is not a successful tool-calling evaluation.** This cell proves plumbing only.

Cost: ~30-60 s on GPU for one short generation.


In [ ]:
# --- 12. Load the export, generate, parse, validate, one agent step --------
from lifemate.agent.tool_parser import parse_model_output
from lifemate.agent.tool_registry import ToolRegistry
from lifemate.data.catalog import ToolCatalog
from lifemate.data.loader import load_samples
from lifemate.env.factory import create_environment
from lifemate.models.generation import HFGenerator
from lifemate.models.loader import load_model_and_tokenizer

bundle = load_model_and_tokenizer(
    MODEL_ID,
    torch_dtype=PRECISION,
    device="cuda",
    tuning_mode="lora",
    checkpoint=ADAPTER_DIR,
    attn_implementation=cfg.model.attn_implementation,
    trust_remote_code=cfg.model.trust_remote_code,
    max_new_tokens=cfg.agent.max_new_tokens,
)
SMOKE = {"adapter": str(ADAPTER_DIR), "model_mode": bundle.model_mode}
print(f"loaded adapter : {bundle.model_mode} | "
      f"{bundle.meta['n_parameters']['total']:,} params | dtype {bundle.meta['torch_dtype']}")

generator = HFGenerator(bundle.model, bundle.tokenizer, max_new_tokens=cfg.agent.max_new_tokens,
                        temperature=0.0, device=bundle.device)
catalog = ToolCatalog.from_file(DATA_ROOT / "train" / "train_tools_catalog.json")

_val_records = [json.loads(line) for line in (TRAIN_DATASET_DIR / "val.jsonl").read_text(encoding="utf-8").splitlines()[:5]]
_record = _val_records[0]
SMOKE["sample_id"] = _record.get("sample_id")
SMOKE["step_index"] = _record.get("step_index")
SMOKE["gold_target"] = _record.get("target")

# The record's own messages are the exact prompt bytes training used.
raw_output = generator.generate(list(_record["messages"]))
SMOKE["raw_output"] = raw_output
print(f"generated      : {raw_output!r}")

registry = ToolRegistry(catalog)                      # every app; a real sample narrows this
env = create_environment("train", str(_record.get("sample_id") or "smoke"), DATA_ROOT,
                         use_mock=True, catalog=catalog)   # MOCK ENV: plumbing only
action = parse_model_output(raw_output, registry, env)
SMOKE["action"] = {"type": action.type, "name": action.name, "arguments": action.arguments,
                   "error": action.error}
print(f"parsed+validated: {json.dumps(SMOKE['action'], ensure_ascii=False)}")
if action.type == "invalid":
    print("note: the model produced an invalid action — that is a training-signal question, not a "
          "plumbing bug. The parser refusing it is the correct behaviour.")

# Agent path: one real episode on a real labelled sample, with the mock tool backend.
samples = {str(s.id): s for s in load_samples(DATA_ROOT, "train")}
sample = samples.get(str(_record.get("sample_id")))
if sample is None:
    print(f"warning: sample {_record.get('sample_id')} not in the split; skipping the agent step")
else:
    from lifemate.agent.agent import ToolAgent
    agent_env = create_environment("train", str(sample.id), DATA_ROOT, use_mock=True, catalog=catalog)
    agent = ToolAgent(generator=generator, registry=ToolRegistry(catalog).for_sample(sample),
                      env=agent_env, config=cfg, max_steps=cfg.agent.max_steps)
    result = agent.run(sample, transcript=sample.audio_text)
    SMOKE["agent"] = {"status": result.status, "finish_reason": result.finish_reason,
                      "n_llm_calls": result.n_llm_calls, "tool_calls": result.tool_calls,
                      "mock_env": True}
    print("MOCK ENV / SYNTHETIC agent episode (plumbing only, not a score):")
    print(json.dumps(SMOKE["agent"], indent=2, ensure_ascii=False)[:2000])

if "agent" not in SMOKE:
    SMOKE["agent"] = {"status": "skipped", "reason": "sample unavailable"}
SMOKE["verdict"] = ("adapter loaded and generated, output parsed by the repo parser; this is NOT a "
                    "tool-calling accuracy measurement")
print(SMOKE["verdict"])
if DRIVE_ROOT is not None:
    (LOCAL_OUTPUT_ROOT / "smoke_test.json").write_text(
        json.dumps(SMOKE, indent=2, ensure_ascii=False), encoding="utf-8")

# Nothing downstream needs the loaded model, and section 13 must sync to Drive with the whole card
# free. release_model_refs() deletes the names (bundle and generator both hold the same weights and
# tokenizer) and then PROVES the namespace no longer reaches a module: setting them to None would
# leave the attribute paths in place, and torch.cuda.empty_cache() alone frees nothing still bound.
SMOKE_RELEASE = release_model_refs()
print("generation model released")



## 13. Resource management, final sync, shutdown

Purpose: leave the session in a clean, resumable state and stop paying. An allocated Colab runtime
bills Compute Units while it does nothing, so the last cell syncs results and releases memory;
then **disconnect the runtime yourself** (`Runtime -> Disconnect and delete runtime`).

Local step directories are redundant once they are durably on Drive — but "durably" is checked
**pairwise per directory**: the Drive counterpart must exist, its marker must verify (sha256 of every
file), its recorded run name / step / model id must match, and the copy must be resumable. A local
set with an EMPTY Drive set is refused outright, which is the bug this replaced (`all([]) is True`
deleted everything). The newest local generation is never removed, and
`DELETE_LOCAL_AFTER_VERIFIED_SYNC` defaults to **False**, so the cell normally prints what *would*
be deleted and deletes nothing.

Cheap vs expensive cells, and why the order is fixed:

* cheap (no GPU work): sections 1-5, 11, 13 — safe to re-run any time;
* GPU-heavy: 6 (weights), 7 (probes), 8 (training), 9 (2-step resume), 10 (evaluation), 12 (generation);
* weights are downloaded once into `HF_HOME` under the runtime-local work dir — re-running 6 does
  not re-download unless the runtime was recycled.

Cost: seconds of GPU (freeing) — and it stops the meter on everything else.


In [ ]:

# --- 13. Final sync + free everything ------------------------------------
# Report whatever exists: a section that failed earlier must not make clean-up crash too.
def _g(name, default=None):
    """Value of a notebook global, or ``default`` when that section never ran."""
    return globals().get(name, default)


# Nothing below needs a model: the smoke test (section 12) loaded the last one, and a stale CUDA
# context only makes the final sync slower and the report noisier.
FINAL_RELEASE = release_model_refs() if callable(globals().get("release_model_refs")) else None

FINAL = {
    "run_name": RUN_NAME,
    "model_id": MODEL_ID,
    "method": TRAINING_METHOD,
    "feature_mode": (_g("FEATURE_REPORT") or {}).get("mode"),
    "features_enabled": (_g("FEATURE_REPORT") or {}).get("enabled"),
    "training_feature_fingerprint": (_g("FEATURE_REPORT") or {}).get("training_fingerprint"),
    "config_hash": (_g("CONTENTS") or {}).get("config_hash"),
    "steps_synced": sorted(_g("SYNCED_CHECKPOINTS") or {}),
    "release": FINAL_RELEASE,
}

if DRIVE_ROOT is not None:
    _report = {
        "run_config": _g("RUN"),
        # The switchboard that produced this run, in the same JSON shape the scripts write:
        # enabled ids, mode (BASELINE/EXPERIMENTAL) and both fingerprints.
        "features": _g("FEATURE_REPORT"),
        "memory_revalidation": _g("MEMORY_REVALIDATION"),
        "dataset": _g("DATASET_REPORT"),
        "model_facts": _g("MODEL_FACTS"),
        "memory": {"estimate": _g("ESTIMATE"), "probe": _g("PROBE")},
        "training_summary": _g("TRAIN_SUMMARY"),
        "teacher_forced": _g("TEACHER_FORCED"),
        "closed_loop": _g("CLOSED_LOOP"),
        "resume_proof": _g("RESUME_PROOF"),
        "smoke_test": _g("SMOKE"),
        "git_commit": (_g("_export_meta") or {}).get("git_commit"),
        "finished_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    (LOCAL_OUTPUT_ROOT / "run_report.json").write_text(
        json.dumps(_report, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

    for _sub, _label in (("configs", f"configs_{RUN_NAME}"), ("export", "exports"),
                         ("evaluation", f"evaluation_{RUN_NAME}")):
        _src_dir = LOCAL_OUTPUT_ROOT / _sub
        if _src_dir.is_dir():
            atomic_sync(_src_dir, DRIVE_PATHS["outputs"] / _label, extra={"kind": _sub})
    _log_dir = DRIVE_PATHS["logs"] / RUN_NAME
    _log_dir.mkdir(parents=True, exist_ok=True)
    for _name in ("train_stdout.log", "run_report.json", "resume_proof.json", "smoke_test.json"):
        _src = LOCAL_OUTPUT_ROOT / _name
        if _src.is_file():
            shutil.copy2(_src, _log_dir / _name)
    with open(_log_dir / "train_stdout.log", "ab") as _handle:  # append-never-truncate + fsync
        _handle.flush()
        os.fsync(_handle.fileno())
    FINAL["drive"] = {
        "checkpoints": str(DRIVE_PATHS["checkpoints"] / RUN_NAME),
        "exports": str(DRIVE_PATHS["outputs"] / "exports"),
        "evaluation": str(DRIVE_PATHS["outputs"] / f"evaluation_{RUN_NAME}"),
        "logs": str(_log_dir),
    }
    print("on Drive:")
    for _label, _path in FINAL["drive"].items():
        print(f"  {_label:<12}: {_path}")

# Local step dirs are redundant once EVERY ONE of them is durably on Drive — and "durably" means
# this checkpoint's Drive counterpart verified (sha256 of every file) AND is itself resumable AND
# belongs to this run/model/step. The pairwise check lives in plan_local_cleanup(), because the old
# form (`all([]) is True` over an empty Drive glob) deleted local checkpoints when the Drive side was
# EMPTY. DELETE_LOCAL_AFTER_VERIFIED_SYNC defaults to False, so the normal outcome is a dry run.
_run_dir = _g("RUN_DIR")
_drive_run_dir = (DRIVE_PATHS["checkpoints"] / RUN_NAME) if DRIVE_ROOT is not None else None
CLEANUP = plan_local_cleanup(
    _run_dir,
    _drive_run_dir,
    run_name=RUN_NAME,
    model_id=MODEL_ID,
    delete=bool(DELETE_LOCAL_AFTER_VERIFIED_SYNC),
) if _run_dir is not None else {"ok": False, "reason": "no RUN_DIR in this session", "pairs": [],
                               "would_delete": [], "performed": []}
if _run_dir is not None:
    print(f"cleanup       : DELETE_LOCAL_AFTER_VERIFIED_SYNC={DELETE_LOCAL_AFTER_VERIFIED_SYNC} | "
          f"would delete: {CLEANUP['would_delete'] or 'none'} | deleted: "
          f"{CLEANUP['performed'] or 'none'}")
    if not DELETE_LOCAL_AFTER_VERIFIED_SYNC and CLEANUP["would_delete"]:
        print("              : dry run only — set DELETE_LOCAL_AFTER_VERIFIED_SYNC=True in section 2 "
              "to actually remove them (the newest generation is never removed)")
FINAL["drive_verification"] = {pair["local"]: {"ok": pair["ok"], "reasons": pair["reasons"]}
                               for pair in CLEANUP.get("pairs", [])}
FINAL["cleanup"] = CLEANUP
free_gpu()
print(json.dumps(FINAL, indent=2, default=str))
print("\nSESSION COMPLETE — nothing is left running. Now: Runtime -> Disconnect and delete runtime. "
      "An allocated-but-idle runtime bills Compute Units exactly like a busy one.")
